# EVTEAS-Py

Framework computacional para Estudo de Viabilidade Técnica, Econômica, Ambiental e Social (EVTEAS) aplicado à piscicultura — dissertação de Rafael Alves Bastos (UFF/MESC).

**Como usar**

1. Execute as células das Seções 1 a 12 (definições do framework). Elas não calculam nenhum resultado.
2. Execute a célula da **Seção 13 — Entrada de dados** e responda às perguntas. Há três opções:
   - **Novo projeto**: todas as entradas são digitadas (sem valores prontos). Só os coeficientes técnicos com referência normativa ou bibliográfica aparecem como sugestão, aceita com Enter e registrada como tal;
   - **Carregar arquivo**: reabre um arquivo `entradas_*.json` salvo anteriormente e permite revisar cada valor;
   - **Exemplo**: carrega o caso ilustrativo incorporado ao pacote, apenas para demonstração.
3. Confira o resumo das entradas, confirme e execute as Seções 14 a 17. A Seção 18 (opcional) mostra a rastreabilidade do artefato segundo as etapas da Design Science Research.

**Relato dos resultados na dissertação.** Os Capítulos 4 e 5 trazem lacunas com código ([E..] para entradas e [S..], [K..] e [N..] para resultados) e espaços para prints ([INSERIR PRINT P..]). A Seção 16 mostra e exporta o valor de cada código nesta execução (`valores_para_o_texto.xlsx`, com o roteiro de prints numa segunda aba), e cada tela do wizard ou saída a capturar é anunciada por uma linha `▶ PRINT P..`. Para a dissertação, execute também a Seção 15 (comparação de alternativas) antes da Seção 16.

Os resultados dependem integralmente das entradas informadas. Ao final da entrada, o arquivo `entradas_<projeto>.json` é salvo (e baixado no Colab) para que o estudo possa ser reaberto e revisado. Digite `sair` em qualquer pergunta para interromper: as respostas já dadas são salvas num arquivo `entradas_<projeto>_rascunho.json`, que pode ser reaberto pela opção 2 para continuar de onde parou. Números podem ser digitados no padrão brasileiro (40.000 ou 1.500,50).

In [ ]:
# Dependências (no Colab, apenas numpy-financial costuma faltar)
%pip install -q numpy pandas matplotlib openpyxl numpy-financial

In [ ]:
from __future__ import annotations
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

## 1. Configuração, premissas e rastreabilidade das fontes

In [ ]:
"""Estruturas de configuração do EVTEAS-Py.

Todas as premissas do estudo ficam concentradas em um único objeto
``EVTEASConfig`` que atravessa o pipeline (Capítulo 4, Seção 4.1). Cada bloco
corresponde a um eixo de modelagem descrito na Seção 2.8.6 da dissertação:
infraestrutura, parâmetros biológicos e eficiência de insumos (técnico);
CAPEX/OPEX, receitas e parâmetros financeiros (econômico); gestão hídrica,
energética e conformidade (ambiental); emprego, segurança e relação com a
comunidade (social); governança (G do ESG); pesos e incerteza.

Os valores numéricos *default* são ilustrativos. A origem de cada premissa é
registrada em ``EVTEASConfig.fontes`` (rastreabilidade exigida na Seção 3.6).
"""

from dataclasses import dataclass, field, fields, is_dataclass, asdict
from typing import Any, Dict, List, Optional

import numpy as np

ARTEFATO = "EVTEAS-Py"

# ---------------------------------------------------------------------------
# Rastreabilidade das premissas (Seção 3.6)
# ---------------------------------------------------------------------------

CATEGORIAS_FONTE = {
    "historico": "Dado histórico do empreendimento",
    "secundario": "Dado secundário (estatística oficial, relatório setorial)",
    "cotacao": "Cotação de mercado",
    "normativo": "Valor normativo (lei, resolução, norma técnica)",
    "bibliografico": "Parâmetro bibliográfico",
    "analogo": "Empreendimento análogo",
    "especialista": "Estimativa de especialista",
    "autor": "Estimativa do autor (a validar na V&V)",
}


@dataclass
class Fonte:
    """Origem declarada de uma premissa."""
    categoria: str = "autor"
    referencia: str = ""

    def validar(self) -> "Fonte":
        if self.categoria not in CATEGORIAS_FONTE:
            raise ValueError(f"Categoria de fonte desconhecida: {self.categoria}")
        return self


@dataclass
class Distribuicao:
    """Distribuição de incerteza de uma variável de entrada.

    ``uniforme`` quando apenas mínimo e máximo são conhecidos; ``triangular``
    quando também há um valor mais provável (Oliveira; Medeiros Neto, 2012);
    ``normal`` truncada em [mínimo, máximo] e ``fixa`` para desligar a
    incerteza de uma variável sem removê-la do registro.
    """
    tipo: str = "triangular"
    minimo: float = 0.0
    moda: float = 0.0
    maximo: float = 0.0

    def validar(self) -> "Distribuicao":
        t = self.tipo.lower()
        if t not in {"triangular", "uniforme", "normal", "fixa"}:
            raise ValueError(f"Distribuição não suportada: {self.tipo}")
        if self.minimo > self.maximo:
            raise ValueError("mínimo não pode ser maior que máximo")
        if t == "triangular" and not (self.minimo <= self.moda <= self.maximo):
            raise ValueError("na triangular, mínimo <= moda <= máximo")
        return self

    def amostrar(self, rng: np.random.Generator, n: int) -> np.ndarray:
        self.validar()
        t = self.tipo.lower()
        if t == "fixa" or self.maximo == self.minimo:
            return np.full(n, self.moda if t != "uniforme" else self.minimo, dtype=float)
        if t == "triangular":
            return rng.triangular(self.minimo, self.moda, self.maximo, n)
        if t == "uniforme":
            return rng.uniform(self.minimo, self.maximo, n)
        mu = self.moda if self.minimo <= self.moda <= self.maximo else (self.minimo + self.maximo) / 2
        sigma = max((self.maximo - self.minimo) / 3.92, 1e-12)  # ±1,96σ ~ [mín, máx]
        return np.clip(rng.normal(mu, sigma, n), self.minimo, self.maximo)

    def media(self) -> float:
        t = self.tipo.lower()
        if t == "triangular":
            return (self.minimo + self.moda + self.maximo) / 3
        if t == "uniforme":
            return (self.minimo + self.maximo) / 2
        return self.moda


# ---------------------------------------------------------------------------
# Dimensão técnica — infraestrutura, biologia e eficiência de insumos
# ---------------------------------------------------------------------------

# Referências indicativas por sistema produtivo (a calibrar pelo especialista em
# piscicultura). capacidade_suporte em kg/m³ ao final do ciclo; renovação em %
# do volume ao dia; energia em kWh por kg produzido (aeração + bombeamento).
SISTEMAS_PRODUTIVOS = {
    "extensivo": {"capacidade_suporte_kg_m3": 0.3, "renovacao_pct_dia": 1.0, "energia_kwh_kg": 0.05},
    "semi-intensivo": {"capacidade_suporte_kg_m3": 1.2, "renovacao_pct_dia": 3.0, "energia_kwh_kg": 0.60},
    "intensivo": {"capacidade_suporte_kg_m3": 3.0, "renovacao_pct_dia": 5.0, "energia_kwh_kg": 1.20},
    "superintensivo": {"capacidade_suporte_kg_m3": 25.0, "renovacao_pct_dia": 0.5, "energia_kwh_kg": 3.00},
}


@dataclass
class TecnicoConfig:
    # Bloco 1 — infraestrutura
    sistema_produtivo: str = "semi-intensivo"
    area_lamina_m2: float = 40000.0
    profundidade_media_m: float = 1.4
    numero_tanques: int = 32
    tanques_ativos: int = 30
    capacidade_suporte_kg_m3: Optional[float] = None   # None => referência do sistema
    produtividade_esperada_kg_m2_ciclo: float = 1.3     # restrição por área (0 = ignora)
    # Bloco 2 — parâmetros biológicos
    especie: str = "Tilápia-do-nilo (Oreochromis niloticus)"
    peso_inicial_g: float = 30.0
    peso_final_g: float = 800.0
    ciclo_dias: int = 180
    ciclos_ano: float = 1.8
    alevinos_por_ciclo: float = 0.0                     # 0 => dimensionado pelo modelo
    desempenho_crescimento_pct: float = 100.0           # peso atingido / peso-alvo
    # Bloco 3 — eficiência de insumos
    fcr: float = 1.50
    mortalidade_pct: float = 10.0
    fcr_referencia: float = 1.40                        # benchmark Lean-Green
    mortalidade_referencia_pct: float = 5.0

    @property
    def volume_util_m3(self) -> float:
        return self.area_lamina_m2 * self.profundidade_media_m


# ---------------------------------------------------------------------------
# Dimensão econômica — CAPEX, OPEX, receitas e parâmetros financeiros
# ---------------------------------------------------------------------------

def _capex_padrao() -> Dict[str, float]:
    return {
        "Escavação e construção de viveiros": 400000.0,
        "Sistema de abastecimento e drenagem": 70000.0,
        "Aeradores e quadro elétrico": 90000.0,
        "Bombas e tubulações": 35000.0,
        "Galpão, depósito de ração e equipamentos": 80000.0,
        "Lagoa de sedimentação / tratamento de efluentes": 45000.0,
        "Licenciamento, outorga e projetos": 25000.0,
    }


# Faixas de exatidão típicas (limite inferior, limite superior) por classe de
# estimativa AACE 18R-97, usadas para a incerteza do CAPEX.
FAIXAS_AACE = {5: (-0.30, 0.50), 4: (-0.20, 0.30), 3: (-0.15, 0.20), 2: (-0.10, 0.15), 1: (-0.05, 0.10)}


@dataclass
class EconomicoConfig:
    tipo_organizacao: str = "coop_agro"          # empresa | coop_trabalho | coop_agro
    capex_itens: Dict[str, float] = field(default_factory=_capex_padrao)
    classe_estimativa_aace: int = 5
    capital_giro: float = 60000.0
    vida_util_anos: float = 15.0                 # depreciação linear
    valor_residual_pct: float = 30.0             # % do CAPEX recuperável no fim
    valor_residual_rs: Optional[float] = None    # se informado em R$, prevalece sobre o percentual
    # Receitas
    preco_venda_kg: float = 10.50
    mix_produtos: List[Dict[str, Any]] = field(default_factory=list)  # [{nome, participacao_pct, preco_kg}]
    receita_servicos_mes: float = 0.0
    # OPEX variável
    custo_racao_kg: float = 3.10
    custo_alevino_milheiro: float = 280.0
    tarifa_energia_kwh: float = 0.75
    outros_custos_variaveis_kg: float = 0.35     # despesca, gelo, transporte, embalagem
    outros_custos_variaveis_mes: float = 0.0     # matéria-prima e insumos não listados (R$/mês a 100%)
    # OPEX fixo (mensal)
    mao_obra_mes: float = 12000.0                # salários / retiradas
    encargos_mao_obra_pct: float = 20.0
    assistencia_tecnica_mes: float = 2000.0
    administrativo_mes: float = 1500.0
    manutencao_capex_pct_aa: float = 2.0
    manutencao_mes: Optional[float] = None       # se informada em R$/mês, prevalece sobre o percentual
    custos_fixos_fator: float = 1.0              # multiplicador para incerteza
    # Tributos
    taxa_impostos_faturamento_pct: float = 2.3   # ex.: contribuições sobre a receita
    taxa_impostos_lucro_pct: float = 0.0         # cooperativas: ato cooperativo
    impostos_configurados: List[Dict[str, Any]] = field(default_factory=list)
    # Parâmetros financeiros
    tma_aa_pct: float = 10.0
    horizonte_anos: int = 10
    crescimento_preco_aa_pct: float = 0.0
    crescimento_custos_aa_pct: float = 0.0
    # Rampa de capacidade (aprendizado da equipe)
    capacidade_inicial_pct: float = 70.0
    meses_rampa: int = 12
    # Cooperativas
    cooperados_trabalhadores: int = 6
    cooperados_fornecedores: int = 0
    perc_sobras_trabalhadores: float = 100.0
    capex_fator: float = 1.0                     # multiplicador para incerteza (AACE)
    # Fomento não reembolsável (fundos mitigatórios/compensatórios, editais)
    fomento_nao_reembolsavel_pct: float = 0.0    # % do CAPEX doado ao beneficiário
    # Pessoal detalhado (CLT por cargo e retirada dos cooperados; somam-se a mao_obra_mes)
    salarios_clt_por_cargo: Dict[str, float] = field(default_factory=dict)   # salário bruto mensal por cargo
    retirada_cooperados_mes: float = 0.0          # pró-labore/retirada mensal total dos cooperados
    nome_materia_prima: str = ""
    # Vendas e calendário
    producao_vendas_kg_mes: float = 0.0           # 0 => produção calculada pelo módulo técnico
    meses_ate_primeira_receita: int = 0           # usado quando carencia_automatica = False (ou > 0)
    carencia_automatica: bool = True              # True => carência = duração do ciclo produtivo
    crescimento_vendas_aa_pct: float = 0.0

    @property
    def capex_total(self) -> float:
        return float(sum(self.capex_itens.values()))


# ---------------------------------------------------------------------------
# Dimensão ambiental — gestão hídrica, energética, efluentes e conformidade
# ---------------------------------------------------------------------------

# Checklist normativo do Score de Conformidade Ambiental (0–10). Os itens
# marcados como eliminatórios bloqueiam a classificação "VIÁVEL".
CHECKLIST_AMBIENTAL = {
    "licenca_ambiental": ("Licença ambiental do empreendimento aquícola (Resolução CONAMA nº 413/2009)", 2.0, True),
    "outorga_agua": ("Outorga de direito de uso de recursos hídricos (Lei nº 9.433/1997)", 1.5, True),
    "respeita_app": ("Instalações fora da Área de Preservação Permanente (Lei nº 12.651/2012)", 1.5, True),
    "tratamento_efluentes": ("Tratamento de efluentes antes do lançamento (CONAMA nº 357/2005 e 430/2011)", 1.5, False),
    "monitoramento_agua": ("Monitoramento periódico da qualidade da água", 1.0, False),
    "car": ("Inscrição no Cadastro Ambiental Rural", 0.5, False),
    "gestao_residuos": ("Plano de gestão de resíduos sólidos e lodo", 0.75, False),
    "prevencao_escape": ("Barreiras e plano de prevenção de escape de espécie exótica", 0.75, False),
    "energia_renovavel": ("Uso de energia renovável ou eficiência energética", 0.5, False),
}

# Limites de N e P total (mg/L) para águas doces classe 2 (Resolução CONAMA nº 357/2005)
CLASSES_CORPO_RECEPTOR = {
    "classe 2 — ambiente lótico": {"n_max_mg_l": 2.18, "p_max_mg_l": 0.10},
    "classe 2 — ambiente intermediário": {"n_max_mg_l": 2.18, "p_max_mg_l": 0.05},
    "classe 2 — ambiente lêntico": {"n_max_mg_l": 1.27, "p_max_mg_l": 0.03},
}

FATORES_EMISSAO_ENERGIA = {  # kgCO2e/kWh — atualizar ao ano-base do estudo
    "rede": 0.0817,
    "solar": 0.0,
    "biomassa": 0.0,
}


@dataclass
class AmbientalConfig:
    # Gestão hídrica
    renovacao_pct_dia: Optional[float] = None    # None => referência do sistema
    esvaziamento_por_ciclo: bool = True
    evaporacao_mm_dia: float = 4.0
    infiltracao_mm_dia: float = 2.0
    fracao_retorno_perdida: float = 0.0           # água devolvida a outra bacia
    # Balanço de nutrientes (pegada hídrica cinza — Hoekstra)
    proteina_racao_pct: float = 32.0
    fosforo_racao_pct: float = 1.0
    nitrogenio_peixe_pct: float = 2.6
    fosforo_peixe_pct: float = 0.7
    remocao_n_tratamento_pct: float = 40.0
    remocao_p_tratamento_pct: float = 60.0
    n_max_mg_l: float = 2.18                      # CONAMA 357/2005, classe 2 lótico
    p_max_mg_l: float = 0.10                      # CONAMA 357/2005, classe 2 lótico
    n_natural_mg_l: float = 0.50
    p_natural_mg_l: float = 0.02
    # Gestão energética e emissões
    energia_kwh_kg: Optional[float] = None        # None => referência do sistema
    fonte_energia: str = "rede"                   # rede | solar | biomassa | mista
    fracao_renovavel_mista: float = 0.0
    fator_emissao_rede_kgco2_kwh: float = 0.0817  # fator médio do SIN (atualizar ao ano-base)
    fator_emissao_racao_kgco2_kg: float = 1.1
    diesel_l_ano: float = 600.0
    fator_emissao_diesel_kgco2_l: float = 2.6
    # Resíduos
    residuos_kg_por_kg: float = 0.04
    residuos_reaproveitados_pct: float = 50.0
    # Conformidade
    distancia_app_m: float = 50.0
    faixa_app_minima_m: float = 30.0
    checklist: Dict[str, bool] = field(default_factory=lambda: {
        "licenca_ambiental": True, "outorga_agua": True, "respeita_app": True,
        "tratamento_efluentes": True, "monitoramento_agua": True, "car": True,
        "gestao_residuos": False, "prevencao_escape": True, "energia_renovavel": False,
    })
    # Referências para normalização / Lean-Green
    energia_referencia_kwh_kg: float = 0.40


# ---------------------------------------------------------------------------
# Dimensão social — Radar Social, LSO, RVL e ODS
# ---------------------------------------------------------------------------

RELACAO_COMUNIDADE = {"inexistente": 0.0, "parcial": 0.5, "consolidado": 1.0}


@dataclass
class SocialConfig:
    empregos_diretos: int = 6
    empregos_indiretos: int = 10
    mao_obra_local_pct: float = 100.0
    compras_locais_pct: float = 25.0            # % das compras de insumos feitas localmente
    salario_minimo: float = 1518.0
    capacitacao_horas_pessoa_ano: float = 40.0
    conformidade_nr_pct: float = 85.0
    relacao_comunidade: str = "parcial"          # inexistente | parcial | consolidado
    canais_formais_comunicacao: bool = True
    reunioes_comunidade_ano: int = 6
    conflitos_registrados_ano: int = 1
    mulheres_pct: float = 40.0
    jovens_pct: float = 20.0
    publico_vulneravel: bool = True              # pescadores artesanais / PGTR


@dataclass
class GovernancaConfig:
    """Checklist do pilar G (ESG)."""
    itens: Dict[str, bool] = field(default_factory=lambda: {
        "estatuto_regimento": True,
        "assembleias_regulares": True,
        "conselho_fiscal": True,
        "prestacao_contas_publica": False,
        "plano_negocios_aprovado": True,
        "registro_auditavel_premissas": True,
        "canal_denuncia_ouvidoria": False,
    })


# ---------------------------------------------------------------------------
# Ponderação, decisão e incerteza
# ---------------------------------------------------------------------------

KPIS_POR_DIMENSAO = {
    "tecnico": ["oee", "fcr", "produtividade"],
    "economico": ["vpl", "roi", "margem_seguranca"],
    "ambiental": ["ph_azul", "ph_cinza", "ecoeficiencia", "intensidade_carbono", "conformidade"],
    "social": ["lso", "rvl", "radar_social"],
}


@dataclass
class PesosConfig:
    metodo: str = "preset"                        # preset | likert | ahp
    tecnico: float = 0.25
    economico: float = 0.35
    ambiental: float = 0.20
    social: float = 0.20
    kpi: Dict[str, float] = field(default_factory=dict)  # vazio => pesos iguais na dimensão
    likert_dimensoes: Dict[str, Dict[str, int]] = field(default_factory=dict)  # {avaliador: {dim: 1..5}}
    likert_kpis: Dict[str, Dict[str, int]] = field(default_factory=dict)        # {avaliador: {kpi: 1..5}}
    ahp_matriz: Optional[List[List[float]]] = None                               # ordem: tec, eco, amb, soc

    def vetor(self) -> Dict[str, float]:
        v = np.array([self.tecnico, self.economico, self.ambiental, self.social], dtype=float)
        if np.any(v < 0) or v.sum() <= 0:
            raise ValueError("Pesos dimensionais inválidos")
        v = v / v.sum()
        return dict(zip(["tecnico", "economico", "ambiental", "social"], v.tolist()))


@dataclass
class DecisaoConfig:
    limiar_indice_viavel: float = 0.60
    limiar_indice_ressalvas: float = 0.50
    prob_vpl_positivo_minima: float = 0.70
    lso_minimo: float = 40.0
    aplicar_vetos: bool = True


def _distribuicoes_padrao() -> Dict[str, Distribuicao]:
    return {
        "economico.preco_venda_kg": Distribuicao("triangular", 9.00, 10.50, 11.50),
        "economico.custo_racao_kg": Distribuicao("triangular", 2.80, 3.10, 3.70),
        "economico.custo_alevino_milheiro": Distribuicao("triangular", 230.0, 280.0, 350.0),
        "economico.tarifa_energia_kwh": Distribuicao("triangular", 0.65, 0.75, 0.95),
        "economico.custos_fixos_fator": Distribuicao("triangular", 0.95, 1.00, 1.15),
        "tecnico.fcr": Distribuicao("triangular", 1.30, 1.50, 1.85),
        "tecnico.mortalidade_pct": Distribuicao("triangular", 5.0, 10.0, 20.0),
        "tecnico.desempenho_crescimento_pct": Distribuicao("triangular", 85.0, 100.0, 105.0),
    }


# Parâmetros que o motor lê pelo Contexto (vetorizados): só eles podem ter distribuição no Monte Carlo.
VARIAVEIS_ESTOCASTICAS = (
    "economico.preco_venda_kg", "economico.custo_racao_kg", "economico.custo_alevino_milheiro",
    "economico.tarifa_energia_kwh", "economico.custos_fixos_fator", "economico.capex_fator",
    "tecnico.fcr", "tecnico.mortalidade_pct", "tecnico.desempenho_crescimento_pct",
)


@dataclass
class MonteCarloConfig:
    iteracoes: int = 10000
    seed: int = 20260612
    distribuicoes: Dict[str, Distribuicao] = field(default_factory=_distribuicoes_padrao)
    incluir_capex_aace: bool = True               # adiciona economico.capex_fator pela classe AACE


@dataclass
class EVTEASConfig:
    projeto: str = "Projeto de Piscicultura — EVTEAS-Py"
    autor: str = "Rafael Alves Bastos"
    artefato: str = ARTEFATO
    tecnico: TecnicoConfig = field(default_factory=TecnicoConfig)
    economico: EconomicoConfig = field(default_factory=EconomicoConfig)
    ambiental: AmbientalConfig = field(default_factory=AmbientalConfig)
    social: SocialConfig = field(default_factory=SocialConfig)
    governanca: GovernancaConfig = field(default_factory=GovernancaConfig)
    pesos: PesosConfig = field(default_factory=PesosConfig)
    decisao: DecisaoConfig = field(default_factory=DecisaoConfig)
    monte_carlo: MonteCarloConfig = field(default_factory=MonteCarloConfig)
    fontes: Dict[str, Fonte] = field(default_factory=dict)  # "bloco.campo" -> Fonte


# ---------------------------------------------------------------------------
# Acesso por caminho ("bloco.campo") e (de)serialização
# ---------------------------------------------------------------------------

def obter_por_caminho(cfg: EVTEASConfig, caminho: str) -> Any:
    obj: Any = cfg
    for parte in caminho.split("."):
        obj = getattr(obj, parte)
    return obj


def definir_por_caminho(cfg: EVTEASConfig, caminho: str, valor: Any) -> None:
    partes = caminho.split(".")
    obj: Any = cfg
    for parte in partes[:-1]:
        obj = getattr(obj, parte)
    if not hasattr(obj, partes[-1]):
        raise AttributeError(f"Parâmetro inexistente: {caminho}")
    setattr(obj, partes[-1], valor)


def caminhos_numericos(cfg: EVTEASConfig) -> List[str]:
    """Lista os parâmetros numéricos escalares da configuração."""
    saida = []
    for bloco in ("tecnico", "economico", "ambiental", "social"):
        obj = getattr(cfg, bloco)
        for f in fields(obj):
            v = getattr(obj, f.name)
            if isinstance(v, (int, float)) and not isinstance(v, bool):
                saida.append(f"{bloco}.{f.name}")
    return saida


def config_para_dict(cfg: EVTEASConfig) -> Dict[str, Any]:
    return asdict(cfg)


def _construir(tipo, dados):
    if not is_dataclass(tipo) or not isinstance(dados, dict):
        return dados
    kwargs = {}
    for f in fields(tipo):
        if f.name not in dados:
            continue
        v = dados[f.name]
        if f.name == "distribuicoes":
            v = {k: Distribuicao(**d) for k, d in v.items()}
        elif f.name == "fontes":
            v = {k: Fonte(**d) for k, d in v.items()}
        elif isinstance(f.type, str) and f.type.endswith("Config") and isinstance(v, dict):
            v = _construir(globals()[f.type], v)
        kwargs[f.name] = v
    return tipo(**kwargs)


def config_de_dict(dados: Dict[str, Any]) -> EVTEASConfig:
    return _construir(EVTEASConfig, dados)

## 2. Engenharia econômica vetorizada (VPL, TIR com diagnóstico, payback)

In [ ]:
"""Utilitários numéricos e de engenharia econômica (vetorizados).

Todas as funções aceitam fluxos com forma (M+1,) ou (n, M+1), em que n é o
número de iterações de Monte Carlo. Assim o modelo determinístico e o
estocástico usam exatamente as mesmas fórmulas (verificação de consistência).
"""

from typing import Dict, Tuple

import numpy as np


def safe_float(x, default: float = 0.0) -> float:
    try:
        x = float(x)
        return x if np.isfinite(x) else default
    except (TypeError, ValueError):
        return default


def pct(x):
    return np.asarray(x, dtype=float) / 100.0


def safe_div(a, b, default=0.0):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        out = np.where(np.abs(b) < 1e-15, default, a / np.where(np.abs(b) < 1e-15, 1.0, b))
    return out if out.ndim else float(out)


def clamp(x, lo=0.0, hi=1.0):
    out = np.clip(np.nan_to_num(np.asarray(x, dtype=float), nan=lo), lo, hi)
    return out if out.ndim else float(out)


def taxa_anual_para_mensal(taxa_aa) -> np.ndarray:
    return (1 + np.asarray(taxa_aa, dtype=float)) ** (1 / 12) - 1


def taxa_mensal_para_anual(taxa_am) -> np.ndarray:
    return (1 + np.asarray(taxa_am, dtype=float)) ** 12 - 1


def _2d(fluxos) -> np.ndarray:
    f = np.asarray(fluxos, dtype=float)
    return f[None, :] if f.ndim == 1 else f


def vpl(taxa, fluxos):
    """Valor presente líquido; o índice 0 é a data focal (não descontado)."""
    f = _2d(fluxos)
    taxa = np.asarray(taxa, dtype=float).reshape(-1, 1)
    t = np.arange(f.shape[1])[None, :]
    with np.errstate(over="ignore", invalid="ignore"):
        out = (f / (1 + taxa) ** t).sum(axis=1)
    return float(out[0]) if np.ndim(fluxos) == 1 else out


def trocas_de_sinal(fluxos) -> np.ndarray:
    f = _2d(fluxos)
    s = np.sign(f)
    out = np.zeros(f.shape[0], dtype=int)
    for i in range(f.shape[0]):
        nz = s[i][s[i] != 0]
        out[i] = int(np.sum(nz[1:] != nz[:-1])) if nz.size > 1 else 0
    return out


def tir(fluxos, lo: float = -0.90, hi: float = 1.0, pontos: int = 160,
        iteracoes: int = 100) -> Tuple[np.ndarray, np.ndarray]:
    """TIR por varredura de malha + bisseção vetorizada, com diagnóstico explícito.

    Retorna (taxa por período, diagnóstico). A taxa é ``nan`` quando não há raiz
    no intervalo: nunca é forçada ao limite numérico (a ausência de raiz não
    pode ser confundida com uma taxa de -0,90). Diagnósticos: ``convencional`` (uma raiz e uma
    troca de sinal no fluxo), ``multiplas`` (mais de uma raiz na malha),
    ``multiplas_possiveis`` (uma raiz, mas mais de uma troca de sinal —
    regra de Descartes) e ``sem_solucao`` (o VPL não muda de sinal).
    """
    f = _2d(fluxos)
    n = f.shape[0]
    malha = np.unique(np.concatenate([np.linspace(lo, -0.5, 12), np.linspace(-0.5, hi, pontos)]))
    valores = np.vstack([vpl(np.full(n, r), f) for r in malha]).T          # (n, pontos)
    troca = (valores[:, :-1] * valores[:, 1:]) <= 0
    n_raizes = troca.sum(axis=1)
    existe = n_raizes > 0
    k = np.where(existe, troca.argmax(axis=1), 0)
    a, b = malha[k].astype(float), malha[k + 1].astype(float)
    fa = valores[np.arange(n), k]
    for _ in range(iteracoes):
        m = (a + b) / 2
        fm = vpl(m, f)
        esquerda = (fa * fm) <= 0
        b = np.where(esquerda, m, b)
        a = np.where(esquerda, a, m)
        fa = np.where(esquerda, fa, fm)
    raiz = np.where(existe, (a + b) / 2, np.nan)
    trocas = trocas_de_sinal(f)
    diag = np.where(~existe, "sem_solucao",
                    np.where(n_raizes > 1, "multiplas",
                             np.where(trocas > 1, "multiplas_possiveis", "convencional")))
    return raiz, diag


def payback(fluxos, taxa=None) -> np.ndarray:
    """Payback (simples se ``taxa`` é None, descontado caso contrário), em períodos.

    Retorna ``nan`` quando o investimento não é recuperado no horizonte.
    Interpolação linear dentro do período de recuperação.
    """
    f = _2d(fluxos)
    if taxa is not None:
        t = np.arange(f.shape[1])[None, :]
        f = f / (1 + np.asarray(taxa, dtype=float).reshape(-1, 1)) ** t
    acum = np.cumsum(f, axis=1)
    out = np.full(f.shape[0], np.nan)
    for i in range(f.shape[0]):
        idx = np.where((acum[i, 1:] >= 0) & (acum[i, :-1] < 0))[0]
        if idx.size:
            k = idx[0] + 1
            out[i] = (k - 1) + safe_div(-acum[i, k - 1], f[i, k], 0.0)
    return out


def percentis(valores, ps=(1, 5, 10, 25, 50, 75, 90, 95, 99)) -> Dict[str, float]:
    x = np.asarray(valores, dtype=float)
    x = x[np.isfinite(x)]
    if not x.size:
        return {f"p{p}": float("nan") for p in ps}
    return {f"p{p}": float(np.percentile(x, p)) for p in ps}

## 3. Núcleo de cálculo: dimensões técnica, econômica, ambiental, social e Lean-Green

In [ ]:
"""Núcleo de cálculo vetorizado das quatro dimensões do EVTEAS.

Cada função recebe um ``Contexto``, que devolve qualquer parâmetro numérico
como vetor de tamanho n. Com n = 1 o modelo é determinístico; com n igual ao número de iterações
ele é a própria simulação de Monte Carlo. Não existe um "modelo simplificado"
para a camada estocástica: o mesmo código produz os dois resultados.
"""

import math
from typing import Any, Dict, Optional

import numpy as np
import pandas as pd



class Contexto:
    """Acesso vetorizado aos parâmetros, com substituições (overrides) por caminho."""

    def __init__(self, cfg: EVTEASConfig, overrides: Optional[Dict[str, np.ndarray]] = None, n: int = 1):
        self.cfg = cfg
        self.overrides = overrides or {}
        self.n = int(n)
        desconhecidos = sorted(set(self.overrides) - set(VARIAVEIS_ESTOCASTICAS))
        if desconhecidos:
            raise ValueError("O modelo não amostra estes parâmetros (sem efeito no Monte Carlo): "
                             + ", ".join(desconhecidos))
        for k, v in self.overrides.items():
            if np.ndim(v) and len(v) != self.n:
                raise ValueError(f"Override '{k}' com tamanho {len(v)} != n={self.n}")

    def v(self, caminho: str) -> np.ndarray:
        if caminho not in VARIAVEIS_ESTOCASTICAS:
            raise KeyError(f"{caminho} não está em VARIAVEIS_ESTOCASTICAS (config.py)")
        if caminho in self.overrides:
            return np.broadcast_to(np.asarray(self.overrides[caminho], dtype=float), (self.n,)).astype(float)
        valor = obter_por_caminho(self.cfg, caminho)
        return np.full(self.n, float(valor), dtype=float)


def referencia_sistema(cfg: EVTEASConfig, chave: str) -> float:
    sistema = cfg.tecnico.sistema_produtivo
    if sistema not in SISTEMAS_PRODUTIVOS:
        raise ValueError(f"Sistema produtivo desconhecido: {sistema}")
    return float(SISTEMAS_PRODUTIVOS[sistema][chave])


def capacidade_suporte(cfg: EVTEASConfig) -> float:
    c = cfg.tecnico.capacidade_suporte_kg_m3
    return float(c) if c is not None else referencia_sistema(cfg, "capacidade_suporte_kg_m3")


def energia_kwh_kg(cfg: EVTEASConfig) -> float:
    e = cfg.ambiental.energia_kwh_kg
    return float(e) if e is not None else referencia_sistema(cfg, "energia_kwh_kg")


def renovacao_pct_dia(cfg: EVTEASConfig) -> float:
    r = cfg.ambiental.renovacao_pct_dia
    return float(r) if r is not None else referencia_sistema(cfg, "renovacao_pct_dia")


# ---------------------------------------------------------------------------
# 1. Dimensão técnica
# ---------------------------------------------------------------------------

def calcular_tecnico(ctx: Contexto) -> Dict[str, Any]:
    t = ctx.cfg.tecnico
    vol = t.area_lamina_m2 * t.profundidade_media_m
    disponibilidade = safe_div(t.tanques_ativos, t.numero_tanques, 0.0)
    cap = capacidade_suporte(ctx.cfg)

    mort = np.clip(ctx.v("tecnico.mortalidade_pct"), 0, 100)
    sobrevivencia = 1 - mort / 100
    desempenho = np.clip(ctx.v("tecnico.desempenho_crescimento_pct"), 0, None) / 100
    fcr = np.maximum(ctx.v("tecnico.fcr"), 0)
    ciclos = max(float(t.ciclos_ano), 0.0)

    # Restrição dupla da biomassa: capacidade de suporte (volume) e produtividade (área)
    limite_volume = vol * cap * disponibilidade
    limite_area = (t.area_lamina_m2 * t.produtividade_esperada_kg_m2_ciclo * disponibilidade
                   if t.produtividade_esperada_kg_m2_ciclo > 0 else np.inf)
    biomassa_alvo = min(limite_volume, limite_area)
    restricao = "capacidade de suporte (volume)" if limite_volume <= limite_area else "produtividade por área"

    peso_final_kg = t.peso_final_g / 1000
    sobrevivencia_planejada = 1 - t.mortalidade_pct / 100
    alevinos = (t.alevinos_por_ciclo if t.alevinos_por_ciclo > 0
                else safe_div(biomassa_alvo, peso_final_kg * sobrevivencia_planejada, 0.0))

    # A estocagem é decisão de projeto; a mortalidade e o crescimento são incertos.
    biomassa_despesca = np.minimum(alevinos * sobrevivencia * peso_final_kg * desempenho, limite_volume)
    biomassa_inicial = alevinos * t.peso_inicial_g / 1000
    ganho_ciclo = np.maximum(biomassa_despesca - biomassa_inicial, 0)
    racao_ciclo = ganho_ciclo * fcr

    producao_ano = biomassa_despesca * ciclos
    performance = np.minimum(desempenho, 1.0)
    oee = disponibilidade * performance * sobrevivencia     # Quadro 5: Qualidade = sobrevivência

    potencial_kg_m2_ano = cap * t.profundidade_media_m * min(ciclos, 365 / max(t.ciclo_dias, 1))
    produtividade_kg_m2_ano = safe_div(producao_ano, t.area_lamina_m2, 0.0)

    return {
        "especie": t.especie,
        "sistema_produtivo": t.sistema_produtivo,
        "volume_util_m3": vol,
        "capacidade_suporte_kg_m3": cap,
        "restricao_biomassa": restricao,
        "disponibilidade": np.full(ctx.n, disponibilidade),
        "performance": performance,
        "qualidade": sobrevivencia,
        "oee": oee,
        "sobrevivencia": sobrevivencia,
        "fcr": fcr,
        "alevinos_ciclo": np.full(ctx.n, alevinos),
        "alevinos_ano": np.full(ctx.n, alevinos * ciclos),
        "densidade_estocagem_peixes_m3": np.full(ctx.n, safe_div(alevinos, vol * disponibilidade, 0.0)),
        "biomassa_inicial_ciclo_kg": np.full(ctx.n, biomassa_inicial),
        "biomassa_despesca_ciclo_kg": biomassa_despesca,
        "ganho_biomassa_ciclo_kg": ganho_ciclo,
        "ganho_biomassa_ano_kg": ganho_ciclo * ciclos,
        "racao_ciclo_kg": racao_ciclo,
        "racao_ano_kg": racao_ciclo * ciclos,
        "racao_por_kg_produzido": safe_div(racao_ciclo, biomassa_despesca, 0.0),
        "producao_kg_ano": producao_ano,
        "producao_kg_mes": producao_ano / 12,
        "produtividade_kg_m2_ano": produtividade_kg_m2_ano,
        "produtividade_volumetrica_kg_m3_ciclo": safe_div(biomassa_despesca, vol, 0.0),
        "produtividade_relativa": clamp(safe_div(produtividade_kg_m2_ano, potencial_kg_m2_ano, 0.0), 0, 1),
        "ciclos_ano": ciclos,
    }


def curva_crescimento(cfg: EVTEASConfig, passo_dias: int = 7) -> pd.DataFrame:
    """Curva de crescimento e de arraçoamento de um ciclo (modelo de TCE).

    Peso: W(d) = W0 · exp(TCE · d), TCE = ln(Wf/W0)/ciclo. Mortalidade
    distribuída exponencialmente ao longo do ciclo. A ração diária é
    proporcional ao ganho de biomassa e calibrada para reproduzir o FCR.
    """
    t = cfg.tecnico
    ctx = Contexto(cfg)
    tec = calcular_tecnico(ctx)
    n0 = float(tec["alevinos_ciclo"][0])
    w0, wf = t.peso_inicial_g / 1000, t.peso_final_g / 1000 * t.desempenho_crescimento_pct / 100
    dias = np.arange(0, t.ciclo_dias + 1)
    tce = math.log(wf / w0) / t.ciclo_dias if wf > w0 > 0 else 0.0
    peso = w0 * np.exp(tce * dias)
    m_diaria = 1 - (1 - t.mortalidade_pct / 100) ** (1 / t.ciclo_dias)
    vivos = n0 * (1 - m_diaria) ** dias
    biomassa = vivos * peso
    ganho = np.diff(biomassa, prepend=biomassa[0]).clip(min=0)
    racao_dia = ganho * t.fcr
    df = pd.DataFrame({
        "Dia": dias, "Peso médio (g)": peso * 1000, "Peixes vivos": vivos,
        "Biomassa (kg)": biomassa, "Ração do dia (kg)": racao_dia,
        "Ração acumulada (kg)": np.cumsum(racao_dia),
        "TCE (%/dia)": tce * 100,
    })
    if passo_dias <= 1:
        return df
    return df[(df["Dia"] % passo_dias == 0) | (df["Dia"] == t.ciclo_dias)].reset_index(drop=True)


# ---------------------------------------------------------------------------
# 2. Dimensão econômica
# ---------------------------------------------------------------------------

def preco_medio(cfg: EVTEASConfig) -> float:
    mix = cfg.economico.mix_produtos
    if not mix:
        return float(cfg.economico.preco_venda_kg)
    part = np.array([float(p.get("participacao_pct", 0)) for p in mix])
    preco = np.array([float(p.get("preco_kg", cfg.economico.preco_venda_kg)) for p in mix])
    if part.sum() <= 0:
        raise ValueError("Participações do mix de produtos devem somar mais que zero")
    return float((part / part.sum() * preco).sum())


BASES_TRIBUTARIAS = {
    # base interna: apelidos aceitos (nomes usados nos notebooks de viabilidade econômica)
    "faturamento": ("faturamento", "faturamento_bruto"),
    "produtos": ("produtos",),
    "servicos": ("servicos", "serviços"),
    "lucro": ("lucro", "lucro_antes_ir_csll"),
    "folha": ("folha",),
    "folha_clt": ("folha_clt",),
    "pro_labore": ("pro_labore", "pro_labore_cooperados"),
}


def aliquotas_configuradas(cfg: EVTEASConfig) -> Dict[str, float]:
    """Consolida alíquotas individuais (ICMS, IPI, ISS, PIS, COFINS, CSLL, IRPJ, INSS, FGTS) por base."""
    base = {k: 0.0 for k in BASES_TRIBUTARIAS}
    base["faturamento"] = float(cfg.economico.taxa_impostos_faturamento_pct) / 100
    base["lucro"] = float(cfg.economico.taxa_impostos_lucro_pct) / 100
    base["folha"] = float(cfg.economico.encargos_mao_obra_pct) / 100
    apelidos = {a: k for k, v in BASES_TRIBUTARIAS.items() for a in v}
    for imp in cfg.economico.impostos_configurados:
        b = apelidos.get(imp.get("base", "faturamento"))
        if b is None:
            raise ValueError(f"Base tributária desconhecida: {imp.get('base')}")
        aliq = imp["aliquota_pct"] / 100 if "aliquota_pct" in imp else float(imp.get("aliquota", 0))
        base[b] += float(aliq)
    return base


def remuneracao_mensal(cfg: EVTEASConfig) -> Dict[str, float]:
    """Remuneração bruta mensal: CLT (por cargo), retirada dos cooperados e mão de obra genérica."""
    e = cfg.economico
    clt = float(sum(e.salarios_clt_por_cargo.values()))
    ret = float(e.retirada_cooperados_mes)
    outros = float(e.mao_obra_mes)
    return {"clt": clt, "retirada": ret, "outros": outros, "total": clt + ret + outros}


def custo_pessoal_mensal(cfg: EVTEASConfig, aliq: Dict[str, float]) -> float:
    r = remuneracao_mensal(cfg)
    return r["clt"] * (1 + aliq["folha_clt"]) + r["retirada"] * (1 + aliq["pro_labore"]) + r["outros"] * (1 + aliq["folha"])


def meses_ate_despesca(cfg: EVTEASConfig) -> int:
    e = cfg.economico
    if e.meses_ate_primeira_receita > 0 or not e.carencia_automatica:
        return int(max(e.meses_ate_primeira_receita, 0))
    return int(math.ceil(cfg.tecnico.ciclo_dias / 30.4375))


def calcular_economico(ctx: Contexto, tec: Dict[str, Any]) -> Dict[str, Any]:
    cfg, e = ctx.cfg, ctx.cfg.economico
    n = ctx.n
    H = int(max(e.horizonte_anos, 1))
    M = H * 12
    meses = np.arange(1, M + 1)
    ano = np.ceil(meses / 12).astype(int)
    L = meses_ate_despesca(cfg)
    if L >= M:
        raise ValueError(f"A carência até a primeira receita ({L} meses) deve ser menor que o horizonte ({M} meses)")
    aliq = aliquotas_configuradas(cfg)

    # Rampa de capacidade (aprendizado) aplicada à produção em engorda
    cap0 = e.capacidade_inicial_pct / 100
    rampa = cap0 + (1 - cap0) * np.minimum((meses - 1) / max(e.meses_rampa, 1), 1.0) if e.meses_rampa > 0 else np.ones(M)
    rampa = rampa[None, :]

    fator_preco = ((1 + e.crescimento_preco_aa_pct / 100) ** (ano - 1))[None, :]
    fator_custo = ((1 + e.crescimento_custos_aa_pct / 100) ** (ano - 1))[None, :]

    fator_vendas = ((1 + e.crescimento_vendas_aa_pct / 100) ** (ano - 1))[None, :]
    prod_mes = tec["producao_kg_mes"][:, None]
    if e.producao_vendas_kg_mes > 0:
        # volume de vendas informado no mix; a incerteza técnica é preservada pela razão amostra/base
        base_tec = float(calcular_tecnico(Contexto(cfg))["producao_kg_mes"][0])
        prod_mes = e.producao_vendas_kg_mes * safe_div(prod_mes, base_tec, 1.0)
    em_engorda = prod_mes * rampa * fator_vendas                    # kg/mês em crescimento
    vendida = np.zeros((n, M))
    vendida[:, L:] = em_engorda[:, : M - L]                        # despesca após o 1º ciclo

    escala_preco = safe_div(ctx.v("economico.preco_venda_kg"), e.preco_venda_kg, 1.0)
    preco = preco_medio(cfg) * escala_preco[:, None] * fator_preco
    receita_produtos = vendida * preco
    receita_servicos = np.where(meses > L, e.receita_servicos_mes, 0.0)[None, :] * fator_preco * np.ones((n, 1))
    receita = receita_produtos + receita_servicos

    custo_racao = em_engorda * tec["racao_por_kg_produzido"][:, None] * ctx.v("economico.custo_racao_kg")[:, None] * fator_custo
    # escala de volume: vendas informadas no mix e crescimento de vendas também movem a estocagem
    escala_volume = safe_div(prod_mes, tec["producao_kg_mes"][:, None], 1.0) * fator_vendas
    alevinos_mes = tec["alevinos_ano"][:, None] / 12 * rampa * escala_volume
    custo_alevinos = alevinos_mes * ctx.v("economico.custo_alevino_milheiro")[:, None] / 1000 * fator_custo
    kwh_kg = energia_kwh_kg(cfg)
    custo_energia = em_engorda * kwh_kg * ctx.v("economico.tarifa_energia_kwh")[:, None] * fator_custo
    # valor mensal informado "a 100% da capacidade": acompanha a rampa e o crescimento de vendas
    outros_var = (vendida * e.outros_custos_variaveis_kg
                  + e.outros_custos_variaveis_mes * rampa * fator_vendas) * fator_custo
    custos_variaveis = custo_racao + custo_alevinos + custo_energia + outros_var

    capex = e.capex_total * ctx.v("economico.capex_fator")
    fixos_fator = ctx.v("economico.custos_fixos_fator")[:, None]
    pessoal = custo_pessoal_mensal(cfg, aliq)
    if e.manutencao_mes is not None:
        # valor informado em R$/mês para o CAPEX estimado; acompanha o CAPEX sorteado no Monte Carlo
        manutencao = float(e.manutencao_mes) * ctx.v("economico.capex_fator")[:, None]
    else:
        manutencao = capex[:, None] * e.manutencao_capex_pct_aa / 100 / 12
    custos_fixos = (pessoal + e.assistencia_tecnica_mes + e.administrativo_mes + manutencao) * fixos_fator * fator_custo

    vida_meses = max(e.vida_util_anos, 0.0) * 12
    depreciacao = np.where(meses <= vida_meses, 1.0, 0.0)[None, :] * safe_div(capex, vida_meses, 0.0)[:, None]

    impostos_fat = (receita_produtos * (aliq["faturamento"] + aliq["produtos"])
                    + receita_servicos * (aliq["faturamento"] + aliq["servicos"]))
    receita_liquida = receita - impostos_fat
    lucro_bruto = receita_liquida - custos_variaveis
    ebitda = lucro_bruto - custos_fixos
    lair = ebitda - depreciacao
    impostos_lucro = np.maximum(lair, 0) * aliq["lucro"]
    lucro_liquido = lair - impostos_lucro
    fco = lucro_liquido + depreciacao

    investimento = capex + e.capital_giro
    residual = (float(e.valor_residual_rs) * ctx.v("economico.capex_fator") if e.valor_residual_rs is not None
                else capex * e.valor_residual_pct / 100)
    estoque_final = (custo_racao + custo_alevinos + custo_energia)[:, M - L:].sum(axis=1) if L > 0 else np.zeros(n)
    terminal = residual + e.capital_giro + estoque_final

    fluxo = np.zeros((n, M + 1))
    fluxo[:, 0] = -investimento
    fluxo[:, 1:] = fco
    fluxo[:, -1] += terminal

    tma_am = float(taxa_anual_para_mensal(e.tma_aa_pct / 100))
    vpl_v = vpl(np.full(n, tma_am), fluxo)
    tir_am, diag_tir = tir(fluxo)
    tir_aa = taxa_mensal_para_anual(tir_am)
    pb_s = payback(fluxo)
    pb_d = payback(fluxo, np.full(n, tma_am))
    il = safe_div(vpl_v + investimento, investimento, np.nan)

    # Perspectiva do beneficiário: parte do CAPEX aportada por fomento não reembolsável
    fomento = capex * e.fomento_nao_reembolsavel_pct / 100
    fluxo_benef = fluxo.copy()
    fluxo_benef[:, 0] += fomento
    vpl_benef = vpl(np.full(n, tma_am), fluxo_benef)
    tir_benef_am, _ = tir(fluxo_benef)

    # Ano de regime: primeiro ano inteiro com capacidade plena e vendas
    ano_regime = int(min(math.ceil((max(e.meses_rampa, 0) + L) / 12) + 1, H))
    sel = ano == ano_regime

    def anual(x):
        return x[:, sel].sum(axis=1)

    rec_r, var_r, fix_r = anual(receita), anual(custos_variaveis), anual(custos_fixos)
    imp_r, dep_r, ll_r = anual(impostos_fat), anual(depreciacao), anual(lucro_liquido)
    kg_r = anual(vendida)
    imc = safe_div(rec_r - imp_r - var_r, rec_r, 0.0)
    pe_receita = np.where(imc > 0, safe_div(fix_r + dep_r, np.where(imc > 0, imc, 1.0), np.nan), np.nan)
    margem_seg = np.where(np.isfinite(pe_receita), safe_div(rec_r - pe_receita, rec_r, np.nan), -1.0)
    valor_adicionado = rec_r - (anual(custo_racao) + anual(custo_alevinos) + anual(custo_energia) + anual(outros_var))

    return {
        "meses_ate_despesca": L,
        "ano_regime": ano_regime,
        "aliquotas": aliq,
        "preco_medio_kg": preco[:, 0],
        "capex": capex,
        "investimento_total": investimento,
        "fluxo_caixa": fluxo,
        "tma_mensal": tma_am,
        "vpl": vpl_v,
        "tir_mensal": tir_am,
        "tir_anual": tir_aa,
        "diagnostico_tir": diag_tir,
        "payback_simples_meses": pb_s,
        "payback_descontado_meses": pb_d,
        "indice_lucratividade": il,
        "fomento_nao_reembolsavel": fomento,
        "fluxo_caixa_beneficiario": fluxo_benef,
        "vpl_beneficiario": vpl_benef,
        "tir_anual_beneficiario": taxa_mensal_para_anual(tir_benef_am),
        "payback_descontado_beneficiario_meses": payback(fluxo_benef, np.full(n, tma_am)),
        "receita_regime": rec_r,
        "custos_variaveis_regime": var_r,
        "custos_fixos_regime": fix_r,
        "impostos_faturamento_regime": imp_r,
        "depreciacao_regime": dep_r,
        "lucro_liquido_regime": ll_r,
        "ebitda_regime": anual(ebitda),
        "kg_vendidos_regime": kg_r,
        "custo_racao_regime": anual(custo_racao),
        "custo_alevinos_regime": anual(custo_alevinos),
        "custo_energia_regime": anual(custo_energia),
        "outros_variaveis_regime": anual(outros_var),
        "pessoal_regime": anual(pessoal * fixos_fator * fator_custo * np.ones((n, M))),
        "remuneracao_regime": {k: anual(v * fixos_fator * fator_custo * np.ones((n, M)))
                               for k, v in remuneracao_mensal(cfg).items()},
        "fator_volume_regime": safe_div(anual(em_engorda), tec["producao_kg_ano"], 1.0),
        "valor_adicionado_regime": valor_adicionado,
        "indice_margem_contribuicao": imc,
        "ponto_equilibrio_receita_ano": pe_receita,
        "ponto_equilibrio_kg_ano": safe_div(pe_receita, safe_div(rec_r, kg_r, np.nan), np.nan),
        "margem_seguranca": margem_seg,
        "roi_anual_regime": safe_div(ll_r, investimento, np.nan),
        "roi_horizonte": safe_div(lucro_liquido.sum(axis=1), investimento, np.nan),
        "margem_liquida_regime": safe_div(ll_r, rec_r, np.nan),
        "custo_total_kg": safe_div(var_r + fix_r + dep_r + imp_r, kg_r, np.nan),
        "custo_operacional_efetivo_kg": safe_div(var_r + fix_r, kg_r, np.nan),
        "receita_total": receita.sum(axis=1),
        "lucro_liquido_total": lucro_liquido.sum(axis=1),
        "_series": {  # séries mensais (n, M) para a DRE
            "meses": meses, "ano": ano, "capacidade": np.broadcast_to(rampa, (n, M)),
            "producao_engorda": em_engorda, "vendida": vendida, "preco": preco,
            "receita_produtos": receita_produtos, "receita_servicos": receita_servicos,
            "receita": receita, "impostos_fat": impostos_fat, "receita_liquida": receita_liquida,
            "custo_racao": custo_racao, "custo_alevinos": custo_alevinos, "custo_energia": custo_energia,
            "outros_var": outros_var, "custos_variaveis": custos_variaveis, "lucro_bruto": lucro_bruto,
            "custos_fixos": custos_fixos, "ebitda": ebitda, "depreciacao": depreciacao, "lair": lair,
            "impostos_lucro": impostos_lucro, "lucro_liquido": lucro_liquido, "fco": fco,
        },
    }


def dre_mensal(eco: Dict[str, Any], i: int = 0) -> pd.DataFrame:
    s = eco["_series"]
    df = pd.DataFrame({
        "Mês": s["meses"], "Ano": s["ano"], "Capacidade (%)": s["capacidade"][i] * 100,
        "Produção em engorda (kg)": s["producao_engorda"][i], "Venda (kg)": s["vendida"][i],
        "Preço (R$/kg)": s["preco"][i], "Receita Produtos": s["receita_produtos"][i],
        "Receita Serviços": s["receita_servicos"][i], "Receita Bruta": s["receita"][i],
        "Impostos s/ Faturamento": s["impostos_fat"][i], "Receita Líquida": s["receita_liquida"][i],
        "Custo Ração": s["custo_racao"][i], "Custo Alevinos": s["custo_alevinos"][i],
        "Custo Energia": s["custo_energia"][i], "Outros Custos Variáveis": s["outros_var"][i],
        "Custos Variáveis": s["custos_variaveis"][i], "Lucro Bruto": s["lucro_bruto"][i],
        "Custos Fixos": s["custos_fixos"][i], "EBITDA": s["ebitda"][i], "Depreciação": s["depreciacao"][i],
        "LAIR": s["lair"][i], "Impostos s/ Lucro": s["impostos_lucro"][i],
        "Resultado Líquido": s["lucro_liquido"][i], "Fluxo de Caixa Operacional": s["fco"][i],
    })
    return df


def dre_anual(df_mensal: pd.DataFrame) -> pd.DataFrame:
    cols = [c for c in df_mensal.columns if c not in ("Mês", "Ano", "Capacidade (%)", "Preço (R$/kg)")]
    return df_mensal.groupby("Ano")[cols].sum().reset_index()


def distribuicao_sobras(cfg: EVTEASConfig, eco: Dict[str, Any]) -> Dict[str, float]:
    """Renda por cooperado (retirada + sobras) em regime — organizações cooperativas."""
    e = cfg.economico
    sobras = float(max(eco["lucro_liquido_regime"][0], 0.0))
    trab = sobras * e.perc_sobras_trabalhadores / 100
    forn = sobras - trab
    n_trab = max(e.cooperados_trabalhadores, 0)
    r = eco["remuneracao_regime"]            # em preços do ano de regime
    retirada_anual = float(r["retirada"][0] + r["outros"][0])
    renda_mensal = safe_div(retirada_anual + trab, n_trab * 12, 0.0) if n_trab else 0.0
    return {
        "termo_resultado": "Lucro líquido" if e.tipo_organizacao == "empresa" else "Sobras líquidas",
        "sobras_regime": sobras,
        "sobras_trabalhadores": trab,
        "sobras_fornecedores": forn,
        "sobra_media_trabalhador": safe_div(trab, n_trab, 0.0),
        "sobra_media_fornecedor": safe_div(forn, e.cooperados_fornecedores, 0.0),
        "renda_mensal_por_trabalhador": float(renda_mensal),
    }


# ---------------------------------------------------------------------------
# 3. Dimensão ambiental
# ---------------------------------------------------------------------------

def fator_emissao_energia(cfg: EVTEASConfig) -> float:
    a = cfg.ambiental
    fatores = dict(FATORES_EMISSAO_ENERGIA, rede=a.fator_emissao_rede_kgco2_kwh)
    if a.fonte_energia == "mista":
        return fatores["rede"] * (1 - a.fracao_renovavel_mista)
    if a.fonte_energia not in fatores:
        raise ValueError(f"Fonte de energia desconhecida: {a.fonte_energia}")
    return fatores[a.fonte_energia]


def score_conformidade(cfg: EVTEASConfig) -> Dict[str, Any]:
    a = cfg.ambiental
    itens = dict(a.checklist)
    itens["respeita_app"] = bool(itens.get("respeita_app", False)) and a.distancia_app_m >= a.faixa_app_minima_m
    total = sum(p for _, p, _ in CHECKLIST_AMBIENTAL.values())
    obtido = sum(p for k, (_, p, _) in CHECKLIST_AMBIENTAL.items() if itens.get(k, False))
    vetos = [CHECKLIST_AMBIENTAL[k][0] for k, (_, _, elim) in CHECKLIST_AMBIENTAL.items() if elim and not itens.get(k, False)]
    detalhe = [{"Item": d, "Peso": p, "Eliminatório": elim, "Atendido": bool(itens.get(k, False))}
               for k, (d, p, elim) in CHECKLIST_AMBIENTAL.items()]
    return {"score_0_10": 10 * obtido / total, "vetos": vetos, "detalhe": detalhe}


def calcular_ambiental(ctx: Contexto, tec: Dict[str, Any], eco: Dict[str, Any]) -> Dict[str, Any]:
    cfg, a, t = ctx.cfg, ctx.cfg.ambiental, ctx.cfg.tecnico
    # volume efetivo no ano de regime (vendas do mix e crescimento de vendas incluídos)
    fator = eco["fator_volume_regime"]
    prod = tec["producao_kg_ano"] * fator
    prod_t = prod / 1000
    vol = tec["volume_util_m3"]
    disp = float(tec["disponibilidade"][0])

    # Gestão hídrica (m³/ano)
    enchimento = vol * disp * t.ciclos_ano if a.esvaziamento_por_ciclo else vol * disp / max(eco_horizonte(cfg), 1)
    renovacao = vol * disp * renovacao_pct_dia(cfg) / 100 * 365
    evaporacao = a.evaporacao_mm_dia / 1000 * t.area_lamina_m2 * disp * 365
    infiltracao = a.infiltracao_mm_dia / 1000 * t.area_lamina_m2 * disp * 365
    captada = enchimento + renovacao + evaporacao + infiltracao
    azul = evaporacao + a.fracao_retorno_perdida * (enchimento + renovacao)
    ph_azul = safe_div(azul, prod_t, np.nan)

    # Balanço de nutrientes e pegada hídrica cinza (poluente crítico)
    racao = tec["racao_ano_kg"] * fator
    ganho = tec["ganho_biomassa_ano_kg"] * fator
    n_in = racao * a.proteina_racao_pct / 100 / 6.25
    p_in = racao * a.fosforo_racao_pct / 100
    n_ret = ganho * a.nitrogenio_peixe_pct / 100
    p_ret = ganho * a.fosforo_peixe_pct / 100
    n_carga = np.maximum(n_in - n_ret, 0) * (1 - a.remocao_n_tratamento_pct / 100)
    p_carga = np.maximum(p_in - p_ret, 0) * (1 - a.remocao_p_tratamento_pct / 100)
    dn = a.n_max_mg_l - a.n_natural_mg_l
    dp = a.p_max_mg_l - a.p_natural_mg_l
    if dn <= 0 or dp <= 0:
        raise ValueError("Concentração máxima deve superar a natural (pegada hídrica cinza indefinida)")
    cinza_n = n_carga * 1000 / dn       # kg -> g; mg/L = g/m³
    cinza_p = p_carga * 1000 / dp
    cinza = np.maximum(cinza_n, cinza_p)
    critico = np.where(cinza_p >= cinza_n, "fósforo total", "nitrogênio total")

    # Energia e emissões (proxy de pegada de carbono)
    energia = energia_kwh_kg(cfg) * prod
    fe = fator_emissao_energia(cfg)
    co2_energia = energia * fe
    co2_racao = racao * a.fator_emissao_racao_kgco2_kg
    co2_diesel = np.full(ctx.n, a.diesel_l_ano * a.fator_emissao_diesel_kgco2_l)
    co2_total = co2_energia + co2_racao + co2_diesel
    intensidade = safe_div(co2_total, prod, np.nan)

    residuos = prod * a.residuos_kg_por_kg
    conf = score_conformidade(cfg)

    va = eco["valor_adicionado_regime"]
    # Ecoeficiência (WBCSD): valor adicionado / influência ambiental. Indefinida se VA <= 0.
    eco_carbono = np.where(va > 0, safe_div(va, co2_total, np.nan), np.nan)
    eco_agua = np.where(va > 0, safe_div(va, azul, np.nan), np.nan)

    return {
        "producao_t_ano": prod_t,
        "agua_captada_m3_ano": captada,
        "agua_enchimento_m3_ano": np.full(ctx.n, enchimento),
        "agua_renovacao_m3_ano": np.full(ctx.n, renovacao),
        "agua_evaporacao_m3_ano": np.full(ctx.n, evaporacao),
        "agua_infiltracao_m3_ano": np.full(ctx.n, infiltracao),
        "ph_azul_m3_ano": np.full(ctx.n, azul) if np.ndim(azul) == 0 else azul,
        "ph_azul_m3_t": ph_azul,
        "indice_uso_agua_kg_m3": safe_div(prod, captada, 0.0),
        "n_aportado_kg": n_in, "n_retido_kg": n_ret, "n_lancado_kg": n_carga,
        "p_aportado_kg": p_in, "p_retido_kg": p_ret, "p_lancado_kg": p_carga,
        "ph_cinza_m3_ano": cinza,
        "ph_cinza_m3_t": safe_div(cinza, prod_t, np.nan),
        "poluente_critico": critico,
        "energia_kwh_ano": energia,
        "fator_emissao_energia": fe,
        "co2e_energia_t": co2_energia / 1000,
        "co2e_racao_t": co2_racao / 1000,
        "co2e_diesel_t": co2_diesel / 1000,
        "co2e_total_t_ano": co2_total / 1000,
        "intensidade_carbono_kgco2e_kg": intensidade,
        "residuos_kg_ano": residuos,
        "residuos_reaproveitados_kg_ano": residuos * a.residuos_reaproveitados_pct / 100,
        "valor_adicionado_regime": va,
        "ecoeficiencia_r_por_kgco2e": eco_carbono,
        "ecoeficiencia_r_por_m3_azul": eco_agua,
        "ecoeficiencia_status": np.where(va > 0, "definida", "não definida (valor adicionado ≤ 0)"),
        "conformidade_0_10": conf["score_0_10"],
        "conformidade_detalhe": conf["detalhe"],
        "vetos_ambientais": conf["vetos"],
    }


def eco_horizonte(cfg: EVTEASConfig) -> int:
    return int(cfg.economico.horizonte_anos)


# ---------------------------------------------------------------------------
# 4. Dimensão social e governança
# ---------------------------------------------------------------------------

def calcular_social(ctx: Contexto, eco: Dict[str, Any]) -> Dict[str, Any]:
    cfg, s = ctx.cfg, ctx.cfg.social
    rem = eco["remuneracao_regime"]          # remuneração anual em preços do ano de regime
    massa_anual = rem["total"]
    massa_local = massa_anual * s.mao_obra_local_pct / 100
    compras = (eco["custos_variaveis_regime"] + eco["custos_fixos_regime"] - eco["pessoal_regime"])
    compras_locais = np.maximum(compras, 0) * s.compras_locais_pct / 100
    custo_operacional = eco["custos_variaveis_regime"] + eco["custos_fixos_regime"]
    rvl = 100 * safe_div(massa_local + compras_locais, custo_operacional, 0.0)

    sobras = np.maximum(eco["lucro_liquido_regime"], 0) * cfg.economico.perc_sobras_trabalhadores / 100
    if cfg.economico.tipo_organizacao == "empresa":
        sobras = np.zeros(ctx.n)
    renda_local = massa_local + sobras * s.mao_obra_local_pct / 100
    iil = safe_div(renda_local, eco["receita_regime"], 0.0)
    if cfg.economico.tipo_organizacao != "empresa" and cfg.economico.cooperados_trabalhadores > 0:
        n_trab = cfg.economico.cooperados_trabalhadores
        renda_mensal = (rem["retirada"] + rem["outros"] + sobras) / n_trab / 12
    else:   # empresa, ou cooperativa sem cooperados na operação: renda paga a quem trabalha
        n_trab = max(s.empregos_diretos, 1)
        renda_mensal = (rem["clt"] + rem["outros"] + rem["retirada"]) / n_trab / 12
    # renda em preços do ano de regime: o salário mínimo é levado ao mesmo nível de preços
    sm_regime = s.salario_minimo * (1 + cfg.economico.crescimento_custos_aa_pct / 100) ** (eco["ano_regime"] - 1)
    renda_sm = renda_mensal / sm_regime

    relacao = RELACAO_COMUNIDADE.get(s.relacao_comunidade)
    if relacao is None:
        raise ValueError(f"relacao_comunidade deve ser uma de {list(RELACAO_COMUNIDADE)}")
    participacao = 1.0 if cfg.governanca.itens.get("assembleias_regulares") else 0.0
    componentes_lso = {
        "Relação com a comunidade": relacao,
        "Canais formais de comunicação": 1.0 if s.canais_formais_comunicacao else 0.0,
        "Frequência de engajamento": min(s.reunioes_comunidade_ano / 12, 1.0),
        "Ausência de conflitos": math.exp(-0.5 * max(s.conflitos_registrados_ano, 0)),
        "Participação na governança": participacao,
    }
    lso = 100 * float(np.mean(list(componentes_lso.values())))
    risco = 1 / (1 + math.exp(8 * (lso / 100 - 0.5)))   # heurística logística (a calibrar)

    radar = {
        "Emprego local": np.full(ctx.n, s.mao_obra_local_pct / 100),
        "Renda": clamp((renda_sm - 0.5) / 1.5),
        "Segurança (NR)": np.full(ctx.n, s.conformidade_nr_pct / 100),
        "Capacitação": np.full(ctx.n, min(s.capacitacao_horas_pessoa_ano / 40, 1.0)),
        "Relação com a comunidade": np.full(ctx.n, relacao),
        "Inclusão": np.full(ctx.n, 0.5 * min(s.mulheres_pct / 50, 1) + 0.5 * min(s.jovens_pct / 30, 1)),
    }
    radar_score = np.mean(np.vstack(list(radar.values())), axis=0)

    return {
        "empregos_diretos": s.empregos_diretos,
        "empregos_indiretos": s.empregos_indiretos,
        "empregos_totais": s.empregos_diretos + s.empregos_indiretos,
        "massa_salarial_anual": massa_anual,
        "massa_salarial_local": massa_local,
        "compras_locais_regime": compras_locais,
        "rvl_pct": rvl,
        "indice_impacto_local": iil,
        "renda_mensal_trabalhador": renda_mensal,
        "renda_em_salarios_minimos": renda_sm,
        "lso_0_100": np.full(ctx.n, lso),
        "lso_componentes": componentes_lso,
        "risco_social": np.full(ctx.n, risco),
        "radar_social": radar,
        "radar_social_score": radar_score,
    }


def calcular_governanca(cfg: EVTEASConfig) -> Dict[str, Any]:
    itens = cfg.governanca.itens
    return {"score_0_1": safe_div(sum(bool(v) for v in itens.values()), len(itens), 0.0), "itens": dict(itens)}


def alinhamento_ods(cfg: EVTEASConfig, tec, eco, amb, soc, gov, i: int = 0):
    """Identificação automática dos ODS atendidos (regras explícitas e auditáveis)."""
    a, s = cfg.ambiental, cfg.social
    regras = [
        ("ODS 1 — Erradicação da pobreza", s.publico_vulneravel and soc["renda_em_salarios_minimos"][i] >= 1.0,
         "renda média por trabalhador ≥ 1 salário mínimo em público vulnerável"),
        ("ODS 2 — Fome zero e agricultura sustentável", tec["producao_kg_ano"][i] > 0,
         "produção de proteína animal de base aquícola"),
        ("ODS 5 — Igualdade de gênero", s.mulheres_pct >= 30, "participação feminina ≥ 30%"),
        ("ODS 6 — Água potável e saneamento", bool(a.checklist.get("tratamento_efluentes")) and bool(a.checklist.get("outorga_agua")),
         "tratamento de efluentes e outorga de uso da água"),
        ("ODS 7 — Energia limpa", a.fonte_energia in ("solar", "biomassa") or (a.fonte_energia == "mista" and a.fracao_renovavel_mista >= 0.5),
         "fonte de energia predominantemente renovável"),
        ("ODS 8 — Trabalho decente e crescimento econômico", s.conformidade_nr_pct >= 80 and soc["renda_em_salarios_minimos"][i] >= 1.0,
         "conformidade com NR ≥ 80% e renda ≥ 1 salário mínimo"),
        ("ODS 12 — Consumo e produção responsáveis", tec["fcr"][i] <= cfg.tecnico.fcr_referencia * 1.15 and a.residuos_reaproveitados_pct >= 50,
         "FCR até 15% acima da referência e reaproveitamento de resíduos ≥ 50%"),
        ("ODS 13 — Ação contra a mudança do clima", amb["intensidade_carbono_kgco2e_kg"][i] <= 2.0,
         "intensidade de carbono ≤ 2,0 kgCO2e/kg"),
        ("ODS 14 — Vida na água (meta 14.b: pescadores artesanais)", s.publico_vulneravel,
         "acesso de pescadores artesanais a recursos e mercados"),
        ("ODS 15 — Vida terrestre", a.distancia_app_m >= a.faixa_app_minima_m and bool(a.checklist.get("prevencao_escape")),
         "APP preservada e prevenção de escape de espécie exótica"),
        ("ODS 16 — Instituições eficazes", gov["score_0_1"] >= 0.7, "governança cooperativa ≥ 70% do checklist"),
    ]
    return pd.DataFrame([{"ODS": o, "Atendido": bool(c), "Critério": crit} for o, c, crit in regras])


# ---------------------------------------------------------------------------
# 5. Lean-Green: custo dos desperdícios evitáveis
# ---------------------------------------------------------------------------

def calcular_lean_green(ctx: Contexto, tec, eco) -> Dict[str, Any]:
    cfg, t = ctx.cfg, ctx.cfg.tecnico
    custo_racao = ctx.v("economico.custo_racao_kg")
    fator = eco["fator_volume_regime"]
    racao_exc = np.maximum(tec["fcr"] - t.fcr_referencia, 0) * tec["ganho_biomassa_ano_kg"] * fator
    mort = ctx.v("tecnico.mortalidade_pct")
    peixes_perdidos = np.maximum(mort - t.mortalidade_referencia_pct, 0) / 100 * tec["alevinos_ano"] * fator
    # perdas ocorrem, em média, na metade do ciclo (metade do peso final e da ração)
    custo_peixe = (ctx.v("economico.custo_alevino_milheiro") / 1000
                   + 0.5 * t.peso_final_g / 1000 * tec["fcr"] * custo_racao)
    kwh = energia_kwh_kg(cfg)
    energia_exc = max(kwh - cfg.ambiental.energia_referencia_kwh_kg, 0) * tec["producao_kg_ano"] * fator
    custos = {
        "Ração excedente (superalimentação/FCR)": racao_exc * custo_racao,
        "Mortalidade evitável": peixes_perdidos * custo_peixe,
        "Energia excedente": energia_exc * ctx.v("economico.tarifa_energia_kwh"),
    }
    total = sum(custos.values())
    opex = eco["custos_variaveis_regime"] + eco["custos_fixos_regime"]
    return {"desperdicios_r_ano": custos, "total_r_ano": total, "pct_opex": safe_div(total, opex, 0.0),
            "racao_excedente_kg": racao_exc, "peixes_perdidos_evitaveis": peixes_perdidos,
            "energia_excedente_kwh": energia_exc}

## 4. Normalização, pesos (preset, Likert, AHP), índice EVTEAS, decisão e TOPSIS

In [ ]:
"""Normalização dos KPIs, ponderação (preset, Likert, AHP), índice EVTEAS,
regras de decisão e TOPSIS para comparação de alternativas."""

from typing import Any, Dict, List, Optional, Sequence

import numpy as np
import pandas as pd


DIMENSOES = ["tecnico", "economico", "ambiental", "social"]

# Referências de normalização explícitas (auditáveis na V&V). sentido: +1
# benefício (maior é melhor), -1 custo (menor é melhor).
REFERENCIAS_NORMALIZACAO = {
    "oee": (+1, 0.0, 1.0, "fração (Nakajima, 1988)"),
    "fcr": (-1, 1.0, 2.5, "kg ração/kg ganho"),
    "produtividade": (+1, 0.0, 1.0, "fração do potencial do sistema"),
    "vpl": (+1, -1.0, 1.0, "VPL / investimento total"),
    "roi": (+1, -0.10, 0.30, "lucro líquido anual / investimento"),
    "margem_seguranca": (+1, 0.0, 0.50, "fração da receita"),
    "ph_azul": (-1, 0.0, 3000.0, "m³/t"),
    "ph_cinza": (-1, 0.0, 100000.0, "m³/t"),
    "ecoeficiencia": (+1, 0.0, 10.0, "R$ de valor adicionado/kgCO2e"),
    "intensidade_carbono": (-1, 0.0, 4.0, "kgCO2e/kg"),
    "conformidade": (+1, 0.0, 10.0, "score 0–10"),
    "lso": (+1, 0.0, 100.0, "score 0–100 (Hurst; Ihlen, 2018)"),
    "rvl": (+1, 0.0, 100.0, "% do custo operacional (GRI 201-1/204-1)"),
    "radar_social": (+1, 0.0, 1.0, "média dos eixos do Radar Social"),
}


def normalizar(valor, sentido: int, minimo: float, maximo: float):
    v = np.nan_to_num(np.asarray(valor, dtype=float), nan=(minimo if sentido > 0 else maximo))
    if maximo <= minimo:
        return np.full_like(v, 0.5)
    z = (v - minimo) / (maximo - minimo) if sentido > 0 else (maximo - v) / (maximo - minimo)
    return np.clip(z, 0, 1)


def kpis_brutos(tec, eco, amb, soc) -> Dict[str, np.ndarray]:
    return {
        "oee": tec["oee"],
        "fcr": tec["fcr"],
        "produtividade": tec["produtividade_relativa"],
        "vpl": safe_div(eco["vpl"], eco["investimento_total"], 0.0),
        "roi": eco["roi_anual_regime"],
        "margem_seguranca": eco["margem_seguranca"],
        "ph_azul": amb["ph_azul_m3_t"],
        "ph_cinza": amb["ph_cinza_m3_t"],
        "ecoeficiencia": amb["ecoeficiencia_r_por_kgco2e"],   # NaN (indefinida) -> 0
        "intensidade_carbono": amb["intensidade_carbono_kgco2e_kg"],
        "conformidade": np.full_like(tec["oee"], amb["conformidade_0_10"]),
        "lso": soc["lso_0_100"],
        "rvl": soc["rvl_pct"],
        "radar_social": soc["radar_social_score"],
    }


# ---------------------------------------------------------------------------
# Pesos: Likert (Quadro 7), AHP e preset
# ---------------------------------------------------------------------------

def pesos_likert(avaliacoes: Dict[str, Dict[str, int]]) -> Dict[str, Any]:
    """Pesos relativos a partir da escala Likert de 1 a 5 (Seção 3.4).

    Para cada avaliador, peso = escore / soma dos escores; o peso final é a
    média entre avaliadores (triangulação). Informa a divergência por critério.
    """
    if not avaliacoes:
        raise ValueError("Nenhuma avaliação Likert informada")
    df = pd.DataFrame(avaliacoes).astype(float)          # linhas: critérios; colunas: avaliadores
    if df.isna().any().any():
        raise ValueError("Todos os avaliadores devem pontuar todos os critérios")
    if ((df < 1) | (df > 5)).any().any() or (df % 1 != 0).any().any():
        raise ValueError("Escores Likert devem ser inteiros entre 1 e 5")
    pesos_av = df / df.sum(axis=0)
    pesos = pesos_av.mean(axis=1)
    pesos = pesos / pesos.sum()
    tabela = df.copy()
    tabela.columns = [f"Escore — {c}" for c in df.columns]
    tabela["Divergência (máx − mín)"] = df.max(axis=1) - df.min(axis=1)
    tabela["Peso normalizado"] = pesos
    return {"pesos": pesos.to_dict(), "tabela": tabela.reset_index().rename(columns={"index": "Critério"}),
            "concordancia_media": float(1 - (df.max(axis=1) - df.min(axis=1)).mean() / 4)}


INDICE_ALEATORIO = {1: 0.0, 2: 0.0, 3: 0.58, 4: 0.90, 5: 1.12, 6: 1.24, 7: 1.32, 8: 1.41, 9: 1.45, 10: 1.49}


def pesos_ahp(matriz: Sequence[Sequence[float]]) -> Dict[str, Any]:
    """Pesos AHP (autovetor principal) e razão de consistência (Saaty)."""
    A = np.asarray(matriz, dtype=float)
    if A.ndim != 2 or A.shape[0] != A.shape[1]:
        raise ValueError("Matriz AHP deve ser quadrada")
    if np.any(A <= 0) or not np.allclose(A * A.T, 1.0, atol=1e-6):
        raise ValueError("Matriz AHP deve ser positiva e recíproca (a_ji = 1/a_ij)")
    vals, vecs = np.linalg.eig(A)
    k = int(np.argmax(vals.real))
    lam = float(vals.real[k])
    w = np.abs(vecs[:, k].real)
    w = w / w.sum()
    n = A.shape[0]
    ci = (lam - n) / (n - 1) if n > 1 else 0.0
    ri = INDICE_ALEATORIO.get(n, 1.49)
    cr = ci / ri if ri > 0 else 0.0
    return {"pesos": w, "lambda_max": lam, "ic": ci, "rc": cr, "consistente": cr <= 0.10}


def resolver_pesos(cfg: EVTEASConfig) -> Dict[str, Any]:
    p = cfg.pesos
    info: Dict[str, Any] = {"metodo": p.metodo}
    if p.metodo == "preset":
        dim = p.vetor()
    elif p.metodo == "likert":
        lk = pesos_likert(p.likert_dimensoes)
        faltando = set(DIMENSOES) - set(lk["pesos"])
        if faltando:
            raise ValueError(f"Likert sem as dimensões: {faltando}")
        dim = {d: lk["pesos"][d] for d in DIMENSOES}
        info["likert_dimensoes"] = lk
    elif p.metodo == "ahp":
        if p.ahp_matriz is None:
            raise ValueError("Método AHP exige ahp_matriz (ordem: técnico, econômico, ambiental, social)")
        ahp = pesos_ahp(p.ahp_matriz)
        dim = dict(zip(DIMENSOES, ahp["pesos"].tolist()))
        info["ahp"] = ahp
    else:
        raise ValueError(f"Método de ponderação desconhecido: {p.metodo}")

    kpi: Dict[str, float] = {}
    if p.likert_kpis and p.metodo == "likert":
        lk = pesos_likert(p.likert_kpis)
        info["likert_kpis"] = lk
        kpi_bruto = lk["pesos"]
    else:
        kpi_bruto = p.kpi
    for d, nomes in KPIS_POR_DIMENSAO.items():
        w = np.array([max(float(kpi_bruto.get(k, 1.0)), 0.0) for k in nomes])
        w = w / w.sum() if w.sum() > 0 else np.full(len(nomes), 1 / len(nomes))
        kpi.update(dict(zip(nomes, w.tolist())))
    info["dimensoes"] = dim
    info["kpi_intra_dimensao"] = kpi
    return info


# ---------------------------------------------------------------------------
# Índice integrado, ESG e decisão
# ---------------------------------------------------------------------------

def calcular_indice(cfg: EVTEASConfig, tec, eco, amb, soc, gov, pesos=None) -> Dict[str, Any]:
    pesos = pesos or resolver_pesos(cfg)
    brutos = kpis_brutos(tec, eco, amb, soc)
    norm = {k: normalizar(v, *REFERENCIAS_NORMALIZACAO[k][:3]) for k, v in brutos.items()}
    dims = {}
    for d, nomes in KPIS_POR_DIMENSAO.items():
        dims[d] = sum(norm[k] * pesos["kpi_intra_dimensao"][k] for k in nomes)
    indice = sum(dims[d] * pesos["dimensoes"][d] for d in DIMENSOES)
    esg = {"E": dims["ambiental"], "S": dims["social"], "G": np.full_like(indice, gov["score_0_1"])}
    esg_score = (esg["E"] + esg["S"] + esg["G"]) / 3
    return {"kpis_brutos": brutos, "kpis_normalizados": norm, "dimensoes": dims,
            "indice_evteas": np.clip(indice, 0, 1), "esg": esg, "esg_score": esg_score, "pesos": pesos}


def classificar(cfg: EVTEASConfig, vpl: float, indice: float, lso: float, vetos_ambientais: List[str],
                prob_vpl_positivo: Optional[float] = None) -> Dict[str, Any]:
    """Regra de decisão em dois estágios: vetos não compensatórios + índice compensatório.

    O índice agregado sozinho pode ocultar fragilidades;
    por isso VPL negativo, item ambiental eliminatório não atendido ou LSO abaixo
    do mínimo impedem a classificação "VIÁVEL", qualquer que seja o índice.
    """
    d = cfg.decisao

    def br(x, casas):                               # número no padrão brasileiro (mensagens vão para o texto)
        return f"{float(x):.{casas}f}".replace(".", ",")

    motivos = []
    if d.aplicar_vetos:
        if vpl < 0:
            motivos.append("VPL negativo (veto econômico)")
        motivos += [f"Item eliminatório não atendido: {v}" for v in vetos_ambientais]
        if lso < d.lso_minimo:
            motivos.append(f"LSO abaixo do mínimo ({br(lso, 1)} < {br(d.lso_minimo, 0)})")
    ressalvas = []
    if prob_vpl_positivo is not None and prob_vpl_positivo < d.prob_vpl_positivo_minima:
        ressalvas.append(f"P(VPL > 0) de {br(100 * prob_vpl_positivo, 1)}%, abaixo do mínimo de "
                         f"{br(100 * d.prob_vpl_positivo_minima, 0)}%")
    if motivos:
        classe = "NÃO VIÁVEL"
    elif indice >= d.limiar_indice_viavel and not ressalvas:
        classe = "VIÁVEL"
    elif indice >= d.limiar_indice_ressalvas:
        classe = "VIÁVEL COM RESSALVAS"
        if indice < d.limiar_indice_viavel:
            ressalvas.append(f"índice EVTEAS de {br(indice, 3)}, abaixo do limiar de {br(d.limiar_indice_viavel, 2)}")
    else:
        classe = "NÃO VIÁVEL"
        motivos.append(f"índice EVTEAS de {br(indice, 3)}, abaixo de {br(d.limiar_indice_ressalvas, 2)}")
    return {"classificacao": classe, "vetos": motivos, "ressalvas": ressalvas}


# ---------------------------------------------------------------------------
# TOPSIS — comparação de alternativas de projeto
# ---------------------------------------------------------------------------

def topsis(df: pd.DataFrame, criterios: List[str], pesos: Sequence[float], beneficio: Sequence[bool]) -> pd.DataFrame:
    X = df[criterios].astype(float).to_numpy()
    if np.isnan(X).any():
        raise ValueError("TOPSIS não aceita valores ausentes na matriz de decisão")
    W = np.asarray(pesos, dtype=float)
    W = W / W.sum()
    den = np.sqrt((X ** 2).sum(axis=0))
    den[den == 0] = 1
    V = X / den * W
    ben = np.asarray(beneficio, dtype=bool)
    ideal = np.where(ben, V.max(axis=0), V.min(axis=0))
    anti = np.where(ben, V.min(axis=0), V.max(axis=0))
    d_mais = np.sqrt(((V - ideal) ** 2).sum(axis=1))
    d_menos = np.sqrt(((V - anti) ** 2).sum(axis=1))
    out = df.copy()
    out["D+"] = d_mais
    out["D-"] = d_menos
    out["TOPSIS"] = d_menos / (d_mais + d_menos + 1e-15)
    out["Ranking"] = out["TOPSIS"].rank(ascending=False, method="min").astype(int)
    return out.sort_values("Ranking").reset_index(drop=True)

## 5. Incerteza: Monte Carlo, sensibilidade, valores críticos e cenários

In [ ]:
"""Tratamento da incerteza: Monte Carlo, sensibilidade, valores críticos e cenários."""

import copy
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd



def distribuicoes_efetivas(cfg: EVTEASConfig) -> Dict[str, Distribuicao]:
    d = dict(cfg.monte_carlo.distribuicoes)
    if cfg.monte_carlo.incluir_capex_aace and "economico.capex_fator" not in d:
        lo, hi = FAIXAS_AACE.get(int(cfg.economico.classe_estimativa_aace), FAIXAS_AACE[5])
        d["economico.capex_fator"] = Distribuicao("triangular", 1 + lo, 1.0, 1 + hi)
    for caminho, dist in d.items():
        obter_por_caminho(cfg, caminho)  # valida o caminho
        if caminho not in VARIAVEIS_ESTOCASTICAS:
            raise ValueError(f"Distribuição em '{caminho}', parâmetro que o modelo não amostra no Monte Carlo. "
                             f"Parâmetros aceitos: {', '.join(VARIAVEIS_ESTOCASTICAS)}")
        dist.validar()
    return d


def amostrar(cfg: EVTEASConfig, n: int, seed: int) -> Dict[str, np.ndarray]:
    rng = np.random.default_rng(seed)
    # ordem fixa (alfabética) => reprodutibilidade independente da ordem de inserção
    return {k: dist.amostrar(rng, n) for k, dist in sorted(distribuicoes_efetivas(cfg).items())}


def monte_carlo(cfg: EVTEASConfig, n: Optional[int] = None, seed: Optional[int] = None) -> Dict[str, Any]:

    n = int(n or cfg.monte_carlo.iteracoes)
    seed = int(cfg.monte_carlo.seed if seed is None else seed)
    entradas = amostrar(cfg, n, seed)
    r = motor(cfg, entradas, n)
    eco, tec, amb, idx = r["economico"], r["tecnico"], r["ambiental"], r["indice"]
    tma = cfg.economico.tma_aa_pct / 100

    amostra = pd.DataFrame({k: v for k, v in entradas.items()})
    amostra["VPL"] = eco["vpl"]
    amostra["TIR_aa"] = eco["tir_anual"]
    amostra["Diagnostico_TIR"] = eco["diagnostico_tir"]
    amostra["Payback_desc_meses"] = eco["payback_descontado_meses"]
    amostra["IL"] = eco["indice_lucratividade"]
    amostra["VPL_beneficiario"] = eco["vpl_beneficiario"]
    amostra["ROI_anual"] = eco["roi_anual_regime"]
    amostra["Producao_kg_ano"] = tec["producao_kg_ano"]
    amostra["OEE"] = tec["oee"]
    amostra["PH_cinza_m3_t"] = amb["ph_cinza_m3_t"]
    amostra["Intensidade_kgCO2e_kg"] = amb["intensidade_carbono_kgco2e_kg"]
    amostra["Indice_EVTEAS"] = idx["indice_evteas"]

    v = amostra["VPL"].to_numpy()
    p5 = np.percentile(v, 5)
    media, dp = float(v.mean()), float(v.std(ddof=1))
    erro = dp / np.sqrt(n)
    tir_valida = amostra["TIR_aa"].to_numpy()
    tir_valida = tir_valida[np.isfinite(tir_valida)]
    estat = {
        "n": n, "seed": seed,
        "media_vpl": media, "mediana_vpl": float(np.median(v)), "desvio_vpl": dp,
        "variancia_vpl": dp ** 2, "cv_vpl": dp / abs(media) if media else float("nan"),
        "ic95_media_vpl": (media - 1.96 * erro, media + 1.96 * erro),
        "prob_vpl_negativo": float((v < 0).mean()),
        "prob_vpl_positivo": float((v > 0).mean()),
        "prob_tir_maior_tma": float((amostra["TIR_aa"] > tma).mean()),
        "media_vpl_beneficiario": float(amostra["VPL_beneficiario"].mean()),
        "prob_vpl_beneficiario_positivo": float((amostra["VPL_beneficiario"] > 0).mean()),
        "fracao_tir_indefinida": float((~np.isfinite(amostra["TIR_aa"])).mean()),
        "var5_vpl": float(p5),
        "cvar5_vpl": float(v[v <= p5].mean()),
        "percentis_vpl": percentis(v),
        "percentis_tir": percentis(tir_valida),
        "percentis_payback_desc": percentis(amostra["Payback_desc_meses"]),
        "percentis_indice": percentis(amostra["Indice_EVTEAS"]),
        "media_indice": float(amostra["Indice_EVTEAS"].mean()),
        "prob_indice_maior_limiar": float((amostra["Indice_EVTEAS"] >= cfg.decisao.limiar_indice_viavel).mean()),
    }
    # Convergência: média acumulada e erro-padrão em pontos de controle
    pontos = sorted({int(x) for x in np.unique(np.geomspace(100, n, num=12).astype(int)) if x <= n} | {n})
    conv = pd.DataFrame([{"Iterações": k, "Média VPL": float(v[:k].mean()),
                          "Erro-padrão": float(v[:k].std(ddof=1) / np.sqrt(k)),
                          "P(VPL>0)": float((v[:k] > 0).mean())} for k in pontos])
    # Importância: correlação de postos (Spearman) entre entradas e saídas
    ranks = amostra[list(entradas) + ["VPL", "Indice_EVTEAS"]].rank()
    constantes = [k for k in ranks if ranks[k].nunique() <= 1]
    ranks[constantes] = np.nan       # correlação indefinida para entradas sem variação
    imp = pd.DataFrame({
        "Variável": list(entradas),
        "Spearman com VPL": [float(ranks[k].corr(ranks["VPL"])) for k in entradas],
        "Spearman com Índice": [float(ranks[k].corr(ranks["Indice_EVTEAS"])) for k in entradas],
    })
    imp = imp.reindex(imp["Spearman com VPL"].abs().sort_values(ascending=False).index).reset_index(drop=True)
    return {"amostra": amostra, "estatisticas": estat, "convergencia": conv, "importancia": imp,
            "distribuicoes": {k: vars(d) for k, d in distribuicoes_efetivas(cfg).items()}}


# ---------------------------------------------------------------------------
# Sensibilidade determinística (um fator por vez)
# ---------------------------------------------------------------------------

VARIAVEIS_SENSIBILIDADE = [
    "economico.preco_venda_kg", "tecnico.fcr", "economico.custo_racao_kg", "tecnico.mortalidade_pct",
    "tecnico.desempenho_crescimento_pct", "economico.capex_fator", "economico.custos_fixos_fator",
    "economico.tarifa_energia_kwh", "economico.custo_alevino_milheiro", "economico.tma_aa_pct",
]


def _vpl_com(cfg: EVTEASConfig, caminho: str, valor: float) -> float:
    if caminho == "economico.tma_aa_pct":
        c = copy.deepcopy(cfg)
        definir_por_caminho(c, caminho, valor)
        return float(motor(c, None, 1)["economico"]["vpl"][0])
    return float(motor(cfg, {caminho: np.array([valor])}, 1)["economico"]["vpl"][0])


def sensibilidade(cfg: EVTEASConfig, variaveis: Optional[List[str]] = None,
                  amplitudes=(0.10, 0.20)) -> pd.DataFrame:
    variaveis = variaveis or VARIAVEIS_SENSIBILIDADE
    base = float(motor(cfg, None, 1)["economico"]["vpl"][0])
    linhas = []
    for var in variaveis:
        x = float(obter_por_caminho(cfg, var))
        for amp in amplitudes:
            d = abs(x) * amp if x != 0 else amp
            lo, hi = _vpl_com(cfg, var, x - d), _vpl_com(cfg, var, x + d)
            elasticidade = ((hi - lo) / base) / (2 * d / x) if (base and x) else float("nan")
            linhas.append({"Variável": var, "Amplitude": amp, "Valor base": x, "VPL base": base,
                           "VPL (−)": lo, "VPL (+)": hi, "Δ VPL (−)": lo - base, "Δ VPL (+)": hi - base,
                           "Amplitude do efeito": abs(hi - lo),
                           "Δ VPL por 1% da variável": (hi - lo) / (2 * amp * 100),
                           "Elasticidade": elasticidade})
    df = pd.DataFrame(linhas)
    ordem = df[df["Amplitude"] == amplitudes[0]].sort_values("Amplitude do efeito", ascending=False)["Variável"]
    df["_o"] = df["Variável"].map({v: i for i, v in enumerate(ordem)})
    return df.sort_values(["_o", "Amplitude"]).drop(columns="_o").reset_index(drop=True)


def valores_criticos(cfg: EVTEASConfig, variaveis: Optional[List[str]] = None) -> pd.DataFrame:
    """Valor de cada variável que zera o VPL (demais premissas nos valores informados)."""
    variaveis = variaveis or [v for v in VARIAVEIS_SENSIBILIDADE if v != "economico.tma_aa_pct"]
    linhas = []
    for var in variaveis:
        x = float(obter_por_caminho(cfg, var))
        a, b = 0.0, max(x * 5, 1.0)
        fa, fb = _vpl_com(cfg, var, a), _vpl_com(cfg, var, b)
        if var == "tecnico.mortalidade_pct":
            b = 99.0
            fb = _vpl_com(cfg, var, b)
        critico = float("nan")
        if fa * fb <= 0:
            for _ in range(80):
                m = (a + b) / 2
                fm = _vpl_com(cfg, var, m)
                if fa * fm <= 0:
                    b, fb = m, fm
                else:
                    a, fa = m, fm
            critico = (a + b) / 2
        folga = (critico - x) / x if (x and np.isfinite(critico)) else float("nan")
        linhas.append({"Variável": var, "Valor base": x, "Valor crítico (VPL = 0)": critico,
                       "Folga relativa": folga})
    return pd.DataFrame(linhas)


def cenarios(cfg: EVTEASConfig) -> pd.DataFrame:
    """Cenários pessimista, base e otimista a partir dos extremos das distribuições."""
    dists = distribuicoes_efetivas(cfg)
    base = motor(cfg, None, 1)["economico"]["vpl"][0]
    pess, otim = {}, {}
    for k, d in dists.items():
        vmin = _vpl_com(cfg, k, d.minimo) if k != "economico.tma_aa_pct" else base
        vmax = _vpl_com(cfg, k, d.maximo) if k != "economico.tma_aa_pct" else base
        favoravel_max = vmax >= vmin
        pess[k] = np.array([d.minimo if favoravel_max else d.maximo])
        otim[k] = np.array([d.maximo if favoravel_max else d.minimo])
    linhas = []
    for nome, ov in (("Pessimista", pess), ("Base", None), ("Otimista", otim)):
        r = motor(cfg, ov, 1)
        e, i = r["economico"], r["indice"]
        linhas.append({"Cenário": nome, "VPL": float(e["vpl"][0]), "TIR a.a.": float(e["tir_anual"][0]),
                       "Payback descontado (meses)": float(e["payback_descontado_meses"][0]),
                       "Produção (kg/ano)": float(r["tecnico"]["producao_kg_ano"][0]),
                       "Índice EVTEAS": float(i["indice_evteas"][0])})
    return pd.DataFrame(linhas)

## 6. Pipeline principal reproduzível

In [ ]:
"""Pipeline principal reproduzível do EVTEAS-Py."""

import copy
import time
from dataclasses import asdict
from typing import Any, Dict, Optional

import numpy as np
import pandas as pd



def motor(cfg: EVTEASConfig, overrides: Optional[Dict[str, np.ndarray]] = None, n: int = 1,
          pesos: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    """Executa as quatro dimensões e o índice para n realizações das premissas."""
    ctx = Contexto(cfg, overrides, n)
    tec = calcular_tecnico(ctx)
    eco = calcular_economico(ctx, tec)
    amb = calcular_ambiental(ctx, tec, eco)
    soc = calcular_social(ctx, eco)
    gov = calcular_governanca(cfg)
    lean = calcular_lean_green(ctx, tec, eco)
    idx = calcular_indice(cfg, tec, eco, amb, soc, gov, pesos)
    return {"tecnico": tec, "economico": eco, "ambiental": amb, "social": soc,
            "governanca": gov, "lean_green": lean, "indice": idx}


def _escalar(obj, i: int = 0):
    """Extrai a realização i (determinística) de estruturas com vetores."""
    if isinstance(obj, dict):
        return {k: _escalar(v, i) for k, v in obj.items() if k != "_series"}
    if isinstance(obj, np.ndarray):
        if obj.ndim == 1:
            v = obj[i]
            return v.item() if hasattr(v, "item") else v
        return obj
    return obj


def tabela_premissas(cfg: EVTEASConfig) -> pd.DataFrame:
    """Registro de origem de cada premissa (Seção 3.6): valor, fonte e distribuição."""
    dists = distribuicoes_efetivas(cfg)
    linhas = []
    for caminho in caminhos_numericos(cfg) + [k for k in dists if k not in caminhos_numericos(cfg)]:
        fonte = cfg.fontes.get(caminho)
        d = dists.get(caminho)
        linhas.append({
            "Parâmetro": caminho, "Valor": obter_por_caminho(cfg, caminho),
            "Categoria da fonte": CATEGORIAS_FONTE[fonte.categoria] if fonte else CATEGORIAS_FONTE["autor"],
            "Referência": fonte.referencia if fonte else "",
            "Distribuição": d.tipo if d else "determinístico",
            "Mínimo": d.minimo if d else None, "Mais provável": d.moda if d else None, "Máximo": d.maximo if d else None,
        })
    return pd.DataFrame(linhas)


def executar_evteas(cfg: EVTEASConfig, executar_mc: bool = True, executar_sens: bool = True,
                    n_mc: Optional[int] = None) -> Dict[str, Any]:
    if not isinstance(cfg, EVTEASConfig):
        raise TypeError("cfg deve ser uma instância de EVTEASConfig")

    inicio = time.perf_counter()
    pesos = resolver_pesos(cfg)
    r = motor(cfg, None, 1, pesos)
    df_m = dre_mensal(r["economico"])
    det = {k: _escalar(v) for k, v in r.items()}
    resultado: Dict[str, Any] = {
        "artefato": ARTEFATO,
        "config": asdict(cfg),
        **det,
        "curva_crescimento": curva_crescimento(cfg),
        "dre_mensal": df_m,
        "dre_anual": dre_anual(df_m),
        "fluxo_caixa": r["economico"]["fluxo_caixa"][0],
        "sobras": distribuicao_sobras(cfg, r["economico"]),
        "ods": alinhamento_ods(cfg, r["tecnico"], r["economico"], r["ambiental"], r["social"], r["governanca"]),
        "premissas": tabela_premissas(cfg),
    }
    prob = None
    if executar_mc:
        resultado["monte_carlo"] = monte_carlo(cfg, n=n_mc)
        prob = resultado["monte_carlo"]["estatisticas"]["prob_vpl_positivo"]
    if executar_sens:
        resultado["sensibilidade"] = sensibilidade(cfg)
        resultado["valores_criticos"] = valores_criticos(cfg)
        resultado["cenarios"] = cenarios(cfg)
    resultado["decisao"] = classificar(cfg, det["economico"]["vpl"], det["indice"]["indice_evteas"],
                                       det["social"]["lso_0_100"], det["ambiental"]["vetos_ambientais"], prob)
    resultado["vv"] = validar_invariantes(cfg, resultado, r)
    resultado["tempo_execucao_s"] = time.perf_counter() - inicio
    return resultado


def comparar_alternativas(alternativas: Dict[str, EVTEASConfig], pesos_criterios: Optional[Dict[str, float]] = None,
                          n_mc: int = 2000) -> Dict[str, Any]:
    """Executa cada alternativa e as ordena por TOPSIS (critérios das quatro dimensões)."""
    criterios = {  # nome: (extrator, benefício?)
        "VPL (R$)": (lambda r, m: r["economico"]["vpl"], True),
        "P(VPL>0)": (lambda r, m: m["estatisticas"]["prob_vpl_positivo"], True),
        "OEE": (lambda r, m: r["tecnico"]["oee"], True),
        "PH cinza (m³/t)": (lambda r, m: r["ambiental"]["ph_cinza_m3_t"], False),
        "Intensidade carbono (kgCO2e/kg)": (lambda r, m: r["ambiental"]["intensidade_carbono_kgco2e_kg"], False),
        "LSO": (lambda r, m: r["social"]["lso_0_100"], True),
        "RVL (%)": (lambda r, m: r["social"]["rvl_pct"], True),
    }
    pesos_criterios = pesos_criterios or {"VPL (R$)": 0.20, "P(VPL>0)": 0.15, "OEE": 0.15,
                                          "PH cinza (m³/t)": 0.15, "Intensidade carbono (kgCO2e/kg)": 0.10,
                                          "LSO": 0.10, "RVL (%)": 0.15}
    linhas, resultados = [], {}
    for nome, cfg in alternativas.items():
        r = executar_evteas(cfg, executar_mc=False, executar_sens=False)
        m = monte_carlo(cfg, n=n_mc)
        resultados[nome] = r
        linha = {"Alternativa": nome}
        for c, (f, _) in criterios.items():
            linha[c] = float(f(r, m))
        linha["Índice EVTEAS"] = r["indice"]["indice_evteas"]
        linha["Classificação"] = classificar(cfg, r["economico"]["vpl"], r["indice"]["indice_evteas"],
                                             r["social"]["lso_0_100"], r["ambiental"]["vetos_ambientais"],
                                             m["estatisticas"]["prob_vpl_positivo"])["classificacao"]
        linhas.append(linha)
    df = pd.DataFrame(linhas)
    nomes = list(criterios)
    # critério indefinido (ex.: pegada por tonelada com produção zero) recebe o pior valor observado
    avisos, matriz = [], df.copy()
    for c in nomes:
        faltam = matriz[c].isna()
        if faltam.any():
            validos = matriz.loc[~faltam, c]
            pior = (validos.min() if criterios[c][1] else validos.max()) if len(validos) else 0.0
            matriz.loc[faltam, c] = pior
            avisos.append(f"{c} indefinido para {', '.join(matriz.loc[faltam, 'Alternativa'])}: adotado o pior valor "
                          "observado no TOPSIS")
    ranking = topsis(matriz, nomes, [pesos_criterios[c] for c in nomes], [criterios[c][1] for c in nomes])
    for c in nomes:                       # a tabela mostra o valor calculado (indefinido continua indefinido)
        ranking[c] = ranking["Alternativa"].map(df.set_index("Alternativa")[c])
    return {"matriz": df, "ranking": ranking, "pesos": pesos_criterios, "resultados": resultados, "avisos": avisos}


def copiar(cfg: EVTEASConfig) -> EVTEASConfig:
    return copy.deepcopy(cfg)

## 7. Verificação (V&V): invariantes e testes de regressão

In [ ]:
"""Verificação computacional (Boehm, 1984): invariantes e testes de regressão.

A verificação responde "estamos construindo o produto corretamente?". A
validação junto aos especialistas (Seção 3.3.2) é conduzida fora do código,
mas seus escores Likert entram em ``PesosConfig`` e ficam rastreáveis.
"""

from typing import Any, Dict, List

import numpy as np



DOMINIO_DISTRIBUICOES = {  # demais variáveis estocásticas: não negativas
    "tecnico.mortalidade_pct": (0.0, 100.0), "tecnico.desempenho_crescimento_pct": (0.0, None),
    "tecnico.fcr": (0.0, None), "economico.capex_fator": (0.0, None),
}


def validar_invariantes(cfg: EVTEASConfig, resultado: Dict[str, Any], bruto: Dict[str, Any] = None) -> Dict[str, Any]:
    checks: List[Dict[str, Any]] = []

    def chk(nome: str, ok: bool, detalhe: str = ""):
        checks.append({"Verificação": nome, "Aprovada": bool(ok), "Detalhe": detalhe})

    t, e, a, s, i = (resultado["tecnico"], resultado["economico"], resultado["ambiental"],
                     resultado["social"], resultado["indice"])
    chk("OEE em [0, 1]", 0 <= t["oee"] <= 1, f"OEE = {t['oee']:.4f}")
    chk("OEE = Disponibilidade × Performance × Qualidade",
        abs(t["oee"] - t["disponibilidade"] * t["performance"] * t["qualidade"]) < 1e-12)
    chk("FCR não negativo", t["fcr"] >= 0)
    chk("Ciclos/ano compatíveis com a duração do ciclo",
        cfg.tecnico.ciclos_ano <= 365 / max(cfg.tecnico.ciclo_dias, 1) + 1e-9,
        f"{cfg.tecnico.ciclos_ano} ≤ {365 / cfg.tecnico.ciclo_dias:.2f}")
    chk("Biomassa de despesca ≤ capacidade de suporte",
        t["biomassa_despesca_ciclo_kg"] <= t["volume_util_m3"] * t["capacidade_suporte_kg_m3"] * t["disponibilidade"] + 1e-6)
    chk("Ração = FCR × ganho de biomassa", abs(t["racao_ciclo_kg"] - t["fcr"] * t["ganho_biomassa_ciclo_kg"]) < 1e-6)

    df = resultado["dre_mensal"]
    close = lambda x, y: bool(np.allclose(x, y, rtol=1e-9, atol=1e-6))
    chk("DRE: Receita Líquida = Receita Bruta − Impostos s/ Faturamento",
        close(df["Receita Líquida"], df["Receita Bruta"] - df["Impostos s/ Faturamento"]))
    chk("DRE: Lucro Bruto = Receita Líquida − Custos Variáveis",
        close(df["Lucro Bruto"], df["Receita Líquida"] - df["Custos Variáveis"]))
    chk("DRE: EBITDA = Lucro Bruto − Custos Fixos", close(df["EBITDA"], df["Lucro Bruto"] - df["Custos Fixos"]))
    chk("DRE: Resultado = LAIR − Impostos s/ Lucro", close(df["Resultado Líquido"], df["LAIR"] - df["Impostos s/ Lucro"]))
    chk("Fluxo de caixa operacional = Resultado + Depreciação",
        close(df["Fluxo de Caixa Operacional"], df["Resultado Líquido"] + df["Depreciação"]))
    chk("Conservação de massa: produção vendida ≤ produção em engorda",
        df["Venda (kg)"].sum() <= df["Produção em engorda (kg)"].sum() + 1e-6)

    fc = resultado["fluxo_caixa"]
    chk("VPL recalculado de forma independente", abs(vpl(e["tma_mensal"], fc) - e["vpl"]) < 1e-6)
    if np.isfinite(e["tir_mensal"]):
        chk("VPL na TIR ≈ 0", abs(vpl(e["tir_mensal"], fc)) < 1e-3 * max(1.0, abs(fc[0])), e["diagnostico_tir"])
    else:
        chk("TIR indefinida acompanhada de diagnóstico", e["diagnostico_tir"] == "sem_solucao",
            e["diagnostico_tir"])
    chk("Payback descontado ≥ payback simples (quando ambos existem)",
        not (np.isfinite(e["payback_descontado_meses"]) and np.isfinite(e["payback_simples_meses"]))
        or e["payback_descontado_meses"] >= e["payback_simples_meses"] - 1e-9)

    chk("Balanço de N: aportado ≥ retido", a["n_aportado_kg"] >= a["n_retido_kg"])
    chk("Balanço de P: aportado ≥ retido", a["p_aportado_kg"] >= a["p_retido_kg"])
    chk("Score de conformidade em [0, 10]", 0 <= a["conformidade_0_10"] <= 10)
    chk("Ecoeficiência indefinida somente com valor adicionado ≤ 0",
        (a["ecoeficiencia_status"] == "definida") == (a["valor_adicionado_regime"] > 0))
    chk("LSO em [0, 100]", 0 <= s["lso_0_100"] <= 100)
    chk("Risco social em [0, 1]", 0 <= s["risco_social"] <= 1)

    for caminho, d in cfg.monte_carlo.distribuicoes.items():
        v = float(obter_por_caminho(cfg, caminho))
        ok = d.minimo - 1e-9 <= v <= d.maximo + 1e-9 and (d.tipo == "uniforme" or abs(d.moda - v) <= 1e-9 * max(1.0, abs(v)))
        chk(f"Distribuição de {caminho} contém o valor determinístico como mais provável", ok,
            f"valor {v:g}; distribuição {d.tipo} ({d.minimo:g}, {d.moda:g}, {d.maximo:g})")
        lo, hi = DOMINIO_DISTRIBUICOES.get(caminho, (0.0, None))
        chk(f"Distribuição de {caminho} dentro do domínio da variável",
            d.minimo >= lo - 1e-12 and (hi is None or d.maximo <= hi + 1e-12),
            f"({d.minimo:g}, {d.maximo:g}) em [{lo:g}, {'∞' if hi is None else f'{hi:g}'}]")

    pesos = i["pesos"]
    chk("Pesos dimensionais somam 1", abs(sum(pesos["dimensoes"].values()) - 1) < 1e-9)
    chk("Pesos dimensionais não negativos", min(pesos["dimensoes"].values()) >= 0)
    chk("Índice EVTEAS em [0, 1]", 0 <= i["indice_evteas"] <= 1, f"{i['indice_evteas']:.4f}")
    chk("KPIs normalizados em [0, 1]", all(0 <= v <= 1 for v in i["kpis_normalizados"].values()))
    if "ahp" in pesos:
        chk("AHP consistente (RC ≤ 0,10)", pesos["ahp"]["consistente"], f"RC = {pesos['ahp']['rc']:.3f}")

    if "monte_carlo" in resultado:
        mc = resultado["monte_carlo"]
        st = mc["estatisticas"]
        chk("Monte Carlo: P(VPL>0) + P(VPL<0) ≤ 1", st["prob_vpl_positivo"] + st["prob_vpl_negativo"] <= 1 + 1e-12)
        chk("Monte Carlo: P5 ≤ mediana ≤ P95",
            st["percentis_vpl"]["p5"] <= st["mediana_vpl"] <= st["percentis_vpl"]["p95"])
        conv = mc["convergencia"]
        chk("Monte Carlo: erro-padrão final < 2% do desvio-padrão",
            conv["Erro-padrão"].iloc[-1] < 0.02 * st["desvio_vpl"] or st["n"] < 2500,
            f"EP = {conv['Erro-padrão'].iloc[-1]:,.0f}")

    falhas = [c for c in checks if not c["Aprovada"]]
    return {"valido": not falhas, "quantidade_verificacoes": len(checks), "quantidade_erros": len(falhas),
            "erros": [c["Verificação"] for c in falhas], "verificacoes": checks}


def teste_regressao_deterministica() -> bool:
    """Casos com solução analítica conhecida (executáveis sem pytest)."""
    assert abs(vpl(0.0, [-100, 50, 50]) - 0.0) < 1e-12
    assert abs(vpl(0.10, [-100, 110]) - 0.0) < 1e-9
    r, d = tir([-100, 110])
    assert abs(r[0] - 0.10) < 1e-9 and d[0] == "convencional"
    r, d = tir([-100, -10, -10])
    assert np.isnan(r[0]) and d[0] == "sem_solucao"
    assert abs(payback([-100, 40, 40, 40])[0] - 2.5) < 1e-12
    assert np.isnan(payback([-100, 10, 10])[0])
    return True

## 8. Caso ilustrativo e alternativas de referência

In [ ]:
"""Configurações de referência: caso ilustrativo e alternativas.

O caso ilustrativo representa um Projeto de Geração de Trabalho e Renda (PGTR) de
piscicultura de tilápia em viveiros escavados, conduzido por cooperativa de
pescadores artesanais. Os valores são estimativas ilustrativas, registradas
com sua origem; não constituem dados de um empreendimento real e devem ser
recalibrados na validação com o especialista em piscicultura.
"""

import copy


REF_KUBITZA = "Kubitza (2017) — faixa zootécnica de referência para tilápia"
REF_CONAMA = "Resolução CONAMA nº 357/2005, classe 2, ambiente lótico"
REF_COTACAO = "Estimativa ilustrativa; substituir por cotação regional atualizada"


def criar_config_caso_base() -> EVTEASConfig:
    cfg = EVTEASConfig(projeto="Caso ilustrativo — PGTR de piscicultura de tilápia em viveiros escavados")
    cfg.tecnico = TecnicoConfig()
    cfg.economico = EconomicoConfig()
    cfg.ambiental = AmbientalConfig()
    cfg.social = SocialConfig()
    cfg.economico.fomento_nao_reembolsavel_pct = 70.0
    cfg.economico.cooperados_fornecedores = 14      # associados que não atuam na operação (mínimo legal de 20)
    cfg.fontes = {
        "economico.fomento_nao_reembolsavel_pct": Fonte("autor", "Hipótese de aporte por projeto de compensação do licenciamento federal (IBAMA, 2010, 2012)"),
        "tecnico.fcr": Fonte("bibliografico", REF_KUBITZA),
        "tecnico.mortalidade_pct": Fonte("bibliografico", REF_KUBITZA),
        "tecnico.peso_final_g": Fonte("bibliografico", REF_KUBITZA),
        "tecnico.ciclo_dias": Fonte("bibliografico", REF_KUBITZA),
        "tecnico.produtividade_esperada_kg_m2_ciclo": Fonte("bibliografico", REF_KUBITZA),
        "tecnico.area_lamina_m2": Fonte("autor", "Dimensionamento conceitual (sem definição de terreno)"),
        "economico.preco_venda_kg": Fonte("cotacao", REF_COTACAO),
        "economico.custo_racao_kg": Fonte("cotacao", REF_COTACAO),
        "economico.custo_alevino_milheiro": Fonte("cotacao", REF_COTACAO),
        "economico.tarifa_energia_kwh": Fonte("cotacao", REF_COTACAO),
        "economico.classe_estimativa_aace": Fonte("normativo", "AACE International (2020), estimativa Classe 5"),
        "economico.tma_aa_pct": Fonte("autor", "Custo de oportunidade de recursos de fomento"),
        "ambiental.n_max_mg_l": Fonte("normativo", REF_CONAMA),
        "ambiental.p_max_mg_l": Fonte("normativo", REF_CONAMA),
        "ambiental.faixa_app_minima_m": Fonte("normativo", "Lei nº 12.651/2012, art. 4º, I, a"),
        "ambiental.proteina_racao_pct": Fonte("bibliografico", REF_KUBITZA),
        "ambiental.evaporacao_mm_dia": Fonte("bibliografico", "Kubitza (2017); calibrar com dados climáticos locais"),
        "social.salario_minimo": Fonte("normativo", "Salário mínimo nacional vigente"),
    }
    return cfg


def criar_alternativas() -> dict:
    """Três alternativas de projeto para comparação por TOPSIS."""
    a = criar_config_caso_base()
    a.projeto = "A — Viveiros semi-intensivos, energia da rede"

    b = criar_config_caso_base()
    b.projeto = "B — A + energia solar + wetland para efluentes"
    b.economico.capex_itens["Usina fotovoltaica"] = 55000.0
    b.economico.capex_itens["Wetland construído"] = 25000.0
    b.economico.tarifa_energia_kwh = 0.12           # custo residual (disponibilidade, O&M)
    b.monte_carlo.distribuicoes["economico.tarifa_energia_kwh"] = Distribuicao("triangular", 0.08, 0.12, 0.18)
    b.ambiental.fonte_energia = "solar"
    b.ambiental.remocao_p_tratamento_pct = 80.0
    b.ambiental.remocao_n_tratamento_pct = 60.0
    b.ambiental.checklist["energia_renovavel"] = True
    b.ambiental.checklist["gestao_residuos"] = True
    b.ambiental.residuos_reaproveitados_pct = 70.0

    c = criar_config_caso_base()
    c.projeto = "C — Intensificação com maior aeração"
    c.tecnico.sistema_produtivo = "intensivo"
    c.tecnico.produtividade_esperada_kg_m2_ciclo = 2.0
    c.economico.capex_itens["Aeradores e quadro elétrico"] = 140000.0
    c.economico.mao_obra_mes = 14000.0
    c.monte_carlo.distribuicoes["tecnico.mortalidade_pct"] = Distribuicao("triangular", 6.0, 12.0, 25.0)
    c.tecnico.mortalidade_pct = 12.0
    return {"A": a, "B": b, "C": c}


def exemplo_validacao_especialistas(cfg: EVTEASConfig) -> EVTEASConfig:
    """Ilustra a parametrização por Likert (Quadro 7) com dois avaliadores.

    Os escores abaixo são *placeholders* de demonstração; devem ser
    substituídos pelas respostas efetivas dos especialistas na Fase 3.
    """
    c = copy.deepcopy(cfg)
    c.pesos = PesosConfig(metodo="likert", likert_dimensoes={
        "Especialista em framework": {"tecnico": 4, "economico": 5, "ambiental": 4, "social": 3},
        "Especialista em piscicultura": {"tecnico": 5, "economico": 5, "ambiental": 3, "social": 4},
    })
    return c

## 9. Resumo executivo, exportação e gráficos

In [ ]:
"""Resumo executivo, exportação (Excel/JSON/Markdown) e gráficos."""

import json
from pathlib import Path
from typing import Any, Dict, List

import numpy as np
import pandas as pd


def moeda(v) -> str:
    if v is None or not np.isfinite(v):
        return "n/d"
    return f"R$ {v:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


def percentual(v, casas: int = 1) -> str:
    if v is None or not np.isfinite(v):
        return "n/d"
    return f"{v * 100:.{casas}f}%".replace(".", ",")


def numero(v, casas: int = 2) -> str:
    if v is None or not np.isfinite(v):
        return "n/d"
    return f"{v:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def resumo_executivo(r: Dict[str, Any]) -> pd.DataFrame:
    """Indicadores principais, no mesmo formato dos valores para o texto (campos_texto)."""
    t, e, a, s, i = r["tecnico"], r["economico"], r["ambiental"], r["social"], r["indice"]
    cfg = r.get("config")
    projeto = getattr(cfg, "projeto", None) if not isinstance(cfg, dict) else cfg.get("projeto")
    linhas = [("Projeto", projeto or "(sem nome)")] if cfg is not None else []
    linhas += [
        ("Classificação", r["decisao"]["classificacao"]),
        ("Produção anual (kg)", num(t["producao_kg_ano"], 0)),
        ("OEE aquícola (0–1)", num(t["oee"], 3)),
        ("FCR (kg ração/kg ganho)", num(t["fcr"], 2)),
        ("Investimento total", reais(e["investimento_total"])),
        ("VPL (projeto)", reais(e["vpl"])),
        ("TIR (projeto)", tir_txt(e["tir_anual"], e["diagnostico_tir"])),
        ("Payback descontado", meses_ou_horizonte(e["payback_descontado_meses"])),
        ("VPL (beneficiário, com fomento)", reais(e["vpl_beneficiario"])),
        ("ROI anual em regime", pct_br(e["roi_anual_regime"])),
        ("Margem de segurança", pct_br(e["margem_seguranca"])),
        ("Custo total (R$/kg)", reais(e["custo_total_kg"], 2)),
        ("Pegada hídrica azul (m³/t)", num(a["ph_azul_m3_t"], 0)),
        ("Pegada hídrica cinza (m³/t)", num(a["ph_cinza_m3_t"], 0) + f" [{a['poluente_critico']}]"),
        ("Intensidade de carbono (kgCO2e/kg)", num(a["intensidade_carbono_kgco2e_kg"], 2)),
        ("Ecoeficiência (R$ VA/kgCO2e)", num(a["ecoeficiencia_r_por_kgco2e"], 2) if a["ecoeficiencia_status"] == "definida"
         else a["ecoeficiencia_status"]),
        ("Conformidade ambiental (0–10)", num(a["conformidade_0_10"], 2)),
        ("LSO (0–100)", num(s["lso_0_100"], 1)),
        ("RVL (%)", num(s["rvl_pct"], 1)),
        ("Renda mensal por trabalhador", reais(s["renda_mensal_trabalhador"])),
        ("Índice EVTEAS (0–1)", num(i["indice_evteas"], 3)),
        ("Score ESG (0–1)", num(i["esg_score"], 3)),
    ]
    if "monte_carlo" in r:
        st = r["monte_carlo"]["estatisticas"]
        linhas += [("P(VPL > 0) — Monte Carlo", pct_br(st["prob_vpl_positivo"])),
                   ("VPL médio — Monte Carlo", reais(st["media_vpl"])),
                   ("P(VPL beneficiário > 0)", pct_br(st["prob_vpl_beneficiario_positivo"]))]
    return pd.DataFrame(linhas, columns=["Indicador", "Valor"])


def tabela_sensibilidade(r: Dict[str, Any]) -> pd.DataFrame:
    """Sensibilidade a ±10%, valor crítico e correlação de Spearman de cada variável, formatadas para o texto."""
    sens = _sens10(r)
    imp = r["monte_carlo"]["importancia"].set_index("Variável")["Spearman com VPL"] if "monte_carlo" in r else {}
    linhas = []
    for _, x in sens.iterrows():
        v = x["Variável"]
        linhas.append({"Variável": _rotulo_var(v), "Δ VPL (−10%)": reais(x["Δ VPL (−)"]), "Δ VPL (+10%)": reais(x["Δ VPL (+)"]),
                       "Valor crítico (VPL = 0) e folga": _critico(r, v),
                       "Spearman com o VPL (Monte Carlo)": num(imp[v], 3) if v in imp else "—"})
    return pd.DataFrame(linhas)


def tabela_topsis(comparacao: Dict[str, Any]) -> pd.DataFrame:
    """Ranking TOPSIS das alternativas, formatado para o texto."""
    t = comparacao["ranking"]
    return pd.DataFrame([{
        "Posição": f"{int(x['Ranking'])}º", "Alternativa": x["Alternativa"], "VPL": reais(x["VPL (R$)"]),
        "P(VPL>0)": pct_br(x["P(VPL>0)"]), "OEE": num(x["OEE"], 3), "PH cinza (m³/t)": num(x["PH cinza (m³/t)"], 0),
        "kgCO2e/kg": num(x["Intensidade carbono (kgCO2e/kg)"], 2), "LSO": num(x["LSO"], 1), "RVL (%)": num(x["RVL (%)"], 1),
        "Índice EVTEAS": num(x["Índice EVTEAS"], 3), "TOPSIS": num(x["TOPSIS"], 3), "Classificação": x["Classificação"]}
        for _, x in t.iterrows()])


def serializar(obj):
    if isinstance(obj, pd.DataFrame):
        return obj.to_dict(orient="records")
    if isinstance(obj, pd.Series):
        return obj.to_dict()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer, np.floating, np.bool_)):
        return obj.item()
    if isinstance(obj, float) and not np.isfinite(obj):
        return None
    if isinstance(obj, dict):
        return {str(k): serializar(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [serializar(v) for v in obj]
    return obj


def _df_dict(d: Dict[str, Any]) -> pd.DataFrame:
    return pd.DataFrame([{"Indicador": k, "Valor": v} for k, v in d.items()
                         if not isinstance(v, (dict, list, pd.DataFrame, np.ndarray))])


def exportar(r: Dict[str, Any], pasta="evteas_output") -> List[str]:
    pasta = Path(pasta)
    pasta.mkdir(parents=True, exist_ok=True)
    arquivos = []
    xlsx = pasta / "EVTEAS_Py_resultados.xlsx"
    with pd.ExcelWriter(xlsx, engine="openpyxl") as w:
        resumo_executivo(r).to_excel(w, sheet_name="Resumo", index=False)
        r["premissas"].to_excel(w, sheet_name="Premissas", index=False)
        _df_dict(r["tecnico"]).to_excel(w, sheet_name="Tecnico", index=False)
        r["curva_crescimento"].to_excel(w, sheet_name="Curva_Crescimento", index=False)
        _df_dict({k: v for k, v in r["economico"].items() if k != "aliquotas"}).to_excel(w, sheet_name="Economico", index=False)
        r["dre_mensal"].to_excel(w, sheet_name="DRE_Mensal", index=False)
        r["dre_anual"].to_excel(w, sheet_name="DRE_Anual", index=False)
        pd.DataFrame({"Mês": range(len(r["fluxo_caixa"])), "Fluxo de caixa": r["fluxo_caixa"]}).to_excel(
            w, sheet_name="Fluxo_Caixa", index=False)
        _df_dict(r["ambiental"]).to_excel(w, sheet_name="Ambiental", index=False)
        pd.DataFrame(r["ambiental"]["conformidade_detalhe"]).to_excel(w, sheet_name="Conformidade", index=False)
        _df_dict(r["social"]).to_excel(w, sheet_name="Social", index=False)
        r["ods"].to_excel(w, sheet_name="ODS", index=False)
        pd.DataFrame([{"Desperdício": k, "R$/ano": v} for k, v in r["lean_green"]["desperdicios_r_ano"].items()]
                     ).to_excel(w, sheet_name="Lean_Green", index=False)
        pd.DataFrame({"KPI": list(r["indice"]["kpis_brutos"]),
                      "Valor bruto": list(r["indice"]["kpis_brutos"].values()),
                      "Normalizado": [r["indice"]["kpis_normalizados"][k] for k in r["indice"]["kpis_brutos"]]}
                     ).to_excel(w, sheet_name="KPIs", index=False)
        pd.DataFrame([{"Dimensão": k, "Score": v, "Peso": r["indice"]["pesos"]["dimensoes"][k]}
                      for k, v in r["indice"]["dimensoes"].items()]).to_excel(w, sheet_name="Indice", index=False)
        if "monte_carlo" in r:
            mc = r["monte_carlo"]
            mc["amostra"].to_excel(w, sheet_name="MC_Amostra", index=False)
            _df_dict({k: (str(v) if isinstance(v, tuple) else v) for k, v in mc["estatisticas"].items()}).to_excel(
                w, sheet_name="MC_Estatisticas", index=False)
            mc["convergencia"].to_excel(w, sheet_name="MC_Convergencia", index=False)
            mc["importancia"].to_excel(w, sheet_name="MC_Importancia", index=False)
        for chave, aba in (("sensibilidade", "Sensibilidade"), ("valores_criticos", "Valores_Criticos"),
                           ("cenarios", "Cenarios")):
            if chave in r:
                r[chave].to_excel(w, sheet_name=aba, index=False)
        pd.DataFrame(r["vv"]["verificacoes"]).to_excel(w, sheet_name="VV", index=False)
    arquivos.append(str(xlsx))
    js = pasta / "EVTEAS_Py_resultados.json"
    leve = {k: v for k, v in r.items() if k not in ("dre_mensal",)}
    if "monte_carlo" in leve:
        leve = dict(leve)
        leve["monte_carlo"] = {k: v for k, v in r["monte_carlo"].items() if k != "amostra"}
    with open(js, "w", encoding="utf-8") as f:
        json.dump(serializar(leve), f, ensure_ascii=False, indent=2, default=str)
    arquivos.append(str(js))
    md = pasta / "EVTEAS_Py_relatorio.md"
    md.write_text(relatorio_markdown(r), encoding="utf-8")
    arquivos.append(str(md))
    return arquivos


def relatorio_markdown(r: Dict[str, Any]) -> str:
    linhas = [f"# Relatório EVTEAS-Py — {r['config']['projeto']}", "", f"Gerado por: {r['artefato']}", "",
              "## Resumo executivo", "", "| Indicador | Valor |", "|---|---|"]
    linhas += [f"| {a} | {b} |" for a, b in resumo_executivo(r).itertuples(index=False)]
    d = r["decisao"]
    linhas += ["", "## Decisão", "", f"**{d['classificacao']}**", ""]
    linhas += [f"- Veto: {v}" for v in d["vetos"]] + [f"- Ressalva: {v}" for v in d["ressalvas"]]
    linhas += ["", "## ODS atendidos", ""] + [f"- {o}" for o, ok in zip(r["ods"]["ODS"], r["ods"]["Atendido"]) if ok]
    vv = r["vv"]
    linhas += ["", "## Verificação", "", f"{vv['quantidade_verificacoes']} verificações; {vv['quantidade_erros']} falha(s)."]
    return "\n".join(linhas) + "\n"


# ---------------------------------------------------------------------------
# Gráficos
# ---------------------------------------------------------------------------

def _plt():
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10, "axes.spines.top": False,
                         "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3,
                         "figure.dpi": 100, "savefig.dpi": 220})
    return plt


AZUL, LARANJA, VERDE, CINZA, VERMELHO = "#2a6f97", "#e07a1f", "#4c956c", "#6c757d", "#c0392b"


def gerar_graficos(r: Dict[str, Any], pasta="evteas_output/figuras", comparacao: Dict[str, Any] = None) -> Dict[str, str]:
    plt = _plt()
    pasta = Path(pasta)
    pasta.mkdir(parents=True, exist_ok=True)
    figs: Dict[str, str] = {}

    from matplotlib.ticker import ScalarFormatter

    class FormatadorBR(ScalarFormatter):               # vírgula decimal e ponto de milhar nos eixos
        def __call__(self, x, pos=None):
            return super().__call__(x, pos).replace(",", "X").replace(".", ",").replace("X", ".")

    def salvar(fig, nome):
        p = pasta / nome
        for ax in fig.axes:
            for eixo in (ax.xaxis, ax.yaxis):
                if type(eixo.get_major_formatter()) is ScalarFormatter and eixo.get_scale() == "linear":
                    eixo.set_major_formatter(FormatadorBR())
        fig.tight_layout()
        fig.savefig(p, bbox_inches="tight")
        plt.close(fig)
        figs[nome.split(".")[0]] = str(p)

    # 1. Curva de crescimento
    cc = r["curva_crescimento"]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(cc["Dia"], cc["Peso médio (g)"], color=AZUL, lw=2, label="Peso médio (g)")
    ax.set_xlabel("Dia do ciclo"); ax.set_ylabel("Peso médio (g)", color=AZUL)
    ax2 = ax.twinx(); ax2.grid(False)
    ax2.plot(cc["Dia"], cc["Biomassa (kg)"] / 1000, color=LARANJA, lw=2, label="Biomassa (t)")
    ax2.plot(cc["Dia"], cc["Ração acumulada (kg)"] / 1000, color=CINZA, lw=1.5, ls="--", label="Ração acumulada (t)")
    ax2.set_ylabel("t", color=LARANJA)
    h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc="upper left", frameon=False)
    ax.set_title("Curva de crescimento, biomassa e arraçoamento por ciclo")
    salvar(fig, "01_curva_crescimento.png")

    # 2. Receita, custos e fluxo acumulado
    df = r["dre_mensal"]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(df["Mês"], df["Receita Bruta"] / 1000, color=AZUL, lw=1.8, label="Receita bruta")
    ax.plot(df["Mês"], (df["Custos Variáveis"] + df["Custos Fixos"]) / 1000, color=LARANJA, lw=1.8, label="Custos operacionais")
    ax.set_xlabel("Mês"); ax.set_ylabel("R$ mil/mês")
    ax2 = ax.twinx(); ax2.grid(False)
    ax2.plot(range(len(r["fluxo_caixa"])), np.cumsum(r["fluxo_caixa"]) / 1000, color=VERDE, lw=2, label="Fluxo de caixa acumulado")
    ax2.axhline(0, color=CINZA, lw=0.8)
    ax2.set_ylabel("R$ mil (acumulado)", color=VERDE)
    h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc="lower right", frameon=False)
    ax.set_title("Projeção mensal: receita, custos e fluxo de caixa acumulado")
    salvar(fig, "02_projecao_economica.png")

    # 3. Scores dimensionais
    dims = r["indice"]["dimensoes"]
    rot = {"tecnico": "Técnica", "economico": "Econômica", "ambiental": "Ambiental", "social": "Social"}
    fig, ax = plt.subplots(figsize=(7, 3.6))
    vals = [dims[k] for k in rot]
    ax.barh(list(rot.values()), vals, color=[AZUL, LARANJA, VERDE, CINZA])
    for y, v in enumerate(vals):
        ax.text(v + 0.01, y, f"{v:.3f}".replace(".", ","), va="center")
    ax.axvline(r["indice"]["indice_evteas"], color=VERMELHO, ls="--", lw=1.2,
               label=f"Índice EVTEAS = {r['indice']['indice_evteas']:.3f}".replace(".", ","))
    ax.set_xlim(0, 1); ax.set_xlabel("Score normalizado (0–1)"); ax.invert_yaxis()
    ax.legend(frameon=False, loc="upper right", bbox_to_anchor=(1.0, -0.18)); ax.set_title("Scores dimensionais e índice integrado")
    salvar(fig, "03_dimensoes.png")

    # 4. Radar social
    radar = r["social"]["radar_social"]
    nomes = list(radar); valores = [radar[k] for k in nomes]
    ang = np.linspace(0, 2 * np.pi, len(nomes), endpoint=False).tolist()
    fig = plt.figure(figsize=(5.2, 5.2)); ax = fig.add_subplot(111, polar=True)
    ax.plot(ang + ang[:1], valores + valores[:1], color=VERDE, lw=2)
    ax.fill(ang + ang[:1], valores + valores[:1], color=VERDE, alpha=0.25)
    ax.set_xticks(ang); ax.set_xticklabels(nomes, fontsize=9); ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0]); ax.set_yticklabels(["0,25", "0,50", "0,75", "1,00"], fontsize=7, color=CINZA)
    ax.tick_params(axis="x", pad=8)
    ax.set_title("Radar Social", pad=18)
    salvar(fig, "04_radar_social.png")

    if "monte_carlo" in r:
        mc = r["monte_carlo"]; v = mc["amostra"]["VPL"] / 1000; st = mc["estatisticas"]
        fig, axs = plt.subplots(1, 2, figsize=(10, 3.8))
        axs[0].hist(v, bins=60, color=AZUL, alpha=0.85)
        axs[0].axvline(0, color=VERMELHO, ls="--", lw=1.2, label="VPL = 0")
        axs[0].axvline(r["economico"]["vpl"] / 1000, color=LARANJA, lw=1.5, label="Determinístico (moda)")
        axs[0].axvline(st["media_vpl"] / 1000, color=VERDE, lw=1.5, label="Média Monte Carlo")
        axs[0].set_xlabel("VPL (R$ mil)"); axs[0].set_ylabel("Frequência"); axs[0].legend(frameon=False, fontsize=8)
        axs[0].set_title("Distribuição do VPL")
        xs = np.sort(v); axs[1].plot(xs, np.arange(1, len(xs) + 1) / len(xs), color=AZUL, lw=2)
        axs[1].axvline(0, color=VERMELHO, ls="--", lw=1.2)
        axs[1].set_xlabel("VPL (R$ mil)"); axs[1].set_ylabel("Probabilidade acumulada")
        axs[1].set_title(f"P(VPL ≤ 0) = {st['prob_vpl_negativo']:.1%}".replace(".", ","))
        salvar(fig, "05_monte_carlo_vpl.png")

        conv = mc["convergencia"]
        fig, ax = plt.subplots(figsize=(7, 3.6))
        ax.errorbar(conv["Iterações"], conv["Média VPL"] / 1000, yerr=1.96 * conv["Erro-padrão"] / 1000,
                    color=AZUL, marker="o", ms=3, capsize=3)
        ax.set_xscale("log"); ax.set_xlabel("Iterações (escala log)"); ax.set_ylabel("Média do VPL (R$ mil) ± IC95%")
        ax.set_title("Convergência da simulação de Monte Carlo")
        salvar(fig, "06_convergencia_mc.png")

    if "sensibilidade" in r:
        s = r["sensibilidade"]; s = s[s["Amplitude"] == s["Amplitude"].min()].copy()
        s = s.sort_values("Amplitude do efeito")
        rotulo = s["Variável"].map(_rotulo_var)
        fig, ax = plt.subplots(figsize=(8, 4.2))
        ax.barh(rotulo, s["Δ VPL (−)"] / 1000, color=LARANJA, label="−10%")
        ax.barh(rotulo, s["Δ VPL (+)"] / 1000, color=AZUL, label="+10%")
        ax.axvline(0, color="black", lw=0.8)
        ax.set_xlabel("Variação do VPL (R$ mil)"); ax.legend(frameon=False)
        ax.set_title("Diagrama de tornado — sensibilidade do VPL a ±10%")
        salvar(fig, "07_tornado.png")

    if comparacao is not None:
        import textwrap
        rk = comparacao["ranking"]
        fig, ax = plt.subplots(figsize=(7, 1.4 + 0.55 * len(rk)))
        nomes_alt = [textwrap.fill(str(a), 30) for a in rk["Alternativa"]]
        ax.barh(nomes_alt, rk["TOPSIS"], color=[VERDE, AZUL, CINZA][: len(rk)])
        for y, v in enumerate(rk["TOPSIS"]):
            ax.text(v + 0.01, y, f"{v:.3f}".replace(".", ","), va="center")
        ax.set_xlim(0, 1); ax.invert_yaxis(); ax.set_xlabel("Proximidade relativa à solução ideal (TOPSIS)")
        ax.set_title("Comparação de alternativas de projeto")
        salvar(fig, "08_topsis.png")
    return figs


def diagrama_arquitetura(caminho="evteas_output/figuras/00_arquitetura.png") -> str:
    """Diagrama do fluxo Input → Processamento → Output → Verificação."""
    plt = _plt()
    from matplotlib.patches import FancyBboxPatch
    fig, ax = plt.subplots(figsize=(10, 4.6)); ax.axis("off"); ax.set_xlim(0, 10); ax.set_ylim(0, 5)
    caixas = [
        (0.1, 3.3, 2.0, 1.4, "Entradas\nEVTEASConfig\n+ registro de fontes", "#dbe9f6"),
        (2.6, 3.3, 2.2, 1.4, "Motor vetorizado\ntécnico → econômico\n→ ambiental → social", "#fde8d0"),
        (5.3, 3.3, 2.0, 1.4, "Normalização\npesos (preset/\nLikert/AHP)", "#e3f1e6"),
        (7.8, 3.3, 2.1, 1.4, "Índice EVTEAS\nESG, ODS\nregra de decisão", "#eee"),
        (2.6, 0.6, 2.2, 1.6, "Incerteza\nMonte Carlo (n iterações)\nsensibilidade, cenários,\nvalores críticos", "#fde8d0"),
        (5.3, 0.6, 2.0, 1.6, "Verificação (V&V)\ninvariantes e\ntestes automatizados", "#e3f1e6"),
        (7.8, 0.6, 2.1, 1.6, "Saídas\nExcel, JSON,\nrelatório e figuras\nTOPSIS", "#eee"),
    ]
    for x, y, w, h, txt, cor in caixas:
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.05", fc=cor, ec="#555", lw=1))
        ax.text(x + w / 2, y + h / 2, txt, ha="center", va="center", fontsize=9)
    seta = dict(arrowstyle="->", color="#333", lw=1.2)
    for (x1, y1), (x2, y2) in [((2.15, 4.0), (2.55, 4.0)), ((4.85, 4.0), (5.25, 4.0)), ((7.35, 4.0), (7.75, 4.0)),
                               ((3.7, 3.25), (3.7, 2.25)), ((4.85, 1.4), (5.25, 1.4)), ((7.35, 1.4), (7.75, 1.4)),
                               ((8.85, 3.25), (8.85, 2.25))]:
        ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=seta)
    ax.set_title("Arquitetura do EVTEAS-Py", fontsize=12)
    Path(caminho).parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(caminho, bbox_inches="tight"); plt.close(fig)
    return caminho

## 10. Entrada de dados: menu inicial, wizard completo, arquivos JSON e análise de preços

In [ ]:
"""Entrada de dados do EVTEAS-Py: menu inicial, wizard completo, arquivos JSON e análise de preços.

O estudo começa SEMPRE pela entrada de dados (``iniciar_entradas``). Há três modos:

1. **Novo projeto** — todas as premissas do projeto precisam ser digitadas; o
   wizard não usa valores prontos. Apenas coeficientes técnicos com referência
   normativa ou bibliográfica (ex.: limites da Resolução CONAMA nº 357/2005,
   teor de nitrogênio no peixe) e alguns parâmetros de configuração do próprio
   artefato (número de iterações, seed, limiares de decisão) aparecem como
   "referência — Enter aceita", e a origem fica registrada no estudo.
2. **Carregar e revisar** — lê um arquivo JSON salvo numa entrada anterior;
   cada pergunta mostra o valor salvo, e Enter o mantém.
3. **Exemplo ilustrativo** — carrega o caso ilustrativo incorporado ao pacote,
   apenas para demonstração do artefato.

O wizard incorpora os campos dos estudos de viabilidade econômica em notebook
utilizados pelo autor: tipo de organização, mix de produtos com
quantidades e preços, serviços de transporte, carência, custos, cooperados,
funcionários CLT por cargo, tributos individuais, rampa, crescimento e TMA. Acrescenta os
parâmetros técnico-ambientais-sociais, a governança, os pesos (preset,
Likert ou AHP), as regras de decisão, as distribuições do Monte Carlo e a
origem das premissas (Seção 3.6).
"""

import copy
import json
import math
import re
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd


LINHA = "-" * 78


# ---------------------------------------------------------------------------
# Arquivos de entradas
# ---------------------------------------------------------------------------

def salvar_config(cfg: EVTEASConfig, caminho) -> str:
    Path(caminho).write_text(json.dumps(config_para_dict(cfg), ensure_ascii=False, indent=2), encoding="utf-8")
    return str(caminho)


def carregar_config(caminho, avisos: Optional[List[str]] = None, permitir_pendentes: bool = False) -> EVTEASConfig:
    """Lê um arquivo de entradas (aceita arquivos incompletos e o JSON de resultados).

    Campos ausentes não são completados com o caso ilustrativo: ficam pendentes e, salvo
    ``permitir_pendentes``, impedem a leitura. Os avisos vão para ``avisos`` (ou ``warnings``)."""
    cfg, msgs, pendentes = ler_entradas(json.loads(Path(caminho).read_text(encoding="utf-8-sig")))
    if avisos is not None:
        avisos.extend(msgs)
    else:
        import warnings
        for m in msgs:
            warnings.warn(m, stacklevel=2)
    if pendentes and not permitir_pendentes:
        raise ValueError("entradas ausentes no arquivo: " + ", ".join(pendentes))
    return cfg


def nome_arquivo(cfg: EVTEASConfig) -> str:
    import unicodedata
    ascii_ = unicodedata.normalize("NFKD", cfg.projeto or "").encode("ascii", "ignore").decode()
    base = re.sub(r"[^A-Za-z0-9]+", "_", ascii_).strip("_")
    if len(base) > 60:                      # nomes longos parecidos não se sobrescrevem
        import hashlib
        base = base[:53].rstrip("_") + "_" + hashlib.sha1((cfg.projeto or "").encode("utf-8")).hexdigest()[:6]
    return f"entradas_{base or 'projeto'}.json"


# ---------------------------------------------------------------------------
# Leitura de valores digitados
# ---------------------------------------------------------------------------

MILHAR_COM_PONTO = re.compile(r"-?[1-9]\d{0,2}(?:\.\d{3})+")    # 40.000 · 1.234.567 (padrão brasileiro)


def usa_ponto_de_milhar(txt: str) -> bool:
    t = txt.strip().replace("R$", "").replace("%", "").replace(" ", "")
    return "," not in t and bool(MILHAR_COM_PONTO.fullmatch(t))


def converter_numero(txt: str) -> float:
    """Aceita 1234,56 · 1.234,56 · 40.000 · 1.234.567 · 1234.56 · 1,234.56 · 12% · R$ 10 · frações como 1/3.

    Ponto seguido de grupos de exatamente três dígitos (40.000) é separador de milhar,
    como no padrão brasileiro; nos demais casos (10.5, 0.280), ponto é decimal."""
    t = txt.strip().replace("R$", "").replace("%", "").replace(" ", "")
    if re.fullmatch(r"-?\d+(?:[.,]\d+)?/\d+(?:[.,]\d+)?", t):
        a, b = t.split("/")
        return converter_numero(a) / converter_numero(b)
    if "," in t and "." in t:
        t = t.replace(".", "").replace(",", ".") if t.rfind(",") > t.rfind(".") else t.replace(",", "")
    elif "," in t:
        t = t.replace(",", ".")
    elif MILHAR_COM_PONTO.fullmatch(t):
        t = t.replace(".", "")
    v = float(t)
    if not math.isfinite(v):
        raise ValueError("valor não finito")
    return v


def _fmt(v, casas: int = 6) -> str:
    """Número no padrão brasileiro (1.500.000; 0,28), sem notação científica."""
    if isinstance(v, (bool, np.bool_)):
        return "s" if v else "n"
    if isinstance(v, (int, np.integer)):
        return f"{int(v):,}".replace(",", ".")
    if isinstance(v, (float, np.floating)):
        v = float(v)
        if not math.isfinite(v):
            return "—"
        if v != 0 and abs(v) < 10 ** -casas:
            return f"{v:.3g}".replace(".", ",")
        s = f"{v:,.{casas}f}".rstrip("0").rstrip(".")
        s = s.replace(",", "X").replace(".", ",").replace("X", ".")
        return "0" if s in ("-0", "") else s
    return str(v)


def vazio(v) -> bool:
    """Valor ainda não informado (None ou NaN), que não pode servir de padrão."""
    return v is None or (isinstance(v, (float, np.floating)) and math.isnan(v))


class EntradaCancelada(Exception):
    """Levantada quando o usuário digita 'sair' ou encerra sem executar a análise.

    No Jupyter/Colab, aparece como uma mensagem curta, e não como traceback."""

    def _render_traceback_(self):
        return [f"⚠ {self}"]


class Perguntador:
    """Faz as perguntas, valida as respostas e registra a origem de cada premissa.

    modo ``novo``: o valor é obrigatório (não há padrão), exceto quando há
    uma referência técnica explícita, que o usuário aceita com Enter.
    modo ``revisar``: Enter mantém o valor atual (carregado de arquivo).
    """

    def __init__(self, entrada: Callable[[str], str] = input, saida: Callable[..., None] = print, modo: str = "novo"):
        if modo not in ("novo", "revisar"):
            raise ValueError("modo deve ser 'novo' ou 'revisar'")
        self.entrada, self.saida, self.modo = entrada, saida, modo
        self.digitados: List[str] = []                    # caminhos informados pelo usuário no bloco atual
        self.referencias: Dict[str, Fonte] = {}           # caminhos aceitos por referência
        self.rotulos: Dict[str, str] = dict(ROTULOS_CAMINHOS)   # caminho -> rótulo exibido ao usuário
        self.anunciar_prints = False                      # entrada principal: anuncia as telas a capturar

    # -- leitura bruta -------------------------------------------------------
    def _ler(self, prompt: str) -> str:
        txt = self.entrada(prompt)
        txt = "" if txt is None else str(txt).strip()
        if txt.lower() in ("sair", "cancelar"):
            raise EntradaCancelada("entrada interrompida pelo usuário")
        return txt

    def _sufixo(self, atual, referencia):
        if self.modo == "revisar" and not vazio(atual):
            return f" [{_fmt(atual)}]"
        if referencia is not None:
            return f" [referência: {_fmt(referencia)} — Enter aceita]"
        return ""

    def _padrao(self, atual, referencia):
        if self.modo == "revisar" and not vazio(atual):
            return atual, "mantido"
        if referencia is not None:
            return referencia, "referencia"
        return None, None

    # -- tipos ---------------------------------------------------------------
    def numero(self, rotulo: str, atual=None, minimo: Optional[float] = None, maximo: Optional[float] = None,
               inteiro: bool = False, referencia=None, unidade: str = "") -> Tuple[float, str]:
        un = f" ({unidade})" if unidade else ""
        while True:
            txt = self._ler(f"{rotulo}{un}{self._sufixo(atual, referencia)}: ")
            if not txt:
                v, origem = self._padrao(atual, referencia)
                if v is None:
                    self.saida("  ⚠ Valor obrigatório. Digite um número (vírgula ou ponto decimal).")
                    continue
            else:
                try:
                    v = converter_numero(txt)
                except (ValueError, ZeroDivisionError):
                    self.saida("  ⚠ Valor inválido. Use apenas números, por exemplo 1500, 1.500 ou 1.500,50.")
                    continue
                if usa_ponto_de_milhar(txt):
                    self.saida(f"  → lido como {_fmt(v)} (ponto como separador de milhar; para decimais use vírgula)")
                origem = "usuario"
            if inteiro and abs(v - round(v)) > 1e-9:
                self.saida("  ⚠ Informe um número inteiro.")
                continue
            if minimo is not None and v < minimo:
                self.saida(f"  ⚠ O valor deve ser maior ou igual a {_fmt(minimo)}.")
                continue
            if maximo is not None and v > maximo:
                self.saida(f"  ⚠ O valor deve ser menor ou igual a {_fmt(maximo)}.")
                continue
            return (int(round(v)) if inteiro else float(v)), origem

    def opcao(self, rotulo: str, opcoes: Sequence[Tuple[str, str]], atual=None, referencia=None) -> Tuple[str, str]:
        self.saida(rotulo)
        for k, (_, desc) in enumerate(opcoes, 1):
            self.saida(f"  {k}. {desc}")
        chaves = [c for c, _ in opcoes]
        idx_atual = chaves.index(atual) + 1 if atual in chaves else None
        idx_ref = chaves.index(referencia) + 1 if referencia in chaves else None
        curto = lambda i: opcoes[i - 1][1] if len(opcoes[i - 1][1]) <= 45 else opcoes[i - 1][1][:44] + "…"
        if self.modo == "revisar" and idx_atual is not None:
            sufixo = f" [{idx_atual}: {curto(idx_atual)}]"
        elif idx_ref is not None:
            sufixo = f" [referência: {idx_ref}: {curto(idx_ref)} — Enter aceita]"
        else:
            sufixo = ""
        while True:
            txt = self._ler(f"Opção (1–{len(opcoes)}){sufixo}: ")
            if not txt:
                v, origem = self._padrao(idx_atual, idx_ref)
                if v is None:
                    self.saida("  ⚠ Escolha obrigatória.")
                    continue
                return chaves[v - 1], origem
            if txt.isdecimal() and 1 <= int(txt) <= len(opcoes):
                return chaves[int(txt) - 1], "usuario"
            if txt in chaves:
                return txt, "usuario"
            self.saida("  ⚠ Opção inválida.")

    def sim_nao(self, rotulo: str, atual: Optional[bool] = None, referencia: Optional[bool] = None) -> Tuple[bool, str]:
        while True:
            txt = self._ler(f"{rotulo} (s/n){self._sufixo(atual, referencia)}: ").lower()
            if not txt:
                v, origem = self._padrao(atual, referencia)
                if v is None:
                    self.saida("  ⚠ Responda s ou n.")
                    continue
                return bool(v), origem
            if txt in ("s", "sim", "y"):
                return True, "usuario"
            if txt in ("n", "nao", "não", "no"):
                return False, "usuario"
            self.saida("  ⚠ Responda s ou n.")

    def texto(self, rotulo: str, atual: Optional[str] = None, obrigatorio: bool = True, referencia: Optional[str] = None) -> str:
        while True:
            txt = self._ler(f"{rotulo}{self._sufixo(atual or None, referencia)}: ")
            if txt:
                return txt
            v, _ = self._padrao(atual or None, referencia)
            if v is not None:
                return v
            if not obrigatorio:
                return ""
            self.saida("  ⚠ Texto obrigatório.")

    # -- atalhos que leem e gravam diretamente na configuração ----------------
    def _registrar(self, caminho: str, origem: str, fonte_ref: Optional[Fonte]):
        if origem == "usuario":
            self.digitados.append(caminho)
            self.referencias.pop(caminho, None)
        elif origem == "referencia" and fonte_ref is not None:
            self.referencias[caminho] = fonte_ref

    def campo(self, cfg: EVTEASConfig, caminho: str, rotulo: str, minimo=None, maximo=None, inteiro=False,
              referencia=None, unidade: str = "", fonte_ref: Optional[Fonte] = None) -> float:
        self.rotulos[caminho] = rotulo + (f" ({unidade})" if unidade else "")
        atual = obter_por_caminho(cfg, caminho)
        v, origem = self.numero(rotulo, atual, minimo, maximo, inteiro, referencia, unidade)
        definir_por_caminho(cfg, caminho, v)
        self._registrar(caminho, origem, fonte_ref)
        return v

    def campo_pct(self, cfg, caminho, rotulo, minimo=0.0, maximo=100.0, referencia=None, fonte_ref=None) -> float:
        return self.campo(cfg, caminho, rotulo, minimo, maximo, False, referencia, "%", fonte_ref)

    def campo_sn(self, cfg, caminho, rotulo, referencia=None) -> bool:
        self.rotulos[caminho] = rotulo
        v, origem = self.sim_nao(rotulo, obter_por_caminho(cfg, caminho), referencia)
        definir_por_caminho(cfg, caminho, v)
        return v

    def bloco(self, titulo: str):
        if self.anunciar_prints:
            numero = titulo.split(".")[0]
            if numero.isdigit() and int(numero) in PRINT_DO_BLOCO:
                self.saida("\n" + titulo_print(PRINT_DO_BLOCO[int(numero)]))
        self.saida("\n" + LINHA + f"\n{titulo}\n" + LINHA)
        self.digitados = []


ROTULOS_CAMINHOS = {  # parâmetros perguntados sem P.campo (montados a partir de várias respostas)
    "tecnico.profundidade_media_m": "Volume útil (profundidade média)",
    "tecnico.capacidade_suporte_kg_m3": "Capacidade de suporte (kg/m³)",
    "economico.capex_itens": "Itens de CAPEX",
    "economico.classe_estimativa_aace": "Maturidade da estimativa de CAPEX (AACE)",
    "economico.valor_residual_rs": "Valor residual (R$)",
    "economico.mix_produtos": "Mix de produtos (quantidades e preços)",
    "economico.preco_venda_kg": "Preço médio de venda (R$/kg)",
    "economico.receita_servicos_mes": "Receita de serviços (R$/mês)",
    "economico.meses_ate_primeira_receita": "Meses até a primeira receita",
    "economico.custo_alevino_milheiro": "Custo do alevino (R$/un)",
    "ambiental.energia_kwh_kg": "Consumo de energia (kWh/kg)",
    "economico.manutencao_mes": "Manutenção (R$/mês)",
    "economico.salarios_clt_por_cargo": "Salários dos funcionários CLT",
    "economico.impostos_configurados": "Tributos e alíquotas",
    "ambiental.renovacao_pct_dia": "Renovação diária de água (%)",
}
ROTULOS_KPI = {
    "oee": "OEE aquícola", "fcr": "Conversão alimentar (FCR)", "produtividade": "Produtividade relativa",
    "vpl": "VPL sobre o investimento", "roi": "ROI anual", "margem_seguranca": "Margem de segurança",
    "ph_azul": "Pegada hídrica azul", "ph_cinza": "Pegada hídrica cinza", "ecoeficiencia": "Ecoeficiência",
    "intensidade_carbono": "Intensidade de carbono", "conformidade": "Conformidade ambiental",
    "lso": "Licença Social para Operar (LSO)", "rvl": "Retenção de Valor Local (RVL)", "radar_social": "Radar Social",
}
FONTE_KUBITZA = Fonte("bibliografico", "Kubitza (2017)")
FONTE_CONAMA = Fonte("normativo", "Resolução CONAMA nº 357/2005")
FONTE_SISTEMA = Fonte("bibliografico", "Referência do sistema produtivo no EVTEAS-Py (calibrar com especialista)")
FONTE_AUTOR = Fonte("autor", "Valor de referência do EVTEAS-Py")


def _fontes_do_bloco(P: Perguntador, cfg: EVTEASConfig, nome_bloco: str):
    """Registra a origem das premissas digitadas no bloco (Seção 3.6)."""
    caminhos = list(dict.fromkeys(P.digitados))
    for c, f in P.referencias.items():
        cfg.fontes[c] = f
    if not caminhos:
        return
    if P.modo == "revisar":
        mudar, _ = P.sim_nao(f"Atualizar a origem dos dados do bloco '{nome_bloco}'?", False)
        if not mudar:
            return
    opcoes = list(CATEGORIAS_FONTE.items())
    cat, _ = P.opcao(f"Origem dos dados informados no bloco '{nome_bloco}':", opcoes)
    ref = P.texto("Referência (documento, cotação, autor, data)", obrigatorio=False)
    for c in caminhos:
        cfg.fontes[c] = Fonte(cat, ref)
    detalhar, _ = P.sim_nao("Algum parâmetro deste bloco tem origem diferente?", False if P.modo == "revisar" else None)
    while detalhar:
        for k, c in enumerate(caminhos, 1):
            P.saida(f"  {k}. {P.rotulos.get(c, c)}")
        k, _ = P.numero("Número do parâmetro", None, 1, len(caminhos), inteiro=True)
        cat_k, _ = P.opcao("Origem desse parâmetro:", opcoes)
        cfg.fontes[caminhos[k - 1]] = Fonte(cat_k, P.texto("Referência", obrigatorio=False))
        detalhar, _ = P.sim_nao("Detalhar outro parâmetro?", False if P.modo == "revisar" else None)


# ---------------------------------------------------------------------------
# Blocos do wizard
# ---------------------------------------------------------------------------

TIPOS_ORGANIZACAO = [("empresa", "Empresa"), ("coop_trabalho", "Cooperativa de Trabalho"),
                     ("coop_agro", "Cooperativa Agropecuária")]
MINIMO_COOPERADOS = {"coop_trabalho": 7, "coop_agro": 20}   # Lei nº 12.690/2012; Lei nº 5.764/1971


def bloco_identificacao(P: Perguntador, cfg: EVTEASConfig):
    P.bloco("1. IDENTIFICAÇÃO DO PROJETO")
    cfg.projeto = P.texto("Nome do projeto", cfg.projeto if P.modo == "revisar" else None)
    cfg.autor = P.texto("Responsável pelo estudo", cfg.autor if P.modo == "revisar" else None)
    tipo, _ = P.opcao("Tipo de pessoa jurídica:", TIPOS_ORGANIZACAO,
                      cfg.economico.tipo_organizacao if P.modo == "revisar" else None)
    cfg.economico.tipo_organizacao = tipo


def bloco_tecnico(P: Perguntador, cfg: EVTEASConfig):
    t = cfg.tecnico
    P.bloco("2. DIMENSÃO TÉCNICA — INFRAESTRUTURA")
    t.especie = P.texto("Espécie cultivada", t.especie if P.modo == "revisar" else None,
                        referencia="Tilápia-do-nilo (Oreochromis niloticus)")
    sistemas = [(k, f"{k} (referência: {_fmt(float(v['capacidade_suporte_kg_m3']))} kg/m³; "
                    f"{_fmt(float(v['energia_kwh_kg']))} kWh/kg)") for k, v in SISTEMAS_PRODUTIVOS.items()]
    sistema_anterior = t.sistema_produtivo
    t.sistema_produtivo, _ = P.opcao("Sistema produtivo:", sistemas, t.sistema_produtivo if P.modo == "revisar" else None)
    ref = SISTEMAS_PRODUTIVOS[t.sistema_produtivo]
    if P.modo == "revisar" and sistema_anterior in SISTEMAS_PRODUTIVOS and sistema_anterior != t.sistema_produtivo:
        _trocar_referencias_do_sistema(P, cfg, sistema_anterior)
    vol_atual = t.area_lamina_m2 * t.profundidade_media_m      # volume salvo (antes de alterar a área)
    P.campo(cfg, "tecnico.area_lamina_m2", "Área de lâmina d'água", minimo=1, unidade="m²")
    vol, origem = P.numero("Volume útil total dos viveiros/tanques", vol_atual if P.modo == "revisar" else None,
                           minimo=0.01, unidade="m³")
    t.profundidade_media_m = vol / t.area_lamina_m2
    if origem == "usuario":
        P.digitados.append("tecnico.profundidade_media_m")
    P.saida(f"  → profundidade média equivalente: {_fmt(round(t.profundidade_media_m, 3))} m")
    if not 0.3 <= t.profundidade_media_m <= 10:
        P.saida("  ⚠ Profundidade fora da faixa usual de viveiros e tanques (0,3 a 10 m): confira a área e o volume.")
    P.campo(cfg, "tecnico.numero_tanques", "Número total de viveiros/tanques", minimo=1, inteiro=True)
    P.campo(cfg, "tecnico.tanques_ativos", "Número de viveiros/tanques ativos", minimo=1, maximo=t.numero_tanques, inteiro=True)
    cap_atual = t.capacidade_suporte_kg_m3
    cap, origem = P.numero("Capacidade de suporte ao final do ciclo", cap_atual if P.modo == "revisar" else None,
                           minimo=0.0001, referencia=ref["capacidade_suporte_kg_m3"], unidade="kg/m³")
    t.capacidade_suporte_kg_m3 = cap
    P._registrar("tecnico.capacidade_suporte_kg_m3", origem, FONTE_SISTEMA)
    P.campo(cfg, "tecnico.produtividade_esperada_kg_m2_ciclo", "Produtividade esperada por área (0 = não restringe)",
            minimo=0, unidade="kg/m²/ciclo")
    _fontes_do_bloco(P, cfg, "técnico — infraestrutura")

    P.bloco("3. DIMENSÃO TÉCNICA — PARÂMETROS BIOLÓGICOS E INSUMOS")
    P.campo(cfg, "tecnico.peso_inicial_g", "Peso inicial dos alevinos/juvenis", minimo=0.01, unidade="g")
    P.campo(cfg, "tecnico.peso_final_g", "Peso final de abate/venda", minimo=t.peso_inicial_g * 1.0001, unidade="g")
    P.campo(cfg, "tecnico.ciclo_dias", "Duração do ciclo produtivo", minimo=1, maximo=1095, inteiro=True, unidade="dias")
    P.campo(cfg, "tecnico.ciclos_ano", "Ciclos por ano (escalonados entre os viveiros)", minimo=0.01,
            maximo=math.floor(365 / t.ciclo_dias * 1e4) / 1e4)
    P.campo(cfg, "tecnico.alevinos_por_ciclo",
            "Alevinos estocados por ciclo — total de todos os viveiros ativos (0 = dimensionar pela biomassa-alvo)",
            minimo=0, unidade="unidades")
    P.campo(cfg, "tecnico.fcr", "Conversão alimentar — FCR/TCA", minimo=0.3, maximo=6, unidade="kg ração/kg ganho")
    P.campo_pct(cfg, "tecnico.mortalidade_pct", "Mortalidade esperada no ciclo", maximo=99)
    P.campo(cfg, "tecnico.desempenho_crescimento_pct", "Desempenho de crescimento (peso atingido / peso-alvo)",
            minimo=1, maximo=150, referencia=100.0, unidade="%", fonte_ref=FONTE_AUTOR)
    P.campo(cfg, "tecnico.fcr_referencia", "FCR de referência para o diagnóstico Lean-Green", minimo=0.3, maximo=6,
            referencia=1.40, fonte_ref=FONTE_KUBITZA)
    P.campo_pct(cfg, "tecnico.mortalidade_referencia_pct", "Mortalidade de referência para o diagnóstico Lean-Green",
                referencia=5.0, fonte_ref=FONTE_KUBITZA)
    tec = calcular_tecnico(Contexto(cfg))
    P.saida(f"  → Produção estimada: {_fmt(round(float(tec['producao_kg_ano'][0]), 1))} kg/ano "
            f"({_fmt(round(float(tec['producao_kg_mes'][0]), 1))} kg/mês); restrição ativa: {tec['restricao_biomassa']}; "
            f"alevinos/ciclo: {_fmt(round(float(tec['alevinos_ciclo'][0])))} "
            f"(≈ {_fmt(round(float(tec['alevinos_ciclo'][0]) / max(t.tanques_ativos, 1)))} por viveiro ativo); "
            f"OEE: {_fmt(round(float(tec['oee'][0]), 3))}")
    _fontes_do_bloco(P, cfg, "técnico — parâmetros biológicos e insumos")


REFERENCIAS_DO_SISTEMA = {  # parâmetro -> chave em SISTEMAS_PRODUTIVOS
    "tecnico.capacidade_suporte_kg_m3": "capacidade_suporte_kg_m3",
    "ambiental.energia_kwh_kg": "energia_kwh_kg",
    "ambiental.renovacao_pct_dia": "renovacao_pct_dia",
}


def _trocar_referencias_do_sistema(P: Perguntador, cfg: EVTEASConfig, anterior: str):
    """Ao trocar o sistema produtivo, os valores que eram a referência do sistema anterior
    passam à referência do novo; valores informados pelo usuário são mantidos, com aviso."""
    novo = cfg.tecnico.sistema_produtivo
    for caminho, chave in REFERENCIAS_DO_SISTEMA.items():
        atual = obter_por_caminho(cfg, caminho)
        ref_antiga, ref_nova = SISTEMAS_PRODUTIVOS[anterior][chave], SISTEMAS_PRODUTIVOS[novo][chave]
        rotulo = P.rotulos.get(caminho, caminho)
        if vazio(atual) or abs(float(atual) - float(ref_antiga)) <= 1e-9 * max(1.0, abs(ref_antiga)):
            definir_por_caminho(cfg, caminho, float(ref_nova))
            cfg.fontes[caminho] = FONTE_SISTEMA
            P.saida(f"  → {rotulo}: referência do sistema {anterior} ({_fmt(float(ref_antiga))}) substituída pela do "
                    f"sistema {novo} ({_fmt(float(ref_nova))}).")
        else:
            P.saida(f"  ⚠ {rotulo} = {_fmt(float(atual))} foi informado para o sistema {anterior}; a referência do "
                    f"sistema {novo} é {_fmt(float(ref_nova))}. Revise-o no bloco correspondente.")


def bloco_investimento(P: Perguntador, cfg: EVTEASConfig):
    e = cfg.economico
    P.bloco("4. DIMENSÃO ECONÔMICA — INVESTIMENTO (CAPEX)")
    detalhar, _ = P.sim_nao("Detalhar o CAPEX por item (s) ou informar o total (n)?",
                            (len(e.capex_itens) > 1) if (P.modo == "revisar" and e.capex_itens) else None)
    if detalhar:
        if P.modo == "revisar" and e.capex_itens:
            P.saida("Itens atuais: " + "; ".join(f"{k}: R$ {_fmt(v)}" for k, v in e.capex_itens.items()))
            refazer, _ = P.sim_nao("Refazer a lista de itens?", False)
        else:
            refazer = True
        if refazer:
            itens: Dict[str, float] = {}
            while True:
                nome = P.texto(f"Item de CAPEX nº {len(itens) + 1} (Enter vazio encerra)", obrigatorio=not itens)
                if not nome:
                    break
                base_nome, k = nome, 2
                while nome in itens:                     # nomes repetidos não se sobrescrevem
                    nome, k = f"{base_nome} ({k})", k + 1
                if nome != base_nome:
                    P.saida(f"  (já existe um item '{base_nome}'; este será registrado como '{nome}')")
                itens[nome], _ = P.numero(f"  Valor de '{nome}'", None, 0, unidade="R$")
                P.saida(f"  → subtotal do CAPEX: R$ {_fmt(sum(itens.values()))}")
            e.capex_itens = itens
    else:
        total, _ = P.numero("Investimento inicial total — CAPEX",
                            e.capex_total if (P.modo == "revisar" and e.capex_itens) else None, 0, unidade="R$")
        e.capex_itens = {"CAPEX total": total}
    P.digitados.append("economico.capex_itens")
    P.saida(f"  → CAPEX total: R$ {_fmt(e.capex_total)}")
    classes = [(k, f"Classe {k} (faixa de exatidão {int(lo * 100)}% a +{int(hi * 100)}%)")
               for k, (lo, hi) in sorted(FAIXAS_AACE.items())]
    cl, origem = P.opcao("Maturidade da estimativa de CAPEX (AACE International):", classes,
                         e.classe_estimativa_aace if P.modo == "revisar" else None, referencia=5)
    e.classe_estimativa_aace = int(cl)
    P._registrar("economico.classe_estimativa_aace", origem, Fonte("normativo", "AACE International (2020)"))
    P.campo(cfg, "economico.capital_giro", "Capital de giro inicial", minimo=0, unidade="R$")
    P.campo(cfg, "economico.vida_util_anos", "Vida útil média dos ativos para depreciação", minimo=0, maximo=100,
            unidade="anos")
    atual_res = (e.valor_residual_rs if e.valor_residual_rs is not None else e.capex_total * e.valor_residual_pct / 100)
    residual, origem = P.numero("Valor residual dos ativos ao final do horizonte",
                                atual_res if P.modo == "revisar" else None, minimo=0, unidade="R$")
    e.valor_residual_rs = residual          # guardado em R$: não muda se o CAPEX for alterado depois
    e.valor_residual_pct = 100 * residual / e.capex_total if e.capex_total else 0.0
    if origem == "usuario":
        P.digitados.append("economico.valor_residual_rs")
    P.campo_pct(cfg, "economico.fomento_nao_reembolsavel_pct", "Parcela do CAPEX coberta por fomento não reembolsável")
    _fontes_do_bloco(P, cfg, "investimento")


def bloco_receitas(P: Perguntador, cfg: EVTEASConfig):
    e = cfg.economico
    P.bloco("5. DIMENSÃO ECONÔMICA — MIX DE PRODUTOS E RECEITAS (a 100% da capacidade)")
    if P.modo == "revisar" and not e.mix_produtos:
        prod = e.producao_vendas_kg_mes or float(calcular_tecnico(Contexto(cfg))["producao_kg_mes"][0])
        e.mix_produtos = [{"nome": "Produto principal", "quantidade_kg_mes": round(prod, 4),
                           "preco_kg": e.preco_venda_kg, "participacao_pct": 100.0}]
    for p in e.mix_produtos:          # produto informado apenas pela participação (%) → quantidade equivalente
        if "quantidade_kg_mes" not in p:
            prod = e.producao_vendas_kg_mes or float(calcular_tecnico(Contexto(cfg))["producao_kg_mes"][0])
            p["quantidade_kg_mes"] = prod * float(p.get("participacao_pct", 100)) / 100
    n_atual = len(e.mix_produtos) if (P.modo == "revisar" and e.mix_produtos) else None
    n, _ = P.numero("Quantidade de produtos no mix de vendas", n_atual, 1, 20, inteiro=True)
    mix = []
    for i in range(n):
        ant = e.mix_produtos[i] if (P.modo == "revisar" and i < len(e.mix_produtos)) else {}
        P.saida(f"\nProduto #{i + 1}")
        nome = P.texto("  Nome do produto", ant.get("nome") or (f"Produto {i + 1}" if ant else None))
        qtd, _ = P.numero("  Quantidade vendida por mês", ant.get("quantidade_kg_mes"), 0, unidade="kg/mês")
        preco, _ = P.numero("  Preço de venda", ant.get("preco_kg"), 0, unidade="R$/kg")
        mix.append({"nome": nome, "quantidade_kg_mes": qtd, "preco_kg": preco})
    total = sum(p["quantidade_kg_mes"] for p in mix)
    for p in mix:
        p["participacao_pct"] = 100 * p["quantidade_kg_mes"] / total if total else 100 / len(mix)
    e.mix_produtos = mix
    e.preco_venda_kg = (sum(p["quantidade_kg_mes"] * p["preco_kg"] for p in mix) / total if total
                        else float(np.mean([p["preco_kg"] for p in mix])))
    P.digitados += ["economico.mix_produtos", "economico.preco_venda_kg"]
    P.saida(f"  → Preço médio ponderado: R$ {_fmt(round(e.preco_venda_kg, 4))}/kg")

    prod_tec = float(calcular_tecnico(Contexto(cfg))["producao_kg_mes"][0])
    opcoes = [("tecnico", f"Produção calculada pelo módulo técnico ({_fmt(round(prod_tec, 1))} kg/mês)")]
    if total > 0:
        opcoes.append(("mix", f"Quantidades informadas no mix ({_fmt(round(total, 1))} kg/mês)"))
    else:
        P.saida("  (as quantidades do mix somam 0 kg/mês: o volume de vendas será a produção técnica)")
    atual = ("mix" if e.producao_vendas_kg_mes > 0 and total > 0 else "tecnico") if P.modo == "revisar" else None
    base, _ = P.opcao("Volume de vendas a utilizar:", opcoes, atual)
    e.producao_vendas_kg_mes = total if base == "mix" else 0.0
    if total and prod_tec > 0 and abs(total - prod_tec) / prod_tec > 0.10:
        P.saida(f"  ⚠ O mix difere {_fmt(round(abs(total / prod_tec - 1) * 100))}% da produção técnica; verifique as premissas.")
    elif prod_tec <= 0:
        P.saida("  ⚠ A produção técnica calculada é zero; verifique os parâmetros técnicos.")

    servicos, _ = P.sim_nao("Há receita de serviços (ex.: transporte para terceiros)?",
                            (e.receita_servicos_mes > 0) if (P.modo == "revisar" and not vazio(e.receita_servicos_mes)) else None)
    if servicos:
        modo_km, _ = P.sim_nao("Calcular por quilômetro rodado (valor/km × km/mês)?", None if P.modo == "novo" else False)
        if modo_km:
            vkm, _ = P.numero("  Valor cobrado por km", None, 0, unidade="R$/km")
            km, _ = P.numero("  Quilômetros rodados por mês", None, 0, unidade="km/mês")
            e.receita_servicos_mes = vkm * km
        else:
            P.campo(cfg, "economico.receita_servicos_mes", "Receita mensal de serviços", minimo=0, unidade="R$/mês")
        P.digitados.append("economico.receita_servicos_mes")
    else:
        e.receita_servicos_mes = 0.0
    L_calc = int(math.ceil(cfg.tecnico.ciclo_dias / 30.4375))
    P.saida(f"  (a primeira despesca ocorre após um ciclo de {cfg.tecnico.ciclo_dias} dias ≈ {L_calc} meses; "
            "informe 0 se a receita começar no primeiro mês)")
    meses, origem = P.numero("Meses até a primeira receita (carência)",
                             meses_ate_despesca(cfg) if P.modo == "revisar" else None, 0, 60, inteiro=True,
                             referencia=L_calc)
    if origem == "referencia":
        e.carencia_automatica, e.meses_ate_primeira_receita = True, 0      # acompanha a duração do ciclo
    elif origem == "usuario":
        e.carencia_automatica, e.meses_ate_primeira_receita = False, meses
    P._registrar("economico.meses_ate_primeira_receita", origem, FONTE_AUTOR)
    _fontes_do_bloco(P, cfg, "receitas")


def bloco_custos(P: Perguntador, cfg: EVTEASConfig):
    e, a = cfg.economico, cfg.ambiental
    P.bloco("6. DIMENSÃO ECONÔMICA — CUSTOS OPERACIONAIS")
    P.campo(cfg, "economico.custo_racao_kg", "Custo da ração", minimo=0, unidade="R$/kg")
    un, origem = P.numero("Custo do alevino por unidade", e.custo_alevino_milheiro / 1000 if P.modo == "revisar" else None,
                          minimo=0, unidade="R$/un")
    e.custo_alevino_milheiro = un * 1000
    if origem == "usuario":
        P.digitados.append("economico.custo_alevino_milheiro")
    ref_kwh = SISTEMAS_PRODUTIVOS[cfg.tecnico.sistema_produtivo]["energia_kwh_kg"]
    kwh, origem = P.numero("Consumo de energia (aeração e bombeamento)", a.energia_kwh_kg if P.modo == "revisar" else None,
                           minimo=0, referencia=ref_kwh, unidade="kWh/kg produzido")
    a.energia_kwh_kg = kwh
    P._registrar("ambiental.energia_kwh_kg", origem, FONTE_SISTEMA)
    P.campo(cfg, "economico.tarifa_energia_kwh", "Tarifa de energia elétrica", minimo=0, unidade="R$/kWh")
    P.campo(cfg, "economico.outros_custos_variaveis_kg", "Outros custos variáveis (despesca, gelo, frete, embalagem)",
            minimo=0, unidade="R$/kg vendido")
    P.campo(cfg, "economico.outros_custos_variaveis_mes",
            "Outros custos variáveis mensais a 100% da capacidade (matéria-prima e insumos não listados; 0 se não houver)",
            minimo=0, unidade="R$/mês")
    P.campo(cfg, "economico.assistencia_tecnica_mes", "Assistência técnica", minimo=0, unidade="R$/mês")
    P.campo(cfg, "economico.administrativo_mes", "Outros custos fixos (aluguel, contabilidade, seguros, administração)",
            minimo=0, unidade="R$/mês")
    atual_man = e.manutencao_mes if e.manutencao_mes is not None else e.capex_total * e.manutencao_capex_pct_aa / 100 / 12
    man, origem = P.numero("Manutenção de instalações e equipamentos", atual_man if P.modo == "revisar" else None,
                           minimo=0, unidade="R$/mês")
    e.manutencao_mes = man                  # guardada em R$/mês: não muda se o CAPEX for alterado depois
    e.manutencao_capex_pct_aa = 100 * man * 12 / e.capex_total if e.capex_total else 0.0
    if origem == "usuario":
        P.digitados.append("economico.manutencao_mes")
    tec = calcular_tecnico(Contexto(cfg))
    prod_tec = float(tec["producao_kg_mes"][0])
    prod = e.producao_vendas_kg_mes or prod_tec
    racao = prod * float(tec["racao_por_kg_produzido"][0]) * e.custo_racao_kg
    # como no modelo: a estocagem acompanha o volume de vendas
    alev = float(tec["alevinos_ano"][0]) / 12 * e.custo_alevino_milheiro / 1000 * (prod / prod_tec if prod_tec else 1.0)
    ener = prod * kwh * e.tarifa_energia_kwh
    outros = prod * e.outros_custos_variaveis_kg + e.outros_custos_variaveis_mes
    P.saida(f"  → Custo variável estimado a 100%: R$ {_fmt(round(racao + alev + ener + outros, 2))}/mês "
            f"(ração R$ {_fmt(round(racao, 2))}; alevinos R$ {_fmt(round(alev, 2))}; energia R$ {_fmt(round(ener, 2))}; "
            f"outros R$ {_fmt(round(outros, 2))})")
    _fontes_do_bloco(P, cfg, "custos")


def bloco_pessoas(P: Perguntador, cfg: EVTEASConfig):
    e = cfg.economico
    P.bloco("7. PESSOAS — COOPERADOS E FUNCIONÁRIOS CLT")
    if e.tipo_organizacao != "empresa":
        minimo = MINIMO_COOPERADOS[e.tipo_organizacao]
        P.campo(cfg, "economico.retirada_cooperados_mes", "Retirada/pró-labore mensal TOTAL dos cooperados",
                minimo=0, unidade="R$/mês")
        while True:
            P.campo(cfg, "economico.cooperados_trabalhadores", "Cooperados que trabalham na operação", minimo=0, inteiro=True)
            forn, _ = P.sim_nao("Há cooperados que apenas entregam matéria-prima ou não atuam na operação?",
                                (e.cooperados_fornecedores > 0) if P.modo == "revisar" else None)
            if forn:
                P.campo(cfg, "economico.cooperados_fornecedores", "  Número desses cooperados", minimo=1, inteiro=True)
                e.nome_materia_prima = P.texto("  Principal matéria-prima fornecida (se houver)",
                                               e.nome_materia_prima or None, obrigatorio=False)
            else:
                e.cooperados_fornecedores = 0
            total = e.cooperados_trabalhadores + e.cooperados_fornecedores
            if total >= minimo:
                break
            P.saida(f"  ⚠ A legislação exige pelo menos {minimo} cooperados (informado: {total}).")
            seguir, _ = P.sim_nao("Prosseguir mesmo assim (ex.: grupo ainda em formalização)?",
                                  True if P.modo == "revisar" else None)
            if seguir:
                break
        if e.cooperados_trabalhadores and e.cooperados_fornecedores:
            P.campo_pct(cfg, "economico.perc_sobras_trabalhadores", "Parcela das sobras destinada aos cooperados trabalhadores")
        else:
            e.perc_sobras_trabalhadores = 100.0 if e.cooperados_trabalhadores else 0.0
    else:
        e.retirada_cooperados_mes = 0.0
        e.cooperados_trabalhadores = 0
        e.cooperados_fornecedores = 0
        e.perc_sobras_trabalhadores = 100.0
    sem_cargos_com_mao_obra = P.modo == "revisar" and not vazio(e.mao_obra_mes) and e.mao_obra_mes > 0
    minimo_clt = 1 if (e.tipo_organizacao == "empresa" and not sem_cargos_com_mao_obra) else 0
    n_atual = len(e.salarios_clt_por_cargo) if P.modo == "revisar" else None
    n, _ = P.numero(f"Número de funcionários CLT (mínimo {minimo_clt})", n_atual, minimo_clt, 500, inteiro=True)
    antigos = list(e.salarios_clt_por_cargo.items())
    cargos: Dict[str, float] = {}
    for i in range(n):
        ant_nome, ant_sal = antigos[i] if (P.modo == "revisar" and i < len(antigos)) else (None, None)
        cargo = P.texto(f"  Cargo do funcionário CLT nº {i + 1}", ant_nome)
        while cargo in cargos:
            cargo = f"{cargo} ({i + 1})"
        cargos[cargo], _ = P.numero(f"  Salário bruto mensal de '{cargo}'", ant_sal, 0, unidade="R$")
    e.salarios_clt_por_cargo = cargos
    if n:
        P.digitados.append("economico.salarios_clt_por_cargo")
    P.campo(cfg, "economico.mao_obra_mes",
            "Outra mão de obra não detalhada acima (diaristas, terceiros; 0 se não houver)", minimo=0, unidade="R$/mês")
    if e.mao_obra_mes > 0:
        P.campo_pct(cfg, "economico.encargos_mao_obra_pct", "Encargos sobre essa mão de obra", maximo=200)
    else:
        e.encargos_mao_obra_pct = 0.0
    _fontes_do_bloco(P, cfg, "pessoas")


TRIBUTOS = [  # nome, base, sugestão de alíquota (%)
    ("ICMS", "produtos", None), ("IPI", "produtos", None), ("ISS", "servicos", None),
    ("PIS (faturamento)", "faturamento", 0.65), ("COFINS", "faturamento", 3.0),
    ("CSLL", "lucro", 9.0), ("IRPJ", "lucro", 15.0),
    ("INSS patronal (CLT)", "folha_clt", 20.0), ("FGTS (CLT)", "folha_clt", 8.0), ("PIS sobre a folha (CLT)", "folha_clt", 1.0),
    ("INSS sobre a retirada dos cooperados", "pro_labore", 20.0),
]


def detalhar_tributos_consolidados(cfg: EVTEASConfig) -> None:
    """Converte alíquotas consolidadas (taxa sobre faturamento/lucro) em itens da lista de
    tributos do wizard, sem alterar o resultado do modelo (as bases são aditivas)."""
    e = cfg.economico
    for campo, base, nome in (("taxa_impostos_faturamento_pct", "faturamento", "Tributos sobre o faturamento (consolidado)"),
                              ("taxa_impostos_lucro_pct", "lucro", "Tributos sobre o lucro (consolidado)")):
        v = getattr(e, campo)
        if v and not (isinstance(v, float) and math.isnan(v)) and v > 0:
            e.impostos_configurados = list(e.impostos_configurados) + [{"nome": nome, "base": base, "aliquota_pct": v}]
        setattr(e, campo, 0.0)


def bloco_tributos(P: Perguntador, cfg: EVTEASConfig):
    e = cfg.economico
    P.bloco("8. CONFIGURAÇÃO TRIBUTÁRIA (responda s para configurar cada tributo)")
    detalhar_tributos_consolidados(cfg)
    atuais = {i.get("nome"): i for i in e.impostos_configurados}
    tem_clt = bool(e.salarios_clt_por_cargo)
    tem_ret = e.retirada_cooperados_mes > 0
    novos = []
    for nome, base, sug in TRIBUTOS:
        if base == "folha_clt" and not tem_clt:
            continue
        if base == "pro_labore" and not tem_ret:
            continue
        ant = atuais.get(nome)
        dica = f" (alíquota usual: {_fmt(sug)}%)" if sug is not None else ""
        # base que passou a existir na revisão (ex.: primeiro funcionário CLT): sem decisão anterior, sem padrão
        sem_decisao = base in ("folha_clt", "pro_labore") and not any(i.get("base") == base for i in atuais.values())
        usar, _ = P.sim_nao(f"Configurar {nome}{dica}?",
                            (ant is not None) if (P.modo == "revisar" and not sem_decisao) else None)
        if usar:
            atual_aliq = None
            if ant:   # arquivo com 'aliquota' em fração
                atual_aliq = ant.get("aliquota_pct", float(ant["aliquota"]) * 100 if "aliquota" in ant else None)
            aliq, _ = P.numero(f"  Alíquota de {nome}", atual_aliq, 0, 100, unidade="%")
            novos.append({"nome": nome, "base": base, "aliquota_pct": aliq})
    outros = [i for i in e.impostos_configurados if i.get("nome") not in {t[0] for t in TRIBUTOS}]
    if P.modo == "revisar" and outros:
        manter, _ = P.sim_nao(f"Manter os demais tributos já cadastrados ({', '.join(i['nome'] for i in outros)})?", True)
        if manter:
            novos += outros
    mais, _ = P.sim_nao("Incluir outro tributo ou contribuição?", False if P.modo == "revisar" else None)
    bases = [("faturamento", "Receita bruta total"), ("produtos", "Receita de produtos"), ("servicos", "Receita de serviços"),
             ("lucro", "Lucro antes de IR/CSLL"), ("folha_clt", "Folha CLT"), ("pro_labore", "Retirada dos cooperados")]
    while mais:
        nome = P.texto("  Nome do tributo")
        base, _ = P.opcao("  Base de cálculo:", bases)
        aliq, _ = P.numero("  Alíquota", None, 0, 100, unidade="%")
        novos.append({"nome": nome, "base": base, "aliquota_pct": aliq})
        mais, _ = P.sim_nao("Incluir outro tributo?", None)
    e.impostos_configurados = novos
    aq = aliquotas_configuradas(cfg)
    nomes_bases = dict(bases) | {"folha": "Outra mão de obra"}
    P.saida("  → Alíquotas consolidadas: " + "; ".join(f"{nomes_bases.get(k, k)} {_fmt(round(v * 100, 4))}%"
                                                     for k, v in aq.items() if v))
    P.digitados.append("economico.impostos_configurados")
    _fontes_do_bloco(P, cfg, "tributos")


def bloco_projecao(P: Perguntador, cfg: EVTEASConfig):
    e = cfg.economico
    P.bloco("9. RAMPA DE CAPACIDADE, PROJEÇÃO DE LONGO PRAZO E TMA")
    rampa_atual = None
    if P.modo == "revisar" and not vazio(e.capacidade_inicial_pct):
        rampa_atual = bool(e.capacidade_inicial_pct < 100 or (not vazio(e.meses_rampa) and e.meses_rampa > 0))
    rampa, _ = P.sim_nao("Haverá aumento gradual da capacidade produtiva (rampa)?", rampa_atual)
    if rampa:
        P.campo_pct(cfg, "economico.capacidade_inicial_pct", "Capacidade produtiva inicial", minimo=1)
        P.campo(cfg, "economico.meses_rampa", "Meses até atingir 100% da capacidade", minimo=1, maximo=120, inteiro=True)
    else:
        e.capacidade_inicial_pct, e.meses_rampa = 100.0, 0
    h_min = meses_ate_despesca(cfg) // 12 + 1
    P.campo(cfg, "economico.horizonte_anos", f"Horizonte de análise (mínimo {h_min}, para incluir a carência)",
            minimo=h_min, maximo=50, inteiro=True, unidade="anos")
    P.campo(cfg, "economico.crescimento_vendas_aa_pct", "Crescimento anual de vendas", minimo=-50, maximo=100, unidade="% a.a.")
    P.campo(cfg, "economico.crescimento_custos_aa_pct", "Inflação anual de custos e despesas", minimo=-50, maximo=100, unidade="% a.a.")
    P.campo(cfg, "economico.crescimento_preco_aa_pct", "Reajuste anual de preços", minimo=-50, maximo=100, unidade="% a.a.")
    P.campo(cfg, "economico.tma_aa_pct", "Taxa Mínima de Atratividade (TMA)", minimo=0, maximo=100, unidade="% a.a.")
    _fontes_do_bloco(P, cfg, "projeção")


FAIXAS_APP = [  # Lei nº 12.651/2012, art. 4º, I
    (30.0, "Curso d'água com menos de 10 m de largura — 30 m"),
    (50.0, "Curso d'água de 10 a 50 m — 50 m"),
    (100.0, "Curso d'água de 50 a 200 m — 100 m"),
    (200.0, "Curso d'água de 200 a 600 m — 200 m"),
    (500.0, "Curso d'água com mais de 600 m — 500 m"),
    (-1.0, "Outra situação (informar a faixa)"),
]


def bloco_ambiental(P: Perguntador, cfg: EVTEASConfig):
    a, t = cfg.ambiental, cfg.tecnico
    ref = SISTEMAS_PRODUTIVOS[t.sistema_produtivo]
    P.bloco("10. DIMENSÃO AMBIENTAL — ÁGUA E EFLUENTES")
    ren, origem = P.numero("Renovação diária de água", a.renovacao_pct_dia if P.modo == "revisar" else None, 0, 100,
                           referencia=ref["renovacao_pct_dia"], unidade="% do volume/dia")
    a.renovacao_pct_dia = ren
    P._registrar("ambiental.renovacao_pct_dia", origem, FONTE_SISTEMA)
    P.campo_sn(cfg, "ambiental.esvaziamento_por_ciclo", "Os viveiros são esvaziados a cada ciclo?")
    P.campo(cfg, "ambiental.evaporacao_mm_dia", "Evaporação", minimo=0, maximo=20, referencia=4.0, unidade="mm/dia",
            fonte_ref=Fonte("bibliografico", "Kubitza (2017); calibrar com dados climáticos locais"))
    P.campo(cfg, "ambiental.infiltracao_mm_dia", "Infiltração", minimo=0, maximo=50, referencia=2.0, unidade="mm/dia",
            fonte_ref=FONTE_KUBITZA)
    P.campo(cfg, "ambiental.fracao_retorno_perdida", "Fração da água devolvida a outra bacia (0 a 1)", minimo=0, maximo=1,
            referencia=0.0, fonte_ref=FONTE_AUTOR)
    P.campo_pct(cfg, "ambiental.proteina_racao_pct", "Proteína bruta da ração", minimo=1, maximo=70, referencia=32.0, fonte_ref=FONTE_KUBITZA)
    P.campo_pct(cfg, "ambiental.fosforo_racao_pct", "Fósforo da ração", maximo=10, referencia=1.0, fonte_ref=FONTE_KUBITZA)
    P.campo_pct(cfg, "ambiental.nitrogenio_peixe_pct", "Nitrogênio retido no peixe (peso vivo)", maximo=10, referencia=2.6, fonte_ref=FONTE_KUBITZA)
    P.campo_pct(cfg, "ambiental.fosforo_peixe_pct", "Fósforo retido no peixe (peso vivo)", maximo=10, referencia=0.7, fonte_ref=FONTE_KUBITZA)
    P.campo_pct(cfg, "ambiental.remocao_n_tratamento_pct", "Remoção de nitrogênio no tratamento de efluentes")
    P.campo_pct(cfg, "ambiental.remocao_p_tratamento_pct", "Remoção de fósforo no tratamento de efluentes")
    classes = [(k, f"{k} (N total {_fmt(float(v['n_max_mg_l']))} mg/L; P total {_fmt(float(v['p_max_mg_l']))} mg/L)")
               for k, v in CLASSES_CORPO_RECEPTOR.items()]
    classes.append(("manual", "Informar os limites manualmente"))
    atual = next((k for k, v in CLASSES_CORPO_RECEPTOR.items()
                  if v["n_max_mg_l"] == a.n_max_mg_l and v["p_max_mg_l"] == a.p_max_mg_l), "manual")
    cl, origem = P.opcao("Corpo receptor dos efluentes (Resolução CONAMA nº 357/2005):", classes,
                         atual if P.modo == "revisar" else None)
    if cl == "manual":
        P.campo(cfg, "ambiental.n_max_mg_l", "Limite de nitrogênio total", minimo=0.001, unidade="mg/L")
        P.campo(cfg, "ambiental.p_max_mg_l", "Limite de fósforo total", minimo=0.0001, unidade="mg/L")
    else:
        a.n_max_mg_l = CLASSES_CORPO_RECEPTOR[cl]["n_max_mg_l"]
        a.p_max_mg_l = CLASSES_CORPO_RECEPTOR[cl]["p_max_mg_l"]
        cfg.fontes["ambiental.n_max_mg_l"] = cfg.fontes["ambiental.p_max_mg_l"] = Fonte("normativo", f"CONAMA 357/2005 — {cl}")
    P.campo(cfg, "ambiental.n_natural_mg_l", "Concentração natural de N total no corpo receptor", minimo=0,
            maximo=a.n_max_mg_l * 0.999, referencia=min(0.5, a.n_max_mg_l * 0.5), unidade="mg/L",
            fonte_ref=Fonte("autor", "Calibrar com monitoramento local"))
    P.campo(cfg, "ambiental.p_natural_mg_l", "Concentração natural de P total no corpo receptor", minimo=0,
            maximo=a.p_max_mg_l * 0.999, referencia=min(0.02, a.p_max_mg_l * 0.5), unidade="mg/L",
            fonte_ref=Fonte("autor", "Calibrar com monitoramento local"))
    _fontes_do_bloco(P, cfg, "água e efluentes")

    P.bloco("11. DIMENSÃO AMBIENTAL — ENERGIA, EMISSÕES E RESÍDUOS")
    a.fonte_energia, _ = P.opcao("Fonte de energia elétrica:", FONTES_ENERGIA, a.fonte_energia if P.modo == "revisar" else None)
    if a.fonte_energia == "mista":
        P.campo(cfg, "ambiental.fracao_renovavel_mista", "Fração renovável da energia (0 a 1)", minimo=0, maximo=1)
    if a.fonte_energia in ("rede", "mista"):
        P.campo(cfg, "ambiental.fator_emissao_rede_kgco2_kwh", "Fator de emissão da rede", minimo=0, maximo=2,
                referencia=0.0817, unidade="kgCO2e/kWh", fonte_ref=Fonte("secundario", "Fator médio do SIN (MCTI); atualizar ao ano-base"))
    P.campo(cfg, "ambiental.fator_emissao_racao_kgco2_kg", "Fator de emissão da ração", minimo=0, maximo=10,
            referencia=1.1, unidade="kgCO2e/kg", fonte_ref=Fonte("bibliografico", "Estimativa bibliográfica; validar"))
    P.campo(cfg, "ambiental.diesel_l_ano", "Consumo de diesel (geradores, veículos, despesca)", minimo=0, unidade="L/ano")
    P.campo(cfg, "ambiental.fator_emissao_diesel_kgco2_l", "Fator de emissão do diesel", minimo=0, maximo=5,
            referencia=2.6, unidade="kgCO2e/L", fonte_ref=Fonte("bibliografico", "Fator de combustão do diesel; validar"))
    P.campo(cfg, "ambiental.residuos_kg_por_kg", "Resíduos sólidos e lodo gerados", minimo=0, maximo=2, referencia=0.04,
            unidade="kg/kg de peixe", fonte_ref=FONTE_AUTOR)
    P.campo_pct(cfg, "ambiental.residuos_reaproveitados_pct", "Resíduos reaproveitados (compostagem, ração, adubo)")
    P.campo(cfg, "ambiental.energia_referencia_kwh_kg", "Consumo de energia de referência (diagnóstico Lean-Green)",
            minimo=0, referencia=0.40, unidade="kWh/kg", fonte_ref=FONTE_AUTOR)
    _fontes_do_bloco(P, cfg, "energia e resíduos")

    P.bloco("12. DIMENSÃO AMBIENTAL — CONFORMIDADE (checklist normativo)")
    faixa_atual = next((f for f, _ in FAIXAS_APP if f == a.faixa_app_minima_m), -1.0)
    faixa, _ = P.opcao("Faixa mínima de APP aplicável (Lei nº 12.651/2012), medida a partir da borda da calha do leito regular:",
                       [(f, d) for f, d in FAIXAS_APP], faixa_atual if P.modo == "revisar" else None)
    if faixa < 0:
        P.campo(cfg, "ambiental.faixa_app_minima_m", "Faixa mínima de APP", minimo=0, unidade="m")
    else:
        a.faixa_app_minima_m = faixa
        cfg.fontes["ambiental.faixa_app_minima_m"] = Fonte("normativo", "Lei nº 12.651/2012, art. 4º, I")
    P.campo(cfg, "ambiental.distancia_app_m",
            "Distância das instalações até a margem do curso d'água (borda da calha do leito regular)", minimo=0, unidade="m")
    if a.distancia_app_m < a.faixa_app_minima_m:
        P.saida(f"  ⚠ As instalações estão a {_fmt(a.distancia_app_m)} m da margem, dentro da faixa de APP de "
                f"{_fmt(a.faixa_app_minima_m)} m: o item 'Instalações fora da APP' será considerado NÃO atendido.")
    for chave, (descricao, _, elim) in CHECKLIST_AMBIENTAL.items():
        marca = " [ELIMINATÓRIO]" if elim else ""
        v, _ = P.sim_nao(f"{descricao}{marca}", a.checklist.get(chave) if P.modo == "revisar" else None)
        a.checklist[chave] = v
    if a.checklist.get("respeita_app") and a.distancia_app_m < a.faixa_app_minima_m:
        P.saida("  ⚠ Pela distância informada, as instalações estão dentro da APP: o item continua NÃO atendido "
                "(a resposta do checklist não prevalece sobre a distância).")
    _fontes_do_bloco(P, cfg, "conformidade")


FONTES_ENERGIA = [("rede", "Rede pública"), ("solar", "Solar fotovoltaica"), ("biomassa", "Biomassa"),
                  ("mista", "Mista (rede + renovável)")]
RELACOES = [("inexistente", "Inexistente"), ("parcial", "Parcial (contatos eventuais)"),
            ("consolidado", "Consolidado (canal permanente e acordos)")]


def bloco_social(P: Perguntador, cfg: EVTEASConfig):
    s, e = cfg.social, cfg.economico
    P.bloco("13. DIMENSÃO SOCIAL — EMPREGO, RENDA E RELAÇÃO COM A COMUNIDADE")
    sug = e.cooperados_trabalhadores + len(e.salarios_clt_por_cargo)
    if P.modo == "revisar" and sug and s.empregos_diretos != sug:
        P.saida(f"  (cooperados na operação + funcionários CLT = {sug}; o valor salvo é {s.empregos_diretos})")
    P.campo(cfg, "social.empregos_diretos", "Empregos diretos", minimo=0, inteiro=True,
            referencia=sug if sug else None, fonte_ref=FONTE_AUTOR)
    P.campo(cfg, "social.empregos_indiretos", "Empregos indiretos", minimo=0, inteiro=True)
    P.campo_pct(cfg, "social.mao_obra_local_pct", "Mão de obra residente no município/comunidade")
    P.campo_pct(cfg, "social.compras_locais_pct", "Compras de insumos e serviços feitas localmente")
    P.campo(cfg, "social.salario_minimo", "Salário mínimo vigente no ano-base do estudo", minimo=1, unidade="R$")
    P.campo(cfg, "social.capacitacao_horas_pessoa_ano", "Capacitação", minimo=0, unidade="h/pessoa/ano")
    P.campo_pct(cfg, "social.conformidade_nr_pct", "Aderência às Normas Regulamentadoras (NR)")
    s.relacao_comunidade, _ = P.opcao("Relação com a comunidade e pescadores artesanais:", RELACOES,
                                      s.relacao_comunidade if P.modo == "revisar" else None)
    P.campo_sn(cfg, "social.canais_formais_comunicacao", "Existem canais formais de comunicação com a comunidade?")
    P.campo(cfg, "social.reunioes_comunidade_ano", "Reuniões com a comunidade por ano", minimo=0, maximo=365, inteiro=True)
    P.campo(cfg, "social.conflitos_registrados_ano", "Conflitos com stakeholders registrados por ano", minimo=0, inteiro=True)
    P.campo_pct(cfg, "social.mulheres_pct", "Participação de mulheres na equipe")
    P.campo_pct(cfg, "social.jovens_pct", "Participação de jovens (até 29 anos) na equipe")
    P.campo_sn(cfg, "social.publico_vulneravel", "O público beneficiário é vulnerável (ex.: pescadores artesanais, PGTR)?")
    _fontes_do_bloco(P, cfg, "social")

    P.bloco("14. GOVERNANÇA (pilar G do ESG)")
    rotulos = {
        "estatuto_regimento": "Estatuto ou regimento interno aprovado",
        "assembleias_regulares": "Assembleias/reuniões deliberativas regulares",
        "conselho_fiscal": "Conselho fiscal ou instância de controle",
        "prestacao_contas_publica": "Prestação de contas pública",
        "plano_negocios_aprovado": "Plano de negócios aprovado",
        "registro_auditavel_premissas": "Registro auditável das premissas do estudo",
        "canal_denuncia_ouvidoria": "Canal de denúncia ou ouvidoria",
    }
    for chave, rot in rotulos.items():
        v, _ = P.sim_nao(rot, cfg.governanca.itens.get(chave) if P.modo == "revisar" else None)
        cfg.governanca.itens[chave] = v


METODOS_PESOS = [("preset", "Informar diretamente os pesos das quatro dimensões"),
                 ("likert", "Escala Likert de 1 a 5 atribuída por avaliadores (Quadro 7)"),
                 ("ahp", "Comparação par a par (AHP — Saaty)")]
DIMENSOES_ROTULOS = [("tecnico", "Técnica"), ("economico", "Econômica"), ("ambiental", "Ambiental"), ("social", "Social")]


def bloco_pesos(P: Perguntador, cfg: EVTEASConfig):
    p = cfg.pesos
    P.bloco("15. PESOS DO ÍNDICE EVTEAS E REGRAS DE DECISÃO")
    p.metodo, _ = P.opcao("Método de ponderação:", METODOS_PESOS, p.metodo if P.modo == "revisar" else None)
    if p.metodo != "likert":
        p.likert_kpis = {}            # escores de KPIs só se aplicam ao método Likert
    if p.metodo == "preset":
        P.saida("Os pesos serão normalizados para somar 1.")
        for d, nome in DIMENSOES_ROTULOS:
            P.campo(cfg, f"pesos.{d}", f"Peso da dimensão {nome}", minimo=0)
        if p.tecnico + p.economico + p.ambiental + p.social <= 0:
            P.saida("  ⚠ A soma dos pesos deve ser positiva; adotados pesos iguais.")
            p.tecnico = p.economico = p.ambiental = p.social = 1.0
    elif p.metodo == "likert":
        P.saida("Escala: 1 irrelevante · 2 pouco relevante · 3 moderadamente relevante · 4 muito relevante · 5 essencial")
        antigos = list(p.likert_dimensoes.items())
        n_av, _ = P.numero("Número de avaliadores", len(antigos) if (P.modo == "revisar" and antigos) else None, 1, 20, inteiro=True)
        lk = {}
        for i in range(n_av):
            nome_ant, esc_ant = antigos[i] if (P.modo == "revisar" and i < len(antigos)) else (None, {})
            nome = P.texto(f"Nome/área do avaliador nº {i + 1}", nome_ant)
            while nome in lk:
                P.saida("  ⚠ Já existe um avaliador com esse nome; informe outro.")
                nome = P.texto(f"Nome/área do avaliador nº {i + 1}")
            lk[nome] = {}
            for d, rot in DIMENSOES_ROTULOS:
                lk[nome][d], _ = P.numero(f"  {nome} — relevância da dimensão {rot}", esc_ant.get(d), 1, 5, inteiro=True)
        p.likert_dimensoes = lk
        usar_kpi, _ = P.sim_nao("Os avaliadores também pontuarão cada KPI dentro das dimensões?",
                                bool(p.likert_kpis) if P.modo == "revisar" else None)
        if usar_kpi:
            kp = {}
            for nome in lk:
                kp[nome] = {}
                for d, kpis in KPIS_POR_DIMENSAO.items():
                    for k in kpis:
                        kp[nome][k], _ = P.numero(f"  {nome} — relevância do KPI {ROTULOS_KPI.get(k, k)} "
                                                  f"(dimensão {dict(DIMENSOES_ROTULOS)[d].lower()})",
                                                  p.likert_kpis.get(nome, {}).get(k), 1, 5, inteiro=True)
            p.likert_kpis = kp
        else:
            p.likert_kpis = {}
    else:
        P.saida("Escala de Saaty: 1 igual · 3 moderada · 5 forte · 7 muito forte · 9 extrema; use 1/3, 1/5... quando a "
                "segunda dimensão for mais importante.")
        while True:
            A = np.ones((4, 4))
            for i in range(4):
                for j in range(i + 1, 4):
                    ant = p.ahp_matriz[i][j] if (P.modo == "revisar" and p.ahp_matriz) else None
                    v, _ = P.numero(f"Importância de {DIMENSOES_ROTULOS[i][1]} em relação a {DIMENSOES_ROTULOS[j][1]}", ant, 1 / 9, 9)
                    A[i, j], A[j, i] = v, 1 / v
            res = pesos_ahp(A)
            P.saida("  → Pesos: " + "; ".join(f"{n} {_fmt(round(float(w), 3))}" for (_, n), w in zip(DIMENSOES_ROTULOS, res["pesos"]))
                    + f" | razão de consistência = {_fmt(round(float(res['rc']), 3))}")
            p.ahp_matriz = A.tolist()
            if res["consistente"]:
                break
            refazer, _ = P.sim_nao("  ⚠ RC > 0,10 (julgamentos inconsistentes). Refazer as comparações?",
                                   False if P.modo == "revisar" else None)
            if not refazer:
                break
    d = cfg.decisao
    P.campo(cfg, "decisao.limiar_indice_viavel", "Índice EVTEAS mínimo para 'VIÁVEL'", 0, 1, referencia=0.60, fonte_ref=FONTE_AUTOR)
    P.campo(cfg, "decisao.limiar_indice_ressalvas", "Índice EVTEAS mínimo para 'VIÁVEL COM RESSALVAS'", 0,
            d.limiar_indice_viavel, referencia=min(0.50, d.limiar_indice_viavel), fonte_ref=FONTE_AUTOR)
    pmin, origem = P.numero("Probabilidade mínima de VPL positivo no Monte Carlo",
                            d.prob_vpl_positivo_minima * 100 if P.modo == "revisar" else None, 0, 100,
                            referencia=70.0, unidade="%")
    d.prob_vpl_positivo_minima = pmin / 100
    P.campo(cfg, "decisao.lso_minimo", "LSO mínima (0–100)", 0, 100, referencia=40.0, fonte_ref=FONTE_AUTOR)
    P.campo_sn(cfg, "decisao.aplicar_vetos", "Aplicar vetos não compensatórios (VPL < 0, item eliminatório, LSO baixa)?",
               referencia=True)


VARIAVEIS_INCERTAS = [  # caminho, rótulo, escala de exibição (valor exibido = valor interno × escala)
    ("economico.preco_venda_kg", "Preço de venda (R$/kg)", 1.0),
    ("economico.custo_racao_kg", "Custo da ração (R$/kg)", 1.0),
    ("economico.custo_alevino_milheiro", "Custo do alevino (R$/unidade)", 1 / 1000),
    ("economico.tarifa_energia_kwh", "Tarifa de energia (R$/kWh)", 1.0),
    ("economico.custos_fixos_fator", "Fator multiplicador dos custos fixos (1 = valor informado)", 1.0),
    ("tecnico.fcr", "FCR", 1.0),
    ("tecnico.mortalidade_pct", "Mortalidade (%)", 1.0),
    ("tecnico.desempenho_crescimento_pct", "Desempenho de crescimento (%)", 1.0),
]
DOMINIO_INCERTAS = {  # limites (na unidade exibida) dos mínimos e máximos das distribuições
    "economico.preco_venda_kg": (0.0, None), "economico.custo_racao_kg": (0.0, None),
    "economico.custo_alevino_milheiro": (0.0, None), "economico.tarifa_energia_kwh": (0.0, None),
    "economico.custos_fixos_fator": (0.0, None), "economico.capex_fator": (0.01, None),
    "tecnico.fcr": (0.3, 6.0), "tecnico.mortalidade_pct": (0.0, 99.0), "tecnico.desempenho_crescimento_pct": (1.0, 150.0),
}
DOMINIO_INTERNO = {c: tuple(None if v is None else v / escala for v in DOMINIO_INCERTAS[c])
                   for c, _, escala in VARIAVEIS_INCERTAS}
DOMINIO_INTERNO["economico.capex_fator"] = DOMINIO_INCERTAS["economico.capex_fator"]
TIPOS_DIST = [("triangular", "Triangular (mínimo, mais provável, máximo)"),
              ("uniforme", "Uniforme (apenas mínimo e máximo conhecidos)"),
              ("normal", "Normal truncada em [mínimo, máximo]"),
              ("fixa", "Sem incerteza (valor fixo)")]


def ajustar_distribuicao(d: Distribuicao, antigo: float, novo: float, caminho: Optional[str] = None) -> Distribuicao:
    """Reposiciona uma distribuição quando o valor determinístico muda (mantém a amplitude relativa,
    dentro do domínio da variável)."""
    if d.tipo == "fixa":
        return Distribuicao("fixa", novo, novo, novo)
    if antigo and novo:
        f = novo / antigo
        lo, hi = sorted((d.minimo * f, d.maximo * f))
    else:   # com valor nulo, o fator proporcional colapsaria a distribuição: desloca a amplitude
        lo, hi = d.minimo + (novo - antigo), d.maximo + (novo - antigo)
    dom_lo, dom_hi = DOMINIO_INTERNO.get(caminho, (None, None))
    if dom_lo is not None:
        lo = max(lo, dom_lo)
    if dom_hi is not None:
        hi = min(hi, dom_hi)
    moda = novo if d.tipo != "uniforme" else (lo + hi) / 2
    return Distribuicao(d.tipo, min(lo, novo), moda, max(hi, novo))


def distribuicao_coerente(d: Distribuicao, valor: float, tol: float = 1e-9) -> bool:
    """O valor determinístico deve estar no intervalo e ser o mais provável (triangular/normal/fixa)."""
    ok = d.minimo - tol <= valor <= d.maximo + tol
    if d.tipo in ("triangular", "normal", "fixa"):
        ok = ok and abs(d.moda - valor) <= tol * max(1.0, abs(valor))
    return ok


def _perguntar_distribuicao(P: Perguntador, cfg: EVTEASConfig, caminho: str, rotulo: str,
                            atual: Optional[Distribuicao], escala: float = 1.0):
    base = float(obter_por_caminho(cfg, caminho))
    if P.modo == "revisar" and atual is not None and not distribuicao_coerente(atual, base):
        referencia_antiga = atual.moda if atual.tipo != "uniforme" else (atual.minimo + atual.maximo) / 2
        atual = ajustar_distribuicao(atual, referencia_antiga, base, caminho)
        P.saida(f"  (a distribuição salva foi reposicionada proporcionalmente ao valor atual de {rotulo})")
    vb = base * escala
    P.saida(f"\n{rotulo} — valor determinístico informado: {_fmt(vb)}")
    # no modo revisar, variável sem distribuição cadastrada continua fixa (Enter preserva o resultado)
    tipo_atual = (atual.tipo if atual else "fixa") if P.modo == "revisar" else None
    tipo, _ = P.opcao("  Tipo de distribuição:", TIPOS_DIST, tipo_atual)
    if tipo == "fixa":
        return Distribuicao("fixa", base, base, base)
    if tipo in ("triangular", "normal"):
        P.saida(f"  Valor mais provável = valor determinístico informado ({_fmt(vb)})")
    lo_dom, hi_dom = DOMINIO_INCERTAS.get(caminho, (None, None))
    revisa = P.modo == "revisar" and atual is not None and atual.tipo != "fixa"
    tol = 1e-9 * max(1.0, abs(vb))
    while True:
        mn, _ = P.numero("  Mínimo", atual.minimo * escala if revisa else None, lo_dom, hi_dom)
        mx, _ = P.numero("  Máximo", atual.maximo * escala if revisa else None, lo_dom, hi_dom)
        if not (mn - tol <= vb <= mx + tol):
            P.saida(f"  ⚠ O intervalo deve conter o valor determinístico ({_fmt(vb)}). Informe novamente.")
            continue
        try:
            moda = base if tipo != "uniforme" else (mn + mx) / 2 / escala
            # valores digitados na unidade exibida: o arredondamento da escala não pode excluir o valor-base
            return Distribuicao(tipo, min(mn / escala, base), moda, max(mx / escala, base)).validar()
        except ValueError as exc:
            P.saida(f"  ⚠ {exc}. Informe novamente.")


def bloco_incerteza(P: Perguntador, cfg: EVTEASConfig):
    mc = cfg.monte_carlo
    P.bloco("16. INCERTEZA — SIMULAÇÃO DE MONTE CARLO")
    P.saida("Para cada variável, escolha a distribuição: uniforme quando só os limites são conhecidos; triangular "
            "quando também há um valor mais provável (Oliveira; Medeiros Neto, 2012).")
    P.campo(cfg, "monte_carlo.iteracoes", "Número de iterações", minimo=100, maximo=200000, inteiro=True,
            referencia=10000, fonte_ref=FONTE_AUTOR)
    P.campo(cfg, "monte_carlo.seed", "Semente aleatória (seed) para reprodutibilidade", minimo=0, inteiro=True,
            referencia=20260612, fonte_ref=FONTE_AUTOR)
    novas: Dict[str, Distribuicao] = {}
    for caminho, rotulo, escala in VARIAVEIS_INCERTAS:
        novas[caminho] = _perguntar_distribuicao(P, cfg, caminho, rotulo, mc.distribuicoes.get(caminho), escala)
    lo, hi = FAIXAS_AACE[int(cfg.economico.classe_estimativa_aace)]
    P.saida(f"\nCAPEX: a estimativa Classe {cfg.economico.classe_estimativa_aace} da AACE implica faixa de "
            f"{int(lo * 100)}% a +{int(hi * 100)}%.")
    usar_aace, _ = P.sim_nao("Usar essa faixa para a incerteza do CAPEX?",
                             ("economico.capex_fator" not in mc.distribuicoes) if P.modo == "revisar" else True)
    mc.incluir_capex_aace = usar_aace
    if not usar_aace:
        novas["economico.capex_fator"] = _perguntar_distribuicao(P, cfg, "economico.capex_fator",
                                                                 "Fator multiplicador do CAPEX (1 = valor informado)",
                                                                 mc.distribuicoes.get("economico.capex_fator"))
    # Só as variáveis que o motor amostra (VARIAVEIS_ESTOCASTICAS) têm efeito no Monte Carlo; distribuições
    # atribuídas a outros parâmetros no arquivo carregado são descartadas com aviso.
    ignoradas = [c for c in mc.distribuicoes if c not in novas and c not in VARIAVEIS_ESTOCASTICAS]
    if ignoradas:
        P.saida("  ⚠ Distribuições descartadas (o modelo não amostra esses parâmetros): "
                + ", ".join(P.rotulos.get(c, c) for c in ignoradas))
    mc.distribuicoes = {c: d for c, d in novas.items() if c in VARIAVEIS_ESTOCASTICAS}


# numeração dos blocos do wizard (títulos 1 a 16) correspondente a cada item dos menus de correção e de alternativas
FAIXA_BLOCOS = {"Identificação": "bloco 1", "Técnico": "blocos 2 e 3", "Investimento": "bloco 4",
                "Mix e receitas": "bloco 5", "Custos operacionais": "bloco 6", "Pessoas": "bloco 7", "Tributos": "bloco 8",
                "Projeção e TMA": "bloco 9", "Ambiental": "blocos 10 a 12", "Social e governança": "blocos 13 e 14",
                "Pesos e decisão": "bloco 15", "Incerteza (Monte Carlo)": "bloco 16"}
BLOCOS = [
    ("Identificação", bloco_identificacao), ("Técnico", bloco_tecnico), ("Investimento", bloco_investimento),
    ("Mix e receitas", bloco_receitas), ("Custos operacionais", bloco_custos), ("Pessoas", bloco_pessoas),
    ("Tributos", bloco_tributos), ("Projeção e TMA", bloco_projecao), ("Ambiental", bloco_ambiental),
    ("Social e governança", bloco_social), ("Pesos e decisão", bloco_pesos), ("Incerteza (Monte Carlo)", bloco_incerteza),
]


# ---------------------------------------------------------------------------
# Resumo, confirmação e ponto de entrada
# ---------------------------------------------------------------------------

def resumo_entradas(cfg: EVTEASConfig) -> pd.DataFrame:
    t, e, a = cfg.tecnico, cfg.economico, cfg.ambiental
    tec = calcular_tecnico(Contexto(cfg))
    rem = remuneracao_mensal(cfg)
    aq = aliquotas_configuradas(cfg)
    linhas = [
        ("Projeto", cfg.projeto), ("Organização", dict(TIPOS_ORGANIZACAO).get(e.tipo_organizacao, e.tipo_organizacao)),
        ("Sistema / espécie", f"{t.sistema_produtivo} / {t.especie}"),
        ("Lâmina d'água / volume", f"{_fmt(t.area_lamina_m2)} m² / {_fmt(round(t.volume_util_m3, 1))} m³"),
        ("Viveiros (ativos/total)", f"{t.tanques_ativos}/{t.numero_tanques}"),
        ("Peso inicial → final; ciclo", f"{_fmt(t.peso_inicial_g)} g → {_fmt(t.peso_final_g)} g; {t.ciclo_dias} dias; {_fmt(t.ciclos_ano)} ciclos/ano"),
        ("FCR; mortalidade", f"{_fmt(t.fcr)}; {_fmt(t.mortalidade_pct)}%"),
        ("Produção técnica estimada", f"{_fmt(round(float(tec['producao_kg_ano'][0]), 1))} kg/ano"),
        ("Volume de vendas", "mix informado" if e.producao_vendas_kg_mes > 0 else "produção técnica"),
        ("Preço médio", f"R$ {_fmt(round(e.preco_venda_kg, 4))}/kg"),
        ("CAPEX (classe AACE)", f"R$ {_fmt(e.capex_total)} (Classe {e.classe_estimativa_aace})"),
        ("Fomento não reembolsável", f"{_fmt(e.fomento_nao_reembolsavel_pct)}% do CAPEX"),
        ("Ração; alevino; energia", f"R$ {_fmt(e.custo_racao_kg)}/kg; R$ {_fmt(e.custo_alevino_milheiro / 1000)}/un; R$ {_fmt(e.tarifa_energia_kwh)}/kWh"),
        ("Remuneração mensal (CLT / retiradas / outras)", f"R$ {_fmt(rem['clt'])} / R$ {_fmt(rem['retirada'])} / R$ {_fmt(rem['outros'])}"),
        ("Tributos (faturamento / lucro)", f"{_fmt(round((aq['faturamento'] + aq['produtos']) * 100, 4))}% / {_fmt(round(aq['lucro'] * 100, 4))}%"),
        ("TMA; horizonte", f"{_fmt(e.tma_aa_pct)}% a.a.; {e.horizonte_anos} anos"),
        ("Energia; remoção de P", f"{dict(FONTES_ENERGIA).get(a.fonte_energia, a.fonte_energia)}; {_fmt(a.remocao_p_tratamento_pct)}%"),
        ("Pesos", dict(METODOS_PESOS).get(cfg.pesos.metodo, cfg.pesos.metodo)),
        ("Monte Carlo", f"{_fmt(cfg.monte_carlo.iteracoes)} iterações; {len(cfg.monte_carlo.distribuicoes)} variáveis incertas"),
        ("Premissas com origem registrada", str(len(cfg.fontes))),
    ]
    return pd.DataFrame(linhas, columns=["Entrada", "Valor"])


BLOCOS_EM_BRANCO = ("tecnico", "economico", "ambiental", "social", "decisao", "monte_carlo")
NEUTROS = {  # parâmetros que não são perguntas (multiplicadores da incerteza) ou só se aplicam em certos casos
    "economico.capex_fator": 1.0, "economico.custos_fixos_fator": 1.0,
}


def config_em_branco() -> EVTEASConfig:
    """Configuração sem nenhum valor numérico: todo parâmetro precisa vir da entrada."""
    from dataclasses import fields as campos
    cfg = EVTEASConfig(projeto="", autor="")
    for bloco in BLOCOS_EM_BRANCO:
        obj = getattr(cfg, bloco)
        for f in campos(obj):
            v = getattr(obj, f.name)
            if (isinstance(v, (int, float)) and not isinstance(v, bool)) or (v is None and bloco == "tecnico"):
                setattr(obj, f.name, float("nan"))
            elif isinstance(v, bool):
                setattr(obj, f.name, None)
    e = cfg.economico
    e.capex_itens, e.mix_produtos, e.impostos_configurados, e.salarios_clt_por_cargo = {}, [], [], {}
    cfg.ambiental.checklist = {k: None for k in cfg.ambiental.checklist}
    cfg.ambiental.renovacao_pct_dia = cfg.ambiental.energia_kwh_kg = float("nan")
    cfg.governanca.itens = {k: None for k in cfg.governanca.itens}
    cfg.monte_carlo.distribuicoes = {}
    cfg.fontes = {}
    for caminho, valor in NEUTROS.items():
        definir_por_caminho(cfg, caminho, valor)
    return cfg


OPCIONAIS = {  # None é válido: usa a referência do sistema produtivo ou o percentual correspondente
    "tecnico.capacidade_suporte_kg_m3", "ambiental.renovacao_pct_dia", "ambiental.energia_kwh_kg",
    "economico.valor_residual_rs", "economico.manutencao_mes",
}


def entradas_pendentes(cfg: EVTEASConfig) -> List[str]:
    """Lista parâmetros que continuam sem valor (NaN, ou None fora dos opcionais)."""
    from dataclasses import fields as campos
    faltam = []
    for bloco in BLOCOS_EM_BRANCO:
        obj = getattr(cfg, bloco)
        for f in campos(obj):
            v, caminho = getattr(obj, f.name), f"{bloco}.{f.name}"
            if (v is None and caminho not in OPCIONAIS) or (isinstance(v, float) and math.isnan(v)):
                faltam.append(caminho)
    faltam += [f"ambiental.checklist.{k}" for k, v in cfg.ambiental.checklist.items() if v is None]
    faltam += [f"governanca.itens.{k}" for k, v in cfg.governanca.itens.items() if v is None]
    if not cfg.economico.capex_itens:
        faltam.append("economico.capex_itens")
    if not cfg.economico.mix_produtos and vazio(cfg.economico.preco_venda_kg):
        faltam.append("economico.mix_produtos")
    for p in cfg.economico.mix_produtos:
        if any(vazio(p.get(k)) for k in ("preco_kg",)):
            faltam.append("economico.mix_produtos")
            break
    return faltam


# ---------------------------------------------------------------------------
# Leitura de arquivos de entradas (inclusive incompletos)
# ---------------------------------------------------------------------------

SECOES_CONFIG = ("tecnico", "economico", "ambiental", "social", "governanca", "pesos", "decisao", "monte_carlo")
NEUTROS_AUSENTES = {"carencia_automatica", "incluir_capex_aace"}   # opções que recebem padrão neutro quando ausentes do arquivo


def _neutro(nome: str, padrao) -> bool:
    """Indica se o campo, quando ausente do arquivo de entradas, pode receber o valor padrão sem alterar o
    estudo (0, vazio, None, fator 1), em vez de ficar pendente."""
    if nome in NEUTROS_AUSENTES or nome.endswith("_fator"):
        return True
    if isinstance(padrao, bool):
        return False
    return padrao is None or padrao == "" or (isinstance(padrao, (int, float)) and padrao == 0) or \
        (isinstance(padrao, (list, dict)) and not padrao)


def _normalizar_formatos_alternativos(dados: Dict[str, Any], avisos: List[str]) -> Dict[str, Any]:
    """Aceita tributos com 'aliquota' em fração e bases com nomes alternativos (apelidos de BASES_TRIBUTARIAS);
    nomeia os produtos do mix informados sem nome."""
    apelidos = {a: k for k, v in BASES_TRIBUTARIAS.items() for a in v}
    e = dados.get("economico")
    if not isinstance(e, dict):
        return dados
    convertidos = 0
    for i, imp in enumerate(e.get("impostos_configurados") or []):
        if not isinstance(imp, dict):
            raise ValueError("tributo em formato inválido no arquivo")
        if "aliquota_pct" not in imp and "aliquota" in imp:
            imp["aliquota_pct"] = float(imp.pop("aliquota")) * 100
            convertidos += 1
        if imp.get("base", "faturamento") in apelidos:
            imp["base"] = apelidos[imp.get("base", "faturamento")]
        imp.setdefault("nome", f"Tributo {i + 1}")
    if convertidos:
        avisos.append(f"{convertidos} tributo(s) com alíquota em fração convertido(s) para %.")
    for i, prod in enumerate(e.get("mix_produtos") or []):
        if isinstance(prod, dict) and not prod.get("nome"):
            prod["nome"] = f"Produto {i + 1}"
    return dados


def _valor_lido(secao: str, nome: str, valor, atual):
    if nome == "distribuicoes":
        if not isinstance(valor, dict):
            raise ValueError("distribuições do Monte Carlo em formato inválido")
        saida = {}
        for c, d in valor.items():
            extras = set(d) - {"tipo", "minimo", "moda", "maximo"} if isinstance(d, dict) else {"?"}
            if extras:
                raise ValueError(f"distribuição de {c} com campos desconhecidos: {', '.join(sorted(map(str, extras)))}")
            saida[c] = Distribuicao(**d)
        return saida
    if (secao, nome) in (("governanca", "itens"), ("ambiental", "checklist")) and isinstance(valor, dict):
        return {**{k: None for k in atual}, **valor}       # item ausente fica pendente
    return valor


def ler_entradas(dados) -> Tuple[EVTEASConfig, List[str], List[str]]:
    """Monta a configuração a partir do conteúdo de um arquivo de entradas, sem completar o que falta com
    o caso ilustrativo. Devolve (configuração, avisos, entradas pendentes)."""
    import copy
    from dataclasses import fields as campos
    avisos: List[str] = []
    if isinstance(dados, dict) and isinstance(dados.get("config"), dict) and "vv" in dados:
        avisos.append("O arquivo é de resultados do EVTEAS-Py: foram usadas as entradas registradas nele.")
        dados = dados["config"]
    if not isinstance(dados, dict) or not all(isinstance(dados.get(k), dict) for k in ("tecnico", "economico")):
        raise ValueError("não é um arquivo de entradas do EVTEAS-Py (faltam as seções 'tecnico' e 'economico')")
    dados = _normalizar_formatos_alternativos(copy.deepcopy(dados), avisos)
    cfg, padrao = config_em_branco(), EVTEASConfig()
    conhecidas = {f.name for f in campos(EVTEASConfig)}
    desconhecidas = [k for k in dados if k not in conhecidas and k != "versao"]   # rótulo do artefato, aceito sem aviso
    neutros, faltam_texto = [], []
    cfg.projeto, cfg.autor = str(dados.get("projeto") or ""), str(dados.get("autor") or "")
    for secao in SECOES_CONFIG:
        bruto = dados.get(secao)
        if bruto is None:
            if secao == "pesos":
                avisos.append("O arquivo não traz os pesos: adotado o preset do EVTEAS-Py (revise no bloco 'Pesos').")
            continue
        if not isinstance(bruto, dict):
            raise ValueError(f"a seção '{secao}' do arquivo está em formato inválido")
        obj, ref = getattr(cfg, secao), getattr(padrao, secao)
        nomes = {f.name for f in campos(obj)}
        desconhecidas += [f"{secao}.{k}" for k in bruto if k not in nomes]
        for f in campos(obj):
            if f.name in bruto:
                setattr(obj, f.name, _valor_lido(secao, f.name, bruto[f.name], getattr(obj, f.name)))
            elif _neutro(f.name, getattr(ref, f.name)):
                setattr(obj, f.name, copy.deepcopy(getattr(ref, f.name)))
                neutros.append(f"{secao}.{f.name}")
            elif isinstance(getattr(ref, f.name), str):
                setattr(obj, f.name, None)                   # texto ausente: pendente
                faltam_texto.append(f"{secao}.{f.name}")
    fontes = dados.get("fontes") or {}
    if not isinstance(fontes, dict):
        raise ValueError("registro de fontes em formato inválido")
    cfg.fontes = {k: Fonte(**v) for k, v in fontes.items() if isinstance(v, dict)}
    # distribuições: só variáveis amostradas pelo motor, coerentes com o valor determinístico
    mc = cfg.monte_carlo
    for c in list(mc.distribuicoes):
        if c not in VARIAVEIS_ESTOCASTICAS:
            del mc.distribuicoes[c]
            avisos.append(f"Distribuição de {c} descartada: o modelo não amostra esse parâmetro no Monte Carlo.")
            continue
        v, d = obter_por_caminho(cfg, c), mc.distribuicoes[c]
        if not vazio(v) and not distribuicao_coerente(d, float(v)):
            antigo = d.moda if d.tipo != "uniforme" else (d.minimo + d.maximo) / 2
            mc.distribuicoes[c] = ajustar_distribuicao(d, antigo, float(v), c)
            avisos.append(f"Distribuição de {c} reposicionada para o valor determinístico do arquivo ({_fmt(float(v))}).")
    _completar_condicionais(cfg)
    if neutros:
        avisos.append("Campos ausentes no arquivo preenchidos com valor neutro: " + ", ".join(neutros))
    if desconhecidas:
        avisos.append("Campos desconhecidos ignorados: " + ", ".join(desconhecidas))
    return cfg, avisos, entradas_pendentes(cfg)


def _completar_condicionais(cfg: EVTEASConfig) -> None:
    """Valores que só são perguntados em certas situações recebem o valor neutro correspondente."""
    a = cfg.ambiental
    if a.fonte_energia != "mista":
        a.fracao_renovavel_mista = 0.0
    if a.fonte_energia not in ("rede", "mista") and (a.fator_emissao_rede_kgco2_kwh is None or math.isnan(a.fator_emissao_rede_kgco2_kwh)):
        a.fator_emissao_rede_kgco2_kwh = 0.0817     # não utilizado com fonte solar ou biomassa


DEPENDENCIAS = {  # ao corrigir um bloco, os dependentes também são revistos (Enter mantém os valores)
    "Identificação": ["Pessoas", "Tributos", "Social e governança"],   # organização: pessoas, encargos, empregos
    "Técnico": ["Mix e receitas", "Custos operacionais", "Ambiental"],  # produção, carência e referências do sistema
    "Pessoas": ["Tributos", "Social e governança"],   # cargos e retiradas: encargos e empregos diretos
    "Mix e receitas": ["Tributos", "Projeção e TMA"],  # receita de serviços (ISS); a carência limita o horizonte
}


def expandir_blocos(blocos: Sequence[str]) -> List[str]:
    pendentes, vistos = list(blocos), set()
    while pendentes:
        b = pendentes.pop()
        if b not in vistos:
            vistos.add(b)
            pendentes += DEPENDENCIAS.get(b, [])
    return [nome for nome, _ in BLOCOS if nome in vistos]


def wizard_evteas(entrada: Callable[[str], str] = input, base: Optional[EVTEASConfig] = None,
                  saida: Callable[..., None] = print, blocos: Optional[Sequence[str]] = None,
                  anunciar_prints: bool = False) -> EVTEASConfig:
    """Wizard completo. Sem ``base``: modo NOVO (valores obrigatórios). Com ``base``: modo REVISAR."""
    if base is None:
        cfg = config_em_branco()
        P = Perguntador(entrada, saida, "novo")
        saida("=" * 78 + "\nEVTEAS-Py — ENTRADA COMPLETA DE DADOS (novo projeto)\n" + "=" * 78)
        saida("Todos os valores do projeto são obrigatórios. Use vírgula ou ponto decimal; digite 'sair' para interromper.")
    else:
        cfg = base
        P = Perguntador(entrada, saida, "revisar")
        saida("=" * 78 + "\nEVTEAS-Py — REVISÃO DAS ENTRADAS (Enter mantém o valor entre colchetes)\n" + "=" * 78)
    P.anunciar_prints = anunciar_prints
    if blocos is not None:
        blocos = expandir_blocos(blocos)
        if len(blocos) > 1:
            saida("Blocos que dependem da correção também serão revistos: " + ", ".join(blocos))
    antes = {c: float(obter_por_caminho(cfg, c)) for c in cfg.monte_carlo.distribuicoes} if base is not None else {}
    try:
        for nome, func in BLOCOS:
            if blocos is None or nome in blocos:
                func(P, cfg)
    except EntradaCancelada as exc:
        exc.cfg = cfg                       # respostas já dadas, para o rascunho
        raise
    if blocos is not None and "Incerteza (Monte Carlo)" not in blocos:
        for c, v0 in antes.items():
            v1 = float(obter_por_caminho(cfg, c))
            if v1 != v0 and c in cfg.monte_carlo.distribuicoes:
                cfg.monte_carlo.distribuicoes[c] = ajustar_distribuicao(cfg.monte_carlo.distribuicoes[c], v0, v1, c)
                saida(f"  → Distribuição de {c} reposicionada para o novo valor ({_fmt(v0)} → {_fmt(v1)}); "
                      "revise-a no bloco 'Incerteza (Monte Carlo)' se necessário.")
    _completar_condicionais(cfg)
    faltam = entradas_pendentes(cfg)
    if faltam:
        raise RuntimeError("Entradas não preenchidas pelo wizard: " + ", ".join(faltam))
    return cfg


def definir_e_comparar_alternativas(cfg: EVTEASConfig, entrada: Callable[[str], str] = input,
                                    saida: Callable[..., None] = print, n_mc: Optional[int] = None):
    """Compara o projeto informado com alternativas definidas pelo usuário (TOPSIS)."""
    import copy
    P = Perguntador(entrada, saida, "novo")
    quer, _ = P.sim_nao("Deseja comparar o projeto com alternativas de projeto (TOPSIS)?")
    if not quer:
        return None
    alternativas = {cfg.projeto or "Projeto informado": copy.deepcopy(cfg)}
    n, _ = P.numero("Quantas alternativas adicionais", None, 1, 5, inteiro=True)
    opcoes = ([(nome, f"{nome} ({FAIXA_BLOCOS[nome]})") for nome, _ in BLOCOS if nome != "Identificação"]
              + [("fim", "Concluir esta alternativa")])
    for i in range(n):
        nome = P.texto(f"\nNome da alternativa {i + 1}")
        while nome in alternativas:
            saida("  ⚠ Já existe uma alternativa (ou o projeto) com esse nome; informe outro.")
            nome = P.texto(f"Nome da alternativa {i + 1}")
        alt = copy.deepcopy(cfg)
        alt.projeto = nome
        while True:
            bloco, _ = P.opcao(f"Bloco a alterar em '{nome}':", opcoes)
            if bloco == "fim":
                break
            alt = wizard_evteas(entrada, alt, saida, blocos=[bloco])
        alternativas[nome] = alt
    criterios = ["VPL (R$)", "P(VPL>0)", "OEE", "PH cinza (m³/t)", "Intensidade carbono (kgCO2e/kg)", "LSO", "RVL (%)"]
    pesos = None
    informar, _ = P.sim_nao("Informar os pesos dos critérios do TOPSIS (n = pesos padrão)?")
    if informar:
        pesos = {c: P.numero(f"  Peso do critério {c}", None, 0)[0] for c in criterios}
        if sum(pesos.values()) <= 0:
            saida("  ⚠ Soma dos pesos nula; usando pesos padrão.")
            pesos = None
    return comparar_alternativas(alternativas, pesos, n_mc=n_mc or min(cfg.monte_carlo.iteracoes, 5000))


def _baixar_no_colab(caminho: str):
    try:
        from google.colab import files  # type: ignore
        files.download(caminho)
    except Exception:
        pass


def baixar_no_colab(caminho: str):
    """Baixa um arquivo para o computador do usuário quando o notebook roda no Google Colab."""
    _baixar_no_colab(caminho)


def _no_colab() -> bool:
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except ImportError:
        return False


def _carregar_arquivo(entrada: Callable[[str], str], saida) -> Tuple[EVTEASConfig, List[str]]:
    """Lê o arquivo de entradas, com nova tentativa em caso de erro. Devolve (configuração, pendências)."""
    colab = _no_colab()
    while True:
        prompt = ("Enter para enviar o arquivo JSON do computador, ou caminho de um arquivo já no ambiente: " if colab
                  else "Caminho do arquivo JSON de entradas: ")
        caminho = (entrada(prompt) or "").strip().strip('"').strip("'")
        if caminho.lower() in ("sair", "cancelar"):
            raise EntradaCancelada("entrada interrompida pelo usuário")
        try:
            if colab and not caminho:
                from google.colab import files  # type: ignore
                saida("Selecione o arquivo JSON de entradas salvo anteriormente.")
                enviado = files.upload()
                if not enviado:
                    raise ValueError("nenhum arquivo selecionado")
                texto = next(iter(enviado.values())).decode("utf-8-sig")
            elif not caminho:
                raise ValueError("informe o caminho do arquivo")
            else:
                texto = Path(caminho).read_text(encoding="utf-8-sig")
            cfg, avisos, pendentes = ler_entradas(json.loads(texto))
        except EntradaCancelada:
            raise
        except (OSError, ValueError, TypeError, KeyError, AttributeError, UnicodeDecodeError) as exc:
            saida(f"  ⚠ Não foi possível usar o arquivo ({exc}). Tente novamente ou digite 'sair'.")
            continue
        for a in avisos:
            saida(f"  ⚠ {a}")
        return cfg, pendentes


def exigir_entradas(cfg: Optional[EVTEASConfig]) -> EVTEASConfig:
    """Usada no notebook antes da análise: sem entradas concluídas, interrompe com mensagem curta."""
    if cfg is None:
        raise EntradaCancelada("A entrada de dados não foi concluída: execute a célula 'Entrada de dados do projeto'.")
    return cfg


def iniciar_entradas(entrada: Callable[[str], str] = input, saida: Callable[..., None] = print,
                     salvar: bool = True, pasta_saida: str = ".") -> EVTEASConfig:
    """Ponto de partida do estudo: nenhum resultado é calculado antes das entradas."""
    saida("=" * 78 + "\nEVTEAS-Py — INÍCIO DO ESTUDO\n" + "=" * 78)
    saida(titulo_print("P01"))
    P = Perguntador(entrada, saida, "novo")
    modo, _ = P.opcao("Como deseja informar os dados do projeto?", [
        ("novo", "Novo projeto — digitar todas as entradas"),
        ("arquivo", "Carregar arquivo de entradas (JSON) e revisar"),
        ("exemplo", "Exemplo ilustrativo incorporado ao pacote (apenas demonstração)"),
    ])
    try:
        if modo == "novo":
            cfg = wizard_evteas(entrada, None, saida, anunciar_prints=True)
        elif modo == "arquivo":
            cfg, pendentes = _carregar_arquivo(entrada, saida)
            if pendentes:
                saida(f"⚠ O arquivo não traz {len(pendentes)} entrada(s), que serão pedidas na revisão a seguir: "
                      + ", ".join(pendentes[:12]) + (" …" if len(pendentes) > 12 else ""))
                revisar = True
            else:
                revisar, _ = P.sim_nao("Revisar as entradas carregadas bloco a bloco?", None)
            if revisar:
                cfg = wizard_evteas(entrada, cfg, saida, anunciar_prints=True)
        else:
            cfg = criar_config_caso_base()
            cfg.projeto = f"EXEMPLO ILUSTRATIVO — {cfg.projeto}"
            saida("⚠ Exemplo ilustrativo carregado: os resultados NÃO representam o seu projeto.")
    except EntradaCancelada as exc:
        parcial = getattr(exc, "cfg", None)
        if salvar and parcial is not None:
            caminho = str(Path(pasta_saida) / nome_arquivo(parcial).replace(".json", "_rascunho.json"))
            salvar_config(parcial, caminho)
            _baixar_no_colab(caminho)
            raise EntradaCancelada(f"Entrada interrompida. As respostas dadas foram salvas em {caminho}; "
                                   "use a opção 2 para continuar de onde parou.") from None
        raise
    while True:
        saida("\n" + titulo_print("P02") + "\nResumo das entradas:")
        saida(resumo_entradas(cfg).to_string(index=False))
        try:
            acao, _ = P.opcao("\nO que deseja fazer?", [("executar", "Confirmar e executar a análise"),
                                                         ("editar", "Corrigir um bloco de entradas"),
                                                         ("sair", "Salvar as entradas e encerrar sem executar")])
        except EntradaCancelada:
            acao = "sair"                    # 'sair' no menu final salva antes de encerrar
        if acao == "editar":
            nomes = [(n, f"{n} ({FAIXA_BLOCOS[n]})") for n, _ in BLOCOS]
            try:
                bloco, _ = P.opcao("Bloco a corrigir:", nomes)
                cfg = wizard_evteas(entrada, copy.deepcopy(cfg), saida, blocos=[bloco], anunciar_prints=True)
            except EntradaCancelada:
                saida("Correção cancelada: as entradas anteriores foram mantidas.")
            continue
        break
    if salvar:
        caminho = str(Path(pasta_saida) / nome_arquivo(cfg))
        salvar_config(cfg, caminho)
        saida(f"Entradas salvas em: {caminho} (use a opção 2 para reabrir e revisar).")
        _baixar_no_colab(caminho)
    if acao == "sair":
        raise EntradaCancelada("Entradas salvas; a análise não foi executada.")
    return cfg


# ---------------------------------------------------------------------------
# Análise de preços a partir de planilha
# ---------------------------------------------------------------------------

def analisar_precos(df: pd.DataFrame, col_preco: str, col_qtd: Optional[str] = None,
                    custo_unitario: float = 0.0, z_corte: float = 3.0) -> Dict[str, Any]:
    """Histórico de preços (estatísticas e estratégias) ou, com quantidades,
    curva de demanda linear e preço que maximiza a margem de contribuição."""
    d = df[[c for c in (col_preco, col_qtd) if c]].apply(pd.to_numeric, errors="coerce").dropna()
    d = d[d[col_preco] > 0]
    if len(d) > 10:
        z = ((d - d.mean()) / d.std(ddof=0)).abs()
        d = d[(z <= z_corte).all(axis=1)]
    if d.empty:
        raise ValueError("Sem dados válidos de preço")
    p = d[col_preco]
    out: Dict[str, Any] = {"n": int(len(d)), "estatisticas": p.describe().to_dict(),
                           "sugestoes": {"Econômico (P25)": float(p.quantile(.25)), "Mediana": float(p.median()),
                                         "Premium (P75)": float(p.quantile(.75))}}
    if col_qtd and len(d) >= 5:
        b, a = np.polyfit(p, d[col_qtd], 1)
        r2 = float(np.corrcoef(p, d[col_qtd])[0, 1] ** 2)
        out["demanda"] = {"intercepto": float(a), "inclinacao": float(b), "r2": r2}
        if b < 0:
            out["preco_otimo"] = float((custo_unitario - a / b) / 2)   # max (P − c)(a + bP)
    return out

## 11. Registro do ciclo da Design Science Research: etapas, requisitos, comparação e aprendizagens

In [ ]:
"""Registro do ciclo da Design Science Research (DSR) na construção do EVTEAS-Py.

Operacionaliza, como dados verificáveis, as 12 etapas de Dresch, Lacerda e
Antunes Jr. (2015) descritas na Seção 3.3.1 da dissertação:

* ETAPAS_DSR: o que cada etapa produziu nesta pesquisa e onde está a evidência;
* COMPARACAO_ARTEFATOS (Etapa 4): propriedades do EVTEAS-Py e dos artefatos
  preexistentes da Seção 2.8.1, no formato de tabela comparativa de Wazlawick (2009);
* REQUISITOS (Etapa 5) e matriz_rastreabilidade() (Etapa 8): requisito →
  funções que o implementam → testes que o verificam, conferidos no próprio código;
* APRENDIZAGENS (Etapa 9): dificuldades, erros superados e decisões de projeto.

A matriz é recalculada a partir do código e dos arquivos de teste, de modo que um
requisito sem função ou sem teste aparece como pendente, e não como atendido.
"""

import ast
import importlib
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import pandas as pd


# ---------------------------------------------------------------------------
# As 12 etapas
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class Etapa:
    numero: int
    nome: str
    fase: str                      # fases macro da Seção 3.2
    produto: str                   # o que a etapa produziu nesta pesquisa
    evidencias: Tuple[str, ...]    # onde verificar (seção, módulo, arquivo)
    situacao: str                  # concluída | primeira iteração concluída | em andamento | prevista


FASE1, FASE2, FASE3 = "Fase 1 – Imersão bibliográfica", "Fase 2 – Ciclo DSR", "Fase 3 – V&V"

ETAPAS_DSR: Tuple[Etapa, ...] = (
    Etapa(1, "Identificação do problema", FASE1,
          "Produtores e investidores da piscicultura sem instrumento sistemático para avaliar de forma integrada "
          "a viabilidade técnica, econômica, ambiental e social de seus empreendimentos.",
          ("Seção 1.3",), "concluída"),
    Etapa(2, "Compreensão do problema", FASE1,
          "Causas: variáveis dissimilares (custos, efluentes, aceitação comunitária) sem modelo que pondere dados "
          "exatos e critérios qualitativos; estudos em planilhas determinísticas; requisitos de uso extraídos dos "
          "notebooks de estudo de viabilidade econômica utilizados pelo autor (entradas, mix, tributos, cooperados).",
          ("Seções 1.2, 2.8 a 2.10",), "concluída"),
    Etapa(3, "Revisão sistemática da literatura", FASE1,
          "Corpus selecionado pelo protocolo PRISMA e referencial teórico dos três eixos temáticos.",
          ("Seções 2.1 a 2.7",), "concluída"),
    Etapa(4, "Identificação de artefatos e configuração da classe de problemas", FASE2,
          "Classe: modelagem de suporte à decisão multicritério sob incerteza em agroindústrias. Comparação de "
          "propriedades com quatro artefatos preexistentes (Wazlawick, 2009).",
          ("Seção 2.8.1", "dsr.COMPARACAO_ARTEFATOS"), "concluída"),
    Etapa(5, "Proposição do artefato", FASE2,
          "EVTEAS-Py como instanciação (framework operante) conjugada a um modelo (representação matemática da "
          "viabilidade), com requisitos de alto nível rastreáveis.",
          ("dsr.REQUISITOS",), "concluída"),
    Etapa(6, "Design do artefato", FASE2,
          "Configuração única (EVTEASConfig) com registro de fontes; motor vetorizado comum às camadas "
          "determinística e estocástica; fluxo entrada → processamento → saída → verificação; interface de "
          "entrada em blocos.",
          ("config.EVTEASConfig", "modelo.Contexto", "pipeline.motor"), "primeira iteração concluída"),
    Etapa(7, "Desenvolvimento do artefato", FASE2,
          "Pacote evteas_py com doze módulos, notebook autocontido para o Google Colab e entrada de dados "
          "interativa ou por arquivo.",
          ("Apêndice A", "pipeline.executar_evteas", "interface.iniciar_entradas"), "primeira iteração concluída"),
    Etapa(8, "Avaliação do artefato", FASE3,
          "Verificação: invariantes a cada execução, suíte de testes automatizados e revisão independente da "
          "entrada de dados. Validação com os dois especialistas a realizar.",
          ("vv.validar_invariantes", "Apêndice B", "dsr.matriz_rastreabilidade"),
          "verificação concluída; validação prevista"),
    Etapa(9, "Explicitação das aprendizagens", FASE3,
          "Registro das dificuldades, dos erros lógicos superados e das decisões de projeto que deles resultaram.",
          ("dsr.APRENDIZAGENS",), "em andamento"),
    Etapa(10, "Conclusão", FASE3,
          "Síntese dos resultados, do atendimento aos objetivos e das limitações do artefato.",
          ("Capítulo 5",), "primeira iteração concluída"),
    Etapa(11, "Generalização para a classe de problemas", FASE3,
          "Princípios de projeto transferíveis a outras cadeias agroindustriais, com recalibração dos parâmetros.",
          ("Seção 5.5",), "primeira iteração concluída"),
    Etapa(12, "Comunicação dos resultados", FASE3,
          "Dissertação, código e testes registrados no repositório do projeto, notebook executável, arquivo de entradas, relatórios "
          "exportados e apêndices com o código-fonte; artigos previstos.",
          ("Apêndices A e B", "relatorios.exportar"), "em andamento"),
)


def etapas_dsr() -> pd.DataFrame:
    return pd.DataFrame([{"Etapa": e.numero, "Nome": e.nome, "Fase": e.fase, "Produto": e.produto,
                          "Evidências": "; ".join(e.evidencias), "Situação": e.situacao} for e in ETAPAS_DSR])


# ---------------------------------------------------------------------------
# Etapa 4: comparação de propriedades com artefatos preexistentes (Wazlawick, 2009)
# ---------------------------------------------------------------------------

SIM, NAO, NI = "sim", "não", "não identificado"

ARTEFATOS = ("Costa Magalhães (2022)", "pyteea (Maier; Weyand, 2025)", "AQUA+ (Ramos et al., 2024)",
             "AquaSpace (Gimpel et al., 2018)", "EVTEAS-Py")

# Somente o que a Seção 2.8.1 descreve sobre cada artefato; o restante fica como "não identificado".
COMPARACAO_ARTEFATOS: Dict[str, Tuple[str, ...]] = {
    "Implementação em Python":                         (SIM, SIM, NI, NI, SIM),
    "Aplicação à aquicultura":                         (NAO, NAO, SIM, SIM, SIM),
    "Avaliação no nível do empreendimento":            (SIM, NI, SIM, NAO, SIM),
    "Viabilidade técnica do processo produtivo":       (NI, SIM, NI, NI, SIM),
    "Viabilidade econômico-financeira (VPL, TIR)":     (SIM, SIM, SIM, NI, SIM),
    "Indicadores ambientais":                          (NI, SIM, NAO, SIM, SIM),
    "Indicadores sociais":                             (NI, NI, NAO, SIM, SIM),
    "Governança, ESG e Lean Green":                    (NI, NI, NAO, NI, SIM),
    "Simulação estocástica (Monte Carlo)":             (NI, NI, NI, NI, SIM),
    "Ponderação multicritério com regra de decisão":   (NI, NI, NI, NI, SIM),
}

EVIDENCIA_EVTEAS = {
    "Implementação em Python": "pacote evteas_py",
    "Aplicação à aquicultura": "piscicultura continental (casos.criar_config_caso_base)",
    "Avaliação no nível do empreendimento": "configuração por projeto (config.EVTEASConfig)",
    "Viabilidade técnica do processo produtivo": "modelo.calcular_tecnico",
    "Viabilidade econômico-financeira (VPL, TIR)": "modelo.calcular_economico; financeiro.tir",
    "Indicadores ambientais": "modelo.calcular_ambiental",
    "Indicadores sociais": "modelo.calcular_social",
    "Governança, ESG e Lean Green": "modelo.calcular_governanca; modelo.calcular_lean_green",
    "Simulação estocástica (Monte Carlo)": "incerteza.monte_carlo",
    "Ponderação multicritério com regra de decisão": "ponderacao.resolver_pesos; ponderacao.classificar",
}


def tabela_comparativa() -> pd.DataFrame:
    linhas = [{"Propriedade": p, **dict(zip(ARTEFATOS, v))} for p, v in COMPARACAO_ARTEFATOS.items()]
    return pd.DataFrame(linhas)


def combinacao_inedita(artefato: str = "EVTEAS-Py") -> Dict[str, object]:
    """Verifica o critério de Wazlawick (2009): propriedades exclusivas ou combinação
    de propriedades que nenhum outro artefato comparado reúne."""
    k = ARTEFATOS.index(artefato)
    proprias = [p for p, v in COMPARACAO_ARTEFATOS.items() if v[k] == SIM]
    exclusivas = [p for p in proprias
                  if all(v != SIM for j, v in enumerate(COMPARACAO_ARTEFATOS[p]) if j != k)]
    cobertura = {a: sum(COMPARACAO_ARTEFATOS[p][j] == SIM for p in proprias)
                 for j, a in enumerate(ARTEFATOS) if j != k}
    return {"propriedades": proprias, "exclusivas": exclusivas, "cobertura_dos_demais": cobertura,
            "inedita": all(c < len(proprias) for c in cobertura.values())}


# ---------------------------------------------------------------------------
# Etapa 5: requisitos e Etapa 8: rastreabilidade requisito → código → teste
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class Requisito:
    codigo: str
    descricao: str
    origem: str
    funcoes: Tuple[str, ...]       # "modulo.objeto" dentro de evteas_py
    testes: Tuple[str, ...]        # "arquivo.py::test_nome" dentro de tests/


REQUISITOS: Tuple[Requisito, ...] = (
    Requisito("R01", "Ser processado em Python, com núcleo reutilizável e notebook executável no Google Colab",
              "Etapa 5", ("pipeline.executar_evteas",),
              ("test_entradas.py::test_notebook_autocontido_sem_colisao_de_nomes",)),
    Requisito("R02", "Começar pela entrada de dados, sem resultado antes das premissas e sem valor herdado em silêncio",
              "Seção 2.8.3 (PMBOK)", ("interface.iniciar_entradas", "interface.config_em_branco",
                                      "interface.entradas_pendentes"),
              ("test_entradas.py::test_config_em_branco_nao_tem_valores",
               "test_entradas.py::test_modo_novo_nao_aceita_enter_sem_referencia",
               "test_entradas.py::test_novo_projeto_reproduz_caso_base",
               "test_entradas.py::test_salvar_e_encerrar_nao_executa",
               "test_entradas_robustez.py::test_sair_salva_rascunho_que_pode_ser_retomado")),
    Requisito("R03", "Registrar a origem de cada premissa e permitir revisar um estudo salvo",
              "Seção 3.6", ("config.Fonte", "pipeline.tabela_premissas", "interface.salvar_config",
                            "interface.carregar_config"),
              ("test_entradas.py::test_fontes_registradas", "test_evteas.py::test_config_json_ida_e_volta",
               "test_entradas.py::test_modo_revisar_enter_mantem_valor",
               "test_entradas_coerencia.py::test_revisar_preserva_mao_de_obra_mista",
               "test_entradas_robustez.py::test_campos_ausentes_nao_viram_caso_ilustrativo")),
    Requisito("R04", "Dimensionar a produção e calcular os KPIs técnicos (OEE, FCR, produtividade)",
              "Quadro 5", ("modelo.calcular_tecnico", "modelo.curva_crescimento"),
              ("test_evteas.py::test_oee_produto_dos_fatores", "test_evteas.py::test_racao_igual_fcr_vezes_ganho",
               "test_evteas.py::test_restricao_dupla_de_biomassa",
               "test_evteas.py::test_curva_crescimento_reproduz_peso_final")),
    Requisito("R05", "Projetar DRE e fluxo de caixa e calcular VPL, TIR, payback e ROI",
              "Seção 2.8.6", ("modelo.calcular_economico", "financeiro.vpl", "financeiro.tir", "financeiro.payback"),
              ("test_evteas.py::test_vpl_igual_numpy_financial", "test_evteas.py::test_tir_igual_numpy_financial",
               "test_evteas.py::test_tir_sem_solucao_nao_vira_limite_numerico",
               "test_evteas.py::test_payback_simples_e_descontado", "test_evteas.py::test_vpl_recalculado_do_fluxo",
               "test_evteas.py::test_impostos_configurados_por_base")),
    Requisito("R06", "Calcular água, pegadas hídricas, emissões, ecoeficiência e conformidade ambiental",
              "Quadro 5", ("modelo.calcular_ambiental", "modelo.score_conformidade"),
              ("test_evteas.py::test_pegada_cinza_calculo_manual",
               "test_evteas.py::test_ecoeficiencia_indefinida_quando_va_negativo",
               "test_evteas.py::test_item_eliminatorio_gera_veto",
               "test_entradas_coerencia.py::test_ambiental_acompanha_volume_vendido")),
    Requisito("R07", "Calcular RVL, LSO, risco e Radar Social, governança e alinhamento aos ODS",
              "Quadro 5", ("modelo.calcular_social", "modelo.calcular_governanca", "modelo.alinhamento_ods"),
              ("test_social_lean_green.py::test_rvl_segue_formula_do_quadro_5",
               "test_social_lean_green.py::test_lso_e_risco_social",
               "test_social_lean_green.py::test_governanca_e_ods",
               "test_entradas_coerencia.py::test_renda_e_rvl_em_precos_do_ano_de_regime")),
    Requisito("R08", "Quantificar em reais os desperdícios Lean-Green",
              "Seção 2.9.1", ("modelo.calcular_lean_green",),
              ("test_social_lean_green.py::test_lean_green_zerado_na_referencia",
               "test_social_lean_green.py::test_lean_green_racao_excedente")),
    Requisito("R09", "Traduzir julgamentos qualitativos em pesos (Likert, AHP ou preset)",
              "Seção 3.4; Quadro 7", ("ponderacao.pesos_likert", "ponderacao.pesos_ahp", "ponderacao.resolver_pesos"),
              ("test_evteas.py::test_likert_normaliza_e_triangula", "test_evteas.py::test_ahp_matriz_consistente",
               "test_evteas.py::test_ahp_rejeita_matriz_nao_reciproca",
               "test_evteas.py::test_pipeline_com_pesos_likert_e_ahp")),
    Requisito("R10", "Integrar as dimensões em índice normalizado com decisão não compensatória",
              "Seção 2.8.6", ("ponderacao.normalizar", "ponderacao.calcular_indice", "ponderacao.classificar"),
              ("test_evteas.py::test_normalizacao_limites", "test_evteas.py::test_indice_entre_zero_e_um",
               "test_evteas.py::test_classificacao_regras")),
    Requisito("R11", "Traduzir a incerteza das entradas em risco financeiro (Monte Carlo no mesmo motor)",
              "Etapa 5; Seção 3.6", ("incerteza.monte_carlo", "pipeline.motor"),
              ("test_evteas.py::test_monte_carlo_reprodutivel",
               "test_evteas.py::test_monte_carlo_degenerado_igual_deterministico",
               "test_evteas.py::test_distribuicoes_invalidas",
               "test_entradas.py::test_monte_carlo_da_entrada_igual_ao_caso_base",
               "test_entradas_robustez.py::test_parametro_nao_amostrado_nao_e_aceito_no_monte_carlo")),
    Requisito("R12", "Identificar variáveis críticas (sensibilidade, valores críticos e cenários)",
              "Seção 3.6", ("incerteza.sensibilidade", "incerteza.valores_criticos", "incerteza.cenarios"),
              ("test_evteas.py::test_sensibilidade_e_valores_criticos", "test_evteas.py::test_cenarios_ordenados")),
    Requisito("R13", "Comparar alternativas de projeto e explicitar trade-offs (TOPSIS)",
              "Objetivo específico 2", ("ponderacao.topsis", "pipeline.comparar_alternativas",
                                         "interface.definir_e_comparar_alternativas"),
              ("test_evteas.py::test_topsis_alternativa_dominante_vence", "test_evteas.py::test_comparar_alternativas")),
    Requisito("R14", "Verificar o próprio resultado a cada execução e manter testes de regressão",
              "Seção 3.3.2 (Boehm, 1984)", ("vv.validar_invariantes", "vv.teste_regressao_deterministica"),
              ("test_evteas.py::test_invariantes_aprovados", "test_evteas.py::test_regressao_embutida",
               "test_evteas.py::test_projeto_deficitario_e_diagnosticado")),
    Requisito("R15", "Exportar resultados auditáveis (Excel, JSON, Markdown e figuras)",
              "Etapa 12", ("relatorios.exportar", "relatorios.gerar_graficos", "relatorios.resumo_executivo"),
              ("test_evteas.py::test_exportacao_e_graficos",)),
    Requisito("R16", "Entregar, a cada execução do usuário, os valores que preenchem o relato dos resultados",
              "Etapa 12", ("campos_texto.valores_para_o_texto", "campos_texto.exportar_valores_para_o_texto"),
              ("test_campos_texto.py::test_todos_os_campos_tem_valor_no_caso_base",
               "test_campos_texto.py::test_valores_acompanham_as_entradas_do_usuario",
               "test_campos_texto.py::test_lacunas_do_texto_existem_no_registro")),
)


def _testes_existentes(pasta: Path) -> Dict[str, set]:
    nomes = {}
    for arq in sorted(pasta.glob("test_*.py")):
        arvore = ast.parse(arq.read_text(encoding="utf-8"))
        nomes[arq.name] = {n.name for n in arvore.body if isinstance(n, ast.FunctionDef) and n.name.startswith("test_")}
    return nomes


def _objeto_existe(caminho: str) -> bool:
    modulo, _, nome = caminho.partition(".")
    try:
        return hasattr(importlib.import_module(f"evteas_py.{modulo}"), nome)
    except ImportError:                          # notebook autocontido: um único espaço de nomes
        return nome in globals()


def pasta_testes_padrao() -> Optional[Path]:
    arquivo = globals().get("__file__")          # ausente no notebook autocontido
    if not arquivo:
        return None
    p = Path(arquivo).resolve().parents[1] / "tests"
    return p if p.is_dir() else None


def matriz_rastreabilidade(pasta_testes: Optional[Path] = None,
                           requisitos: Sequence[Requisito] = REQUISITOS) -> pd.DataFrame:
    """Confere, no código e nos arquivos de teste, cada vínculo requisito → função → teste.

    Situação: "implementado e verificado" (todas as funções e testes existem),
    "implementado, sem teste" ou "pendente" (alguma função não existe). Sem a pasta
    de testes (por exemplo, no notebook autocontido), os testes não são conferidos.
    """
    pasta = Path(pasta_testes) if pasta_testes else pasta_testes_padrao()
    existentes = _testes_existentes(pasta) if pasta else None
    linhas = []
    for r in requisitos:
        faltam_f = [f for f in r.funcoes if not _objeto_existe(f)]
        if existentes is None:
            faltam_t, conferido = [], False
        else:
            faltam_t = [t for t in r.testes if t.split("::")[1] not in existentes.get(t.split("::")[0], set())]
            conferido = True
        if faltam_f:
            situacao = "pendente"
        elif not r.testes or len(faltam_t) == len(r.testes):
            situacao = "implementado, sem teste"
        elif faltam_t:
            situacao = "implementado, testes incompletos"
        else:
            situacao = "implementado e verificado" if conferido else "implementado (testes não conferidos)"
        linhas.append({"Requisito": r.codigo, "Descrição": r.descricao, "Origem": r.origem,
                       "Funções": ", ".join(r.funcoes), "Testes": len(r.testes) - len(faltam_t),
                       "Faltando": "; ".join(faltam_f + faltam_t), "Situação": situacao})
    return pd.DataFrame(linhas)


# ---------------------------------------------------------------------------
# Etapa 9: aprendizagens explicitadas durante a construção
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class Aprendizagem:
    numero: int
    dificuldade: str               # o que foi observado
    causa: str                     # por que ocorria
    decisao: str                   # o que mudou no artefato
    evidencia: str                 # teste ou função que impede a regressão


APRENDIZAGENS: Tuple[Aprendizagem, ...] = (
    Aprendizagem(1, "Na implementação inicial, a média do VPL simulado ficava acima do VPL determinístico, mesmo com riscos "
                    "desfavoráveis.",
                 "Monte Carlo usava um modelo anual simplificado, diferente do modelo mensal determinístico.",
                 "Motor único vetorizado: o mesmo código produz o resultado pontual (n = 1) e a simulação (n = número de iterações).",
                 "test_evteas.py::test_monte_carlo_degenerado_igual_deterministico"),
    Aprendizagem(2, "TIR reportada como −0,95 em fluxos sem raiz, confundida com um resultado.",
                 "A bisseção devolvia o limite do intervalo quando não havia troca de sinal.",
                 "Varredura de malha seguida de bisseção, sempre com diagnóstico (convencional, múltiplas, múltiplas possíveis ou sem solução).",
                 "test_evteas.py::test_tir_sem_solucao_nao_vira_limite_numerico"),
    Aprendizagem(3, "Ecoeficiência igual a zero em projetos com prejuízo.",
                 "O numerador era zerado quando o lucro era negativo.",
                 "Ecoeficiência como valor adicionado por unidade de impacto (WBCSD, 2000), declarada indefinida quando o valor adicionado não é positivo.",
                 "test_evteas.py::test_ecoeficiencia_indefinida_quando_va_negativo"),
    Aprendizagem(4, "RVL e OEE divergiam das fórmulas do Quadro 5.",
                 "A RVL era dividida pela receita e a qualidade do OEE era um parâmetro independente.",
                 "RVL sobre o custo operacional; qualidade do OEE igual à sobrevivência.",
                 "test_social_lean_green.py::test_rvl_segue_formula_do_quadro_5"),
    Aprendizagem(5, "Rotinas de AHP e TOPSIS existiam, mas não eram chamadas pelo pipeline.",
                 "Funções desenvolvidas sem requisito e sem teste que as exigisse.",
                 "Rastreabilidade requisito → função → teste; AHP e TOPSIS integrados ao fluxo.",
                 "test_evteas.py::test_pipeline_com_pesos_likert_e_ahp"),
    Aprendizagem(6, "O notebook produzia resultados ao ser executado, antes de o usuário informar o projeto.",
                 "Premissas do caso ilustrativo funcionavam como valores padrão silenciosos.",
                 "Estudo começa pela entrada de dados; modo novo parte de configuração em branco e recusa pendências.",
                 "test_entradas.py::test_modo_novo_nao_aceita_enter_sem_referencia"),
    Aprendizagem(7, "Valores informados em reais (manutenção, valor residual) mudavam quando o CAPEX era revisto.",
                 "Eram convertidos e guardados como percentual do CAPEX.",
                 "Valores em reais guardados em reais; percentual apenas quando informado como percentual.",
                 "test_entradas_coerencia.py::test_valores_em_reais_nao_mudam_com_o_capex"),
    Aprendizagem(8, "Corrigir um bloco de entradas deixava distribuições e blocos dependentes incoerentes.",
                 "Cada bloco era tratado como independente, e as distribuições não acompanhavam o valor determinístico.",
                 "Dependências declaradas entre blocos e reposicionamento das distribuições na revisão.",
                 "test_entradas_coerencia.py::test_corrigir_preco_reposiciona_distribuicao"),
    Aprendizagem(9, "A rastreabilidade de requisitos revelou funções sociais, de governança e Lean-Green sem teste.",
                 "Os testes tinham sido escritos a partir dos cálculos financeiros, e não dos requisitos.",
                 "Testes dedicados para RVL, LSO, governança, ODS e Lean-Green.",
                 "test_social_lean_green.py::test_lso_e_risco_social"),
    Aprendizagem(10, "Valores digitados no padrão brasileiro (40.000) eram lidos como 40, sem aviso.",
                 "O conversor tratava todo ponto como separador decimal.",
                 "Ponto seguido de grupos de três dígitos é lido como milhar, e o valor interpretado é mostrado.",
                 "test_entradas_robustez.py::test_ponto_de_milhar_e_formato_sem_notacao_cientifica"),
    Aprendizagem(11, "Um arquivo de entradas incompleto ou de outro tipo era completado com o caso ilustrativo.",
                 "Os valores padrão das estruturas de configuração eram os do caso ilustrativo.",
                 "Leitura a partir de configuração em branco: o que falta vira pendência e obriga a revisão.",
                 "test_entradas_robustez.py::test_campos_ausentes_nao_viram_caso_ilustrativo"),
    Aprendizagem(12, "Incerteza informada em parâmetros que o motor não amostra não chegava ao resultado.",
                 "O wizard oferecia qualquer parâmetro numérico, mas só nove são lidos de forma vetorizada.",
                 "Lista única de variáveis estocásticas, verificada pelo próprio motor.",
                 "test_entradas_robustez.py::test_parametro_nao_amostrado_nao_e_aceito_no_monte_carlo"),
)


def aprendizagens() -> pd.DataFrame:
    return pd.DataFrame([{"Nº": a.numero, "Dificuldade observada": a.dificuldade, "Causa": a.causa,
                          "Decisão de projeto": a.decisao, "Evidência": a.evidencia} for a in APRENDIZAGENS])


def relatorio_dsr(pasta_testes: Optional[Path] = None) -> Dict[str, pd.DataFrame]:
    return {"DSR_Etapas": etapas_dsr(), "DSR_Comparacao": tabela_comparativa(),
            "DSR_Rastreabilidade": matriz_rastreabilidade(pasta_testes), "DSR_Aprendizagens": aprendizagens()}

## 12. Valores para o texto: lacunas E, R, K e N do relato dos resultados

In [ ]:
"""Campos do texto da dissertação preenchidos com a execução do usuário.

O relato dos resultados (Capítulo 4) traz lacunas identificadas por código, como
``[E14 – FCR]`` ou ``[S23 – VPL do projeto]``. Este módulo é o registro único desses
códigos: o gerador do texto lê daqui a descrição de cada lacuna, e o notebook usa a
mesma lista para mostrar, ao final da análise, o valor de cada código na execução
feita pelo usuário (``valores_para_o_texto``). Assim, texto e resultados ficam
atrelados às entradas informadas, e não a uma simulação prévia.

Prefixos: E = entrada informada no wizard; S = saída (resultado) da análise; K = valor bruto
de um KPI; N = valor normalizado de um KPI. (O prefixo R fica reservado aos requisitos do módulo dsr.)
"""

import math
from dataclasses import dataclass
from typing import Any, Callable, Dict, List, Optional

import numpy as np
import pandas as pd



# ---------------------------------------------------------------------------
# Formatação pt-BR
# ---------------------------------------------------------------------------

def _f(v) -> float:
    if isinstance(v, (list, tuple, np.ndarray)):
        v = np.asarray(v).ravel()[0]
    return float(v)


def num(v, casas: int = 2) -> str:
    try:
        x = _f(v)
    except (TypeError, ValueError):
        return str(v)
    if not math.isfinite(x):
        return "não definido"
    s = f"{x:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return s.replace("-", "−")


def reais(v, casas: int = 0) -> str:
    x = _f(v)
    if not math.isfinite(x):
        return "não definido"
    return ("−" if x < 0 else "") + "R$ " + num(abs(x), casas)


def pct_br(v, casas: int = 1) -> str:
    x = _f(v)
    return "não definido" if not math.isfinite(x) else num(x * 100, casas) + "%"


def lista(itens) -> str:
    itens = [str(i) for i in itens]
    if not itens:
        return "nenhum"
    return itens[0] if len(itens) == 1 else ", ".join(itens[:-1]) + " e " + itens[-1]


# ---------------------------------------------------------------------------
# Registro dos campos
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class Campo:
    codigo: str
    secao: str                       # seção do texto em que o campo aparece
    descricao: str                   # rótulo curto exibido na lacuna do texto
    valor: Callable[[EVTEASConfig, Dict[str, Any], Optional[Dict[str, Any]]], str]
    caminhos: tuple = ()             # parâmetros cuja origem declarada acompanha o valor (entradas)

    @property
    def lacuna(self) -> str:
        return f"[{self.codigo} – {self.descricao}]"


SEC_ESTUDO, SEC_PREMISSAS, SEC_TEC = "Estudo de caso", "Premissas", "Dimensão técnica"
SEC_ECO, SEC_AMB, SEC_SOC = "Dimensão econômica", "Dimensão ambiental", "Dimensão social e ESG"
SEC_IND, SEC_MC, SEC_SENS = "Índice integrado e decisão", "Monte Carlo", "Sensibilidade e alternativas"
SEC_VV = "Verificação e execução"

NOMES_ORGANIZACAO = {"empresa": "empresa", "coop_trabalho": "cooperativa de trabalho", "coop_agro": "cooperativa agropecuária"}
NOMES_ENERGIA = {"rede": "rede pública", "solar": "solar fotovoltaica", "biomassa": "biomassa", "mista": "mista (rede + renovável)"}
NOMES_RELACAO = {"inexistente": "inexistente", "parcial": "parcial", "consolidado": "consolidada"}
NOMES_METODO = {"preset": "pesos informados diretamente", "likert": "escala Likert", "ahp": "AHP"}
NOMES_RESTRICAO = {"capacidade_suporte": "capacidade de suporte", "produtividade_area": "produtividade por área"}
NOMES_DIM = {"tecnico": "técnica", "economico": "econômica", "ambiental": "ambiental", "social": "social"}
DIAGNOSTICO_TIR = {"convencional": "fluxo convencional", "multiplas": "múltiplas TIR",
                   "multiplas_possiveis": "possíveis múltiplas TIR", "sem_solucao": "sem solução"}
ROTULO_KPI = {"oee": "OEE", "fcr": "FCR", "produtividade": "Produtividade relativa", "vpl": "VPL/investimento",
              "roi": "ROI anual", "margem_seguranca": "Margem de segurança", "ph_azul": "PH azul (m³/t)",
              "ph_cinza": "PH cinza (m³/t)", "ecoeficiencia": "Ecoeficiência (R$/kgCO2e)",
              "intensidade_carbono": "Intensidade de carbono", "conformidade": "Conformidade (0–10)",
              "lso": "LSO (0–100)", "rvl": "RVL (%)", "radar_social": "Radar Social"}
PROSA_KPI = {"oee": "OEE", "fcr": "FCR", "produtividade": "produtividade relativa", "vpl": "VPL por investimento",
             "roi": "ROI anual", "margem_seguranca": "margem de segurança", "ph_azul": "pegada hídrica azul",
             "ph_cinza": "pegada hídrica cinza", "ecoeficiencia": "ecoeficiência",
             "intensidade_carbono": "intensidade de carbono", "conformidade": "conformidade ambiental",
             "lso": "LSO", "rvl": "RVL", "radar_social": "Radar Social"}
VARIAVEIS_DIST = [  # caminho, nome (tabelas), escala de exibição, casas, nome em prosa, unidade
    ("economico.preco_venda_kg", "Preço de venda (R$/kg)", 1.0, 2, "preço de venda", "R$/kg"),
    ("economico.custo_racao_kg", "Custo da ração (R$/kg)", 1.0, 2, "custo da ração", "R$/kg"),
    ("economico.custo_alevino_milheiro", "Custo do alevino (R$/un)", 1 / 1000, 3, "custo do alevino", "R$/un"),
    ("economico.tarifa_energia_kwh", "Tarifa de energia (R$/kWh)", 1.0, 2, "tarifa de energia", "R$/kWh"),
    ("economico.custos_fixos_fator", "Fator dos custos fixos", 1.0, 2, "custos fixos", "(fator multiplicativo)"),
    ("tecnico.fcr", "FCR", 1.0, 2, "FCR", ""),
    ("tecnico.mortalidade_pct", "Mortalidade (%)", 1.0, 1, "mortalidade", "%"),
    ("tecnico.desempenho_crescimento_pct", "Desempenho de crescimento (%)", 1.0, 1, "desempenho de crescimento", "%"),
    ("economico.capex_fator", "Fator do CAPEX", 1.0, 2, "CAPEX", "(fator multiplicativo)"),
]
_VAR = {v[0]: v for v in VARIAVEIS_DIST}
PRINT_DO_BLOCO = {2: "P03", 4: "P05", 7: "P06", 10: "P08", 13: "P09", 15: "P11", 16: "P14"}


def lista_itens(itens) -> str:
    """Itens de checklist entre aspas, com inicial minúscula, separados por ponto e vírgula."""
    itens = [f"“{i[:1].lower() + i[1:]}”" for i in itens]
    if not itens:
        return "nenhum"
    return itens[0] if len(itens) == 1 else "; ".join(itens[:-1]) + "; e " + itens[-1]


def plural(n, singular, plural_):
    n = int(round(_f(n)))
    return f"{'nenhum' if n == 0 else n} {singular if n in (0, 1) else plural_}"


def meses_ou_horizonte(v) -> str:
    x = _f(v)
    return f"{num(x, 1)} meses" if math.isfinite(x) else "prazo superior ao horizonte de análise"


def tir_txt(v, diagnostico=None) -> str:
    x = _f(v)
    base = f"{num(100 * x, 1)}% a.a." if math.isfinite(x) else "não definida"
    return f"{base} ({DIAGNOSTICO_TIR.get(diagnostico, diagnostico)})" if diagnostico else base


def _remuneracao(cfg):
    r = remuneracao_mensal(cfg)
    return reais(r["clt"] + r["retirada"] + r["outros"])


def _aliquota(cfg, bases):
    aq = aliquotas_configuradas(cfg)
    return num(100 * sum(aq[b] for b in bases), 2) + "%"


def _encargos(cfg):
    aq = aliquotas_configuradas(cfg)
    partes = [(aq["folha_clt"], "sobre a folha CLT"), (aq["pro_labore"], "sobre a retirada dos cooperados"),
              (aq["folha"], "de encargos sobre a mão de obra informada em valor global"), (aq["servicos"], "sobre os serviços")]
    textos = [f"{num(100 * a, 2)}% {onde}" for a, onde in partes if a > 0]
    return lista(textos) if textos else "nenhuma"


def _distribuicao(cfg, r, caminho, escala, casas):
    d = distribuicoes_efetivas(cfg).get(caminho)
    if d is None:
        return "sem incerteza (valor fixo)"
    t = d.tipo.lower()
    if t == "fixa":
        return f"fixa em {num(d.moda * escala, casas)}"
    if t == "normal":
        texto = (f"normal (média {num(d.moda * escala, casas)}; truncada entre {num(d.minimo * escala, casas)} e "
                 f"{num(d.maximo * escala, casas)})")
    else:
        mais_provavel = "" if t == "uniforme" else f"; mais provável {num(d.moda * escala, casas)}"
        texto = f"{t} (mínimo {num(d.minimo * escala, casas)}{mais_provavel}; máximo {num(d.maximo * escala, casas)})"
    if caminho == "economico.capex_fator" and caminho not in cfg.monte_carlo.distribuicoes:
        texto += f", derivada da faixa da Classe {int(cfg.economico.classe_estimativa_aace)} da AACE"
    return texto


def _mc(r):
    if "monte_carlo" not in r:
        raise KeyError("execute a análise com Monte Carlo (executar_mc=True)")
    return r["monte_carlo"]["estatisticas"]


def _sens10(r):
    s = r["sensibilidade"]
    return s[np.isclose(s["Amplitude"], s["Amplitude"].min())].reset_index(drop=True)


def _rotulo_var(caminho):
    return _VAR[caminho][1] if caminho in _VAR else ("TMA (% a.a.)" if caminho.endswith("tma_aa_pct") else caminho)


def _prosa_var(caminho):
    return _VAR[caminho][4] if caminho in _VAR else ("TMA" if caminho.endswith("tma_aa_pct") else caminho)


def _alternativas(cfg, comp):
    """Nomes das alternativas comparadas, sem o próprio projeto (que entra na matriz como referência)."""
    return [a for a in comp["matriz"]["Alternativa"] if a != cfg.projeto]


def _topsis(comp, f):
    if not comp:
        return "comparação não realizada"
    return f(comp["ranking"])


def _e(codigo, descricao, valor, *caminhos, secao=SEC_PREMISSAS):
    return Campo(codigo, secao, descricao, lambda cfg, r, c: valor(cfg), tuple(caminhos))


def _r(codigo, secao, descricao, valor):
    return Campo(codigo, secao, descricao, lambda cfg, r, c: valor(r))


def _arquivo(cfg):
    return nome_arquivo(cfg)


def _n_mc(cfg, r):
    """Iterações e seed efetivamente usadas na execução (ou as configuradas, se não houve Monte Carlo)."""
    st = r.get("monte_carlo", {}).get("estatisticas", {}) if isinstance(r, dict) else {}
    return int(st.get("n", cfg.monte_carlo.iteracoes)), int(st.get("seed", cfg.monte_carlo.seed))


def _rampa(c):
    e = c.economico
    if e.capacidade_inicial_pct >= 100:
        return "operação a 100% da capacidade desde o início, sem rampa"
    return f"rampa de capacidade de {num(e.capacidade_inicial_pct, 0)}% a 100% em {plural(e.meses_rampa, 'mês', 'meses')}"


def _decisao(c):
    d = c.decisao
    vetos = "vetos não compensatórios aplicados" if d.aplicar_vetos else "vetos não compensatórios desativados"
    return (f"{vetos}; índice EVTEAS ≥ {num(d.limiar_indice_viavel, 2)} para “viável” e ≥ "
            f"{num(d.limiar_indice_ressalvas, 2)} para “viável com ressalvas”; P(VPL > 0) mínima de "
            f"{pct_br(d.prob_vpl_positivo_minima, 0)}; LSO mínima de {num(d.lso_minimo, 0)}")


def _alevinos(c):
    a = c.tecnico.alevinos_por_ciclo
    return "dimensionado pelo modelo" if not a or a <= 0 else f"informado pelo usuário ({num(a, 0)} por ciclo)"


CAMPOS: List[Campo] = [
    # ---------------------------------------------------------------- estudo de caso e entradas
    _e("E00", "arquivo de entradas", lambda c: _arquivo(c), secao=SEC_ESTUDO),
    _e("E01", "nome do projeto", lambda c: c.projeto, secao=SEC_ESTUDO),
    _e("E02", "tipo de organização", lambda c: NOMES_ORGANIZACAO.get(c.economico.tipo_organizacao, c.economico.tipo_organizacao), secao=SEC_ESTUDO),
    _e("E03", "espécie", lambda c: c.tecnico.especie, "tecnico.especie", secao=SEC_ESTUDO),
    _e("E04", "sistema produtivo", lambda c: c.tecnico.sistema_produtivo, secao=SEC_ESTUDO),
    _e("E05", "área de lâmina d'água (m²)", lambda c: num(c.tecnico.area_lamina_m2, 0), "tecnico.area_lamina_m2"),
    _e("E06", "profundidade média (m)", lambda c: num(c.tecnico.profundidade_media_m, 2), "tecnico.profundidade_media_m"),
    _e("E07", "viveiros/tanques no total", lambda c: num(c.tecnico.numero_tanques, 0), "tecnico.numero_tanques"),
    _e("E08", "viveiros/tanques ativos", lambda c: num(c.tecnico.tanques_ativos, 0), "tecnico.tanques_ativos"),
    _e("E09", "produtividade esperada (kg/m²/ciclo)", lambda c: num(c.tecnico.produtividade_esperada_kg_m2_ciclo, 2),
       "tecnico.produtividade_esperada_kg_m2_ciclo"),
    _e("E10", "peso inicial (g)", lambda c: num(c.tecnico.peso_inicial_g, 1), "tecnico.peso_inicial_g"),
    _e("E11", "peso final (g)", lambda c: num(c.tecnico.peso_final_g, 0), "tecnico.peso_final_g"),
    _e("E12", "duração do ciclo (dias)", lambda c: num(c.tecnico.ciclo_dias, 0), "tecnico.ciclo_dias"),
    _e("E13", "ciclos por ano", lambda c: num(c.tecnico.ciclos_ano, 2), "tecnico.ciclos_ano"),
    _e("E14", "FCR", lambda c: num(c.tecnico.fcr, 2), "tecnico.fcr"),
    _e("E15", "mortalidade esperada (%)", lambda c: num(c.tecnico.mortalidade_pct, 1) + "%", "tecnico.mortalidade_pct"),
    _e("E44", "número de alevinos por ciclo", _alevinos, "tecnico.alevinos_por_ciclo"),
    _e("E16", "CAPEX total", lambda c: reais(c.economico.capex_total), "economico.capex_itens"),
    _e("E17", "classe AACE da estimativa", lambda c: f"Classe {c.economico.classe_estimativa_aace}", "economico.classe_estimativa_aace"),
    _e("E18", "capital de giro", lambda c: reais(c.economico.capital_giro), "economico.capital_giro"),
    _e("E19", "fomento não reembolsável (% do CAPEX)", lambda c: num(c.economico.fomento_nao_reembolsavel_pct, 1) + "%",
       "economico.fomento_nao_reembolsavel_pct"),
    _e("E20", "preço médio de venda (R$/kg)", lambda c: reais(c.economico.preco_venda_kg, 2), "economico.mix_produtos", "economico.preco_venda_kg"),
    _e("E21", "custo da ração (R$/kg)", lambda c: reais(c.economico.custo_racao_kg, 2), "economico.custo_racao_kg"),
    _e("E22", "custo do alevino (R$/un)", lambda c: reais(c.economico.custo_alevino_milheiro / 1000, 3), "economico.custo_alevino_milheiro"),
    _e("E23", "tarifa de energia (R$/kWh)", lambda c: reais(c.economico.tarifa_energia_kwh, 2), "economico.tarifa_energia_kwh"),
    _e("E24", "remuneração mensal da equipe (R$/mês)", _remuneracao, "economico.salarios_clt_por_cargo",
       "economico.retirada_cooperados_mes", "economico.mao_obra_mes"),
    _e("E25", "cooperados na operação", lambda c: plural(c.economico.cooperados_trabalhadores, "cooperado na operação",
                                                         "cooperados na operação"), "economico.cooperados_trabalhadores"),
    _e("E26", "TMA (% a.a.)", lambda c: num(c.economico.tma_aa_pct, 1) + "% a.a.", "economico.tma_aa_pct"),
    _e("E27", "horizonte (anos)", lambda c: num(c.economico.horizonte_anos, 0), "economico.horizonte_anos"),
    _e("E28", "rampa de capacidade", _rampa, "economico.capacidade_inicial_pct", "economico.meses_rampa"),
    _e("E29", "alíquota sobre o faturamento", lambda c: _aliquota(c, ("faturamento", "produtos")), "economico.impostos_configurados"),
    _e("E30", "alíquota sobre o lucro", lambda c: _aliquota(c, ("lucro",)), "economico.impostos_configurados"),
    _e("E45", "alíquotas sobre a folha, a retirada dos cooperados e os serviços", _encargos,
       "economico.impostos_configurados", "economico.encargos_mao_obra_pct"),
    _e("E31", "limites de N e P do corpo receptor", lambda c: f"{num(c.ambiental.n_max_mg_l, 2)} mg/L de N e "
                                                             f"{num(c.ambiental.p_max_mg_l, 2)} mg/L de P", "ambiental.n_max_mg_l"),
    _e("E32", "remoção de N e P no tratamento", lambda c: f"{num(c.ambiental.remocao_n_tratamento_pct, 0)}% do N e "
                                                       f"{num(c.ambiental.remocao_p_tratamento_pct, 0)}% do P",
       "ambiental.remocao_n_tratamento_pct", "ambiental.remocao_p_tratamento_pct"),
    _e("E33", "fonte de energia", lambda c: NOMES_ENERGIA.get(c.ambiental.fonte_energia, c.ambiental.fonte_energia)),
    _e("E34", "empregos diretos e indiretos", lambda c: f"{plural(c.social.empregos_diretos, 'emprego direto', 'empregos diretos')} "
                                                       f"e {plural(c.social.empregos_indiretos, 'indireto', 'indiretos')}",
       "social.empregos_diretos", "social.empregos_indiretos"),
    _e("E35", "mão de obra local (%)", lambda c: num(c.social.mao_obra_local_pct, 0) + "%", "social.mao_obra_local_pct"),
    _e("E36", "compras locais (%)", lambda c: num(c.social.compras_locais_pct, 0) + "%", "social.compras_locais_pct"),
    _e("E37", "relação com a comunidade", lambda c: NOMES_RELACAO.get(c.social.relacao_comunidade, c.social.relacao_comunidade)),
    _e("E38", "reuniões e conflitos por ano", lambda c: f"{plural(c.social.reunioes_comunidade_ano, 'reunião', 'reuniões')} e "
                                                     f"{plural(c.social.conflitos_registrados_ano, 'conflito registrado', 'conflitos registrados')}",
       "social.reunioes_comunidade_ano", "social.conflitos_registrados_ano"),
    _e("E39", "método e pesos das dimensões", lambda c: _pesos_texto(c)),
    _e("E40", "regras de decisão", _decisao),
    Campo("E41", SEC_MC, "iterações do Monte Carlo", lambda cfg, r, c: num(_n_mc(cfg, r)[0], 0)),
    Campo("E42", SEC_MC, "seed", lambda cfg, r, c: str(_n_mc(cfg, r)[1])),
    _e("E43", "referências Lean-Green", lambda c: f"FCR de {num(c.tecnico.fcr_referencia, 2)}, mortalidade de "
                                                   f"{num(c.tecnico.mortalidade_referencia_pct, 1)}% e consumo de energia de "
                                                   f"{num(c.ambiental.energia_referencia_kwh_kg, 2)} kWh/kg",
       "tecnico.fcr_referencia", "tecnico.mortalidade_referencia_pct", "ambiental.energia_referencia_kwh_kg", secao=SEC_TEC),
]
CAMPOS += [Campo(f"E{50 + i}", SEC_MC, f"distribuição — {nome}",
                 (lambda cam, esc, cas: lambda cfg, r, c: _distribuicao(cfg, r, cam, esc, cas))(cam, esc, cas))
           for i, (cam, nome, esc, cas, _, _) in enumerate(VARIAVEIS_DIST)]


def _pesos_texto(c):
    w = resolver_pesos(c)["dimensoes"]
    return (f"{NOMES_METODO.get(c.pesos.metodo, c.pesos.metodo)}, com " +
            lista([f"{NOMES_DIM[d]} {num(w[d], 3)}" for d in ("tecnico", "economico", "ambiental", "social")]))


def _balanco(r, nutriente):
    a = r[Am]
    return (f"{num(a[f'{nutriente}_aportado_kg'], 0)} kg aportados pela ração, {num(a[f'{nutriente}_retido_kg'], 0)} kg "
            f"retidos na biomassa e {num(a[f'{nutriente}_lancado_kg'], 0)} kg lançados após o tratamento")


def _piores_kpis(r):
    norm = {k: _f(v) for k, v in r["indice"]["kpis_normalizados"].items()}
    ordem = sorted(norm, key=lambda k: norm[k])
    corte = norm[ordem[min(1, len(ordem) - 1)]]
    return lista([PROSA_KPI[k] for k in ordem if norm[k] <= corte + 1e-12])


T, Ec, Am, So = "tecnico", "economico", "ambiental", "social"
CAMPOS += [
    # ---------------------------------------------------------------- técnica e Lean-Green
    _r("S01", SEC_TEC, "restrição de biomassa ativa", lambda r: NOMES_RESTRICAO.get(r[T]["restricao_biomassa"], str(r[T]["restricao_biomassa"]))),
    _r("S02", SEC_TEC, "alevinos estocados por ciclo", lambda r: num(r[T]["alevinos_ciclo"], 0)),
    _r("S03", SEC_TEC, "densidade de estocagem (peixes/m³)", lambda r: num(r[T]["densidade_estocagem_peixes_m3"], 2)),
    _r("S04", SEC_TEC, "biomassa despescada por ciclo (kg)", lambda r: num(r[T]["biomassa_despesca_ciclo_kg"], 0)),
    _r("S05", SEC_TEC, "produção anual (kg/ano)", lambda r: num(r[T]["producao_kg_ano"], 0)),
    _r("S06", SEC_TEC, "produtividade (kg/m²/ano)", lambda r: num(r[T]["produtividade_kg_m2_ano"], 2)),
    _r("S07", SEC_TEC, "produtividade relativa ao potencial", lambda r: pct_br(r[T]["produtividade_relativa"])),
    _r("S08", SEC_TEC, "OEE", lambda r: num(r[T]["oee"], 3)),
    _r("S09", SEC_TEC, "disponibilidade", lambda r: num(r[T]["disponibilidade"], 3)),
    _r("S10", SEC_TEC, "performance", lambda r: num(r[T]["performance"], 3)),
    _r("S11", SEC_TEC, "sobrevivência (qualidade)", lambda r: num(r[T]["sobrevivencia"], 3)),
    _r("S12", SEC_TEC, "ração consumida (kg/ano)", lambda r: num(r[T]["racao_ano_kg"], 0)),
    _r("S13", SEC_TEC, "desperdício de ração (R$/ano)", lambda r: reais(r["lean_green"]["desperdicios_r_ano"]["Ração excedente (superalimentação/FCR)"])),
    _r("S14", SEC_TEC, "mortalidade evitável (R$/ano)", lambda r: reais(r["lean_green"]["desperdicios_r_ano"]["Mortalidade evitável"])),
    _r("S15", SEC_TEC, "energia excedente (R$/ano)", lambda r: reais(r["lean_green"]["desperdicios_r_ano"]["Energia excedente"])),
    _r("S16", SEC_TEC, "desperdícios evitáveis (R$/ano)", lambda r: reais(r["lean_green"]["total_r_ano"])),
    _r("S17", SEC_TEC, "desperdícios / custo operacional", lambda r: pct_br(r["lean_green"]["pct_opex"])),
    # ---------------------------------------------------------------- econômica
    _r("S20", SEC_ECO, "meses até a primeira receita", lambda r: num(r[Ec]["meses_ate_despesca"], 0)),
    _r("S21", SEC_ECO, "investimento total", lambda r: reais(r[Ec]["investimento_total"])),
    _r("S22", SEC_ECO, "investimento do beneficiário", lambda r: reais(_f(r[Ec]["investimento_total"]) - _f(r[Ec]["fomento_nao_reembolsavel"]))),
    _r("S23", SEC_ECO, "VPL do projeto", lambda r: reais(r[Ec]["vpl"])),
    _r("S24", SEC_ECO, "TIR anual do projeto (diagnóstico)", lambda r: tir_txt(r[Ec]["tir_anual"], r[Ec]["diagnostico_tir"])),
    _r("S25", SEC_ECO, "payback descontado", lambda r: meses_ou_horizonte(r[Ec]["payback_descontado_meses"])),
    _r("S26", SEC_ECO, "payback simples", lambda r: meses_ou_horizonte(r[Ec]["payback_simples_meses"])),
    _r("S27", SEC_ECO, "índice de lucratividade", lambda r: num(r[Ec]["indice_lucratividade"], 3)),
    _r("S28", SEC_ECO, "ano de regime", lambda r: num(r[Ec]["ano_regime"], 0)),
    _r("S29", SEC_ECO, "receita bruta no ano de regime", lambda r: reais(r[Ec]["receita_regime"])),
    _r("S30", SEC_ECO, "resultado líquido no ano de regime", lambda r: reais(r[Ec]["lucro_liquido_regime"])),
    _r("S31", SEC_ECO, "ROI anual em regime", lambda r: pct_br(r[Ec]["roi_anual_regime"])),
    _r("S32", SEC_ECO, "ponto de equilíbrio (kg/ano)", lambda r: num(r[Ec]["ponto_equilibrio_kg_ano"], 0)),
    _r("S33", SEC_ECO, "margem de segurança", lambda r: pct_br(r[Ec]["margem_seguranca"])),
    _r("S34", SEC_ECO, "custo total (R$/kg)", lambda r: reais(r[Ec]["custo_total_kg"], 2)),
    _r("S35", SEC_ECO, "custo operacional efetivo (R$/kg)", lambda r: reais(r[Ec]["custo_operacional_efetivo_kg"], 2)),
    _r("S36", SEC_ECO, "participação da ração nos custos operacionais",
       lambda r: pct_br(_f(r[Ec]["custo_racao_regime"]) / (_f(r[Ec]["custos_variaveis_regime"]) + _f(r[Ec]["custos_fixos_regime"])))),
    _r("S37", SEC_ECO, "VPL do beneficiário", lambda r: reais(r[Ec]["vpl_beneficiario"])),
    _r("S38", SEC_ECO, "TIR anual do beneficiário", lambda r: tir_txt(r[Ec]["tir_anual_beneficiario"])),
    _r("S39", SEC_ECO, "payback descontado do beneficiário", lambda r: meses_ou_horizonte(r[Ec]["payback_descontado_beneficiario_meses"])),
    _r("S40", SEC_ECO, "renda mensal por trabalhador", lambda r: reais(r[So]["renda_mensal_trabalhador"])),
    # ---------------------------------------------------------------- ambiental
    _r("S41", SEC_AMB, "água captada (m³/ano)", lambda r: num(r[Am]["agua_captada_m3_ano"], 0)),
    _r("S42", SEC_AMB, "índice de uso da água (kg/m³)", lambda r: num(r[Am]["indice_uso_agua_kg_m3"], 3)),
    _r("S43", SEC_AMB, "pegada hídrica azul (m³/t)", lambda r: num(r[Am]["ph_azul_m3_t"], 0)),
    _r("S44", SEC_AMB, "pegada hídrica cinza (m³/t)", lambda r: num(r[Am]["ph_cinza_m3_t"], 0)),
    _r("S45", SEC_AMB, "poluente crítico", lambda r: str(r[Am]["poluente_critico"])),
    _r("S46", SEC_AMB, "balanço do nitrogênio (kg/ano)", lambda r: _balanco(r, "n")),
    _r("S47", SEC_AMB, "balanço do fósforo (kg/ano)", lambda r: _balanco(r, "p")),
    _r("S50", SEC_AMB, "emissões totais (tCO2e/ano)", lambda r: num(r[Am]["co2e_total_t_ano"], 1)),
    _r("S51", SEC_AMB, "emissões por fonte",
       lambda r: f"{num(r[Am]['co2e_racao_t'], 1)} tCO2e/ano da ração, {num(r[Am]['co2e_energia_t'], 1)} tCO2e/ano da energia "
                 f"e {num(r[Am]['co2e_diesel_t'], 1)} tCO2e/ano do diesel"),
    _r("S52", SEC_AMB, "participação da ração nas emissões", lambda r: pct_br(_f(r[Am]["co2e_racao_t"]) / _f(r[Am]["co2e_total_t_ano"]))),
    _r("S53", SEC_AMB, "intensidade de carbono (kgCO2e/kg)", lambda r: num(r[Am]["intensidade_carbono_kgco2e_kg"], 2)),
    _r("S54", SEC_AMB, "ecoeficiência",
       lambda r: (reais(r[Am]["ecoeficiencia_r_por_kgco2e"], 2) + " de valor adicionado por kgCO2e"
                  if r[Am]["ecoeficiencia_status"] == "definida" else "não definida (valor adicionado não positivo)")),
    _r("S55", SEC_AMB, "score de conformidade ambiental (0–10)", lambda r: num(r[Am]["conformidade_0_10"], 2)),
    _r("S56", SEC_AMB, "itens do checklist não atendidos",
       lambda r: lista_itens(d["Item"] for d in r[Am]["conformidade_detalhe"] if not d["Atendido"])),
    _r("S57", SEC_AMB, "itens eliminatórios não atendidos", lambda r: lista_itens(r[Am]["vetos_ambientais"])),
    # ---------------------------------------------------------------- social e ESG
    _r("S60", SEC_SOC, "RVL (% do custo operacional)", lambda r: num(r[So]["rvl_pct"], 1) + "%"),
    _r("S61", SEC_SOC, "índice de impacto local", lambda r: pct_br(r[So]["indice_impacto_local"])),
    _r("S62", SEC_SOC, "renda em salários mínimos", lambda r: num(r[So]["renda_em_salarios_minimos"], 2)),
    _r("S63", SEC_SOC, "LSO (0–100)", lambda r: num(r[So]["lso_0_100"], 1)),
    _r("S59", SEC_SOC, "componentes da LSO (0–1)",
       lambda r: lista([f"{k[:1].lower() + k[1:]} {num(v, 2)}" for k, v in r[So]["lso_componentes"].items()])),
    _r("S64", SEC_SOC, "score de risco social", lambda r: num(r[So]["risco_social"], 3)),
    _r("S65", SEC_SOC, "Radar Social (média dos eixos)", lambda r: num(r[So]["radar_social_score"], 3)),
    _r("S66", SEC_SOC, "eixos do Radar com menores valores",
       lambda r: lista([k[:1].lower() + k[1:] for k, _ in sorted(r[So]["radar_social"].items(), key=lambda kv: _f(kv[1]))[:2]])),
    _r("S67", SEC_SOC, "ODS atendidos", lambda r: lista(o.split(" — ")[0] for o in r["ods"].loc[r["ods"]["Atendido"], "ODS"])),
    _r("S68", SEC_SOC, "ODS não atendidos", lambda r: lista(o.split(" — ")[0] for o in r["ods"].loc[~r["ods"]["Atendido"], "ODS"])),
    _r("S69", SEC_SOC, "itens de governança atendidos",
       lambda r: f"{sum(bool(v) for v in r['governanca']['itens'].values())} de {len(r['governanca']['itens'])} itens "
                 f"atendidos (score {num(r['governanca']['score_0_1'], 3)})"),
    _r("S70", SEC_SOC, "scores ESG",
       lambda r: f"ambiental {num(r['indice']['esg']['E'], 3)}, social {num(r['indice']['esg']['S'], 3)} e governança "
                 f"{num(r['indice']['esg']['G'], 3)}, com score ESG (média) de {num(r['indice']['esg_score'], 3)}"),
    # ---------------------------------------------------------------- índice e decisão
    _r("S71", SEC_IND, "scores dimensionais",
       lambda r: lista([f"{NOMES_DIM[d]} {num(r['indice']['dimensoes'][d], 3)}" for d in ("tecnico", "economico", "ambiental", "social")])),
    _r("S72", SEC_IND, "índice EVTEAS", lambda r: num(r["indice"]["indice_evteas"], 3)),
    _r("S73", SEC_IND, "KPIs de pior desempenho relativo", _piores_kpis),
    _r("S74", SEC_IND, "classificação", lambda r: r["decisao"]["classificacao"]),
    _r("S75", SEC_IND, "vetos e ressalvas", lambda r: lista(list(r["decisao"]["vetos"]) + list(r["decisao"]["ressalvas"]))),
    # ---------------------------------------------------------------- Monte Carlo
    _r("S80", SEC_MC, "média do VPL simulado (IC 95% da média)",
       lambda r: f"{reais(_mc(r)['media_vpl'])} (IC 95% da média: {reais(_mc(r)['ic95_media_vpl'][0])} a "
                 f"{reais(_mc(r)['ic95_media_vpl'][1])})"),
    _r("S81", SEC_MC, "mediana; desvio-padrão; CV do VPL",
       lambda r: f"{reais(_mc(r)['mediana_vpl'])}; {reais(_mc(r)['desvio_vpl'])}; {num(_mc(r)['cv_vpl'], 2)}"),
    _r("S82", SEC_MC, "VPL P5 / P50 / P95",
       lambda r: " / ".join(reais(_mc(r)["percentis_vpl"][p]) for p in ("p5", "p50", "p95"))),
    _r("S83", SEC_MC, "VaR 5% e CVaR 5% do VPL", lambda r: f"{reais(_mc(r)['var5_vpl'])} e {reais(_mc(r)['cvar5_vpl'])}"),
    _r("S84", SEC_MC, "P(VPL > 0)", lambda r: pct_br(_mc(r)["prob_vpl_positivo"])),
    _r("S85", SEC_MC, "probabilidade de VPL negativo", lambda r: pct_br(1 - _f(_mc(r)["prob_vpl_positivo"]))),
    _r("S86", SEC_MC, "TIR anual P5 / P50 / P95", lambda r: " / ".join(pct_br(_mc(r)["percentis_tir"][p]) for p in ("p5", "p50", "p95"))),
    _r("S87", SEC_MC, "índice médio; P(índice ≥ limiar)",
       lambda r: f"{num(_mc(r)['media_indice'], 3)}; {pct_br(_mc(r)['prob_indice_maior_limiar'])}"),
    _r("S88", SEC_MC, "VPL do beneficiário: média e P(VPL > 0)",
       lambda r: f"média de {reais(_mc(r)['media_vpl_beneficiario'])} e P(VPL > 0) de {pct_br(_mc(r)['prob_vpl_beneficiario_positivo'])}"),
    _r("S89", SEC_MC, "erro-padrão da média do VPL e precisão relativa", lambda r: _precisao(r)),
    # ---------------------------------------------------------------- sensibilidade, cenários e alternativas
    _r("S90", SEC_SENS, "variável de maior impacto no VPL (±10%)", lambda r: _prosa_var(_sens10(r).iloc[0]["Variável"])),
    _r("S91", SEC_SENS, "variação do VPL com −10% e +10% na variável de maior impacto",
       lambda r: f"{reais(_sens10(r).iloc[0]['Δ VPL (−)'])} e {reais(_sens10(r).iloc[0]['Δ VPL (+)'])}"),
    _r("S92", SEC_SENS, "três variáveis de maior impacto", lambda r: lista(_prosa_var(v) for v in _sens10(r)["Variável"][:3])),
    _r("S93", SEC_SENS, "valor que anula o VPL na variável de maior impacto", lambda r: _critico_frase(r, _sens10(r).iloc[0]["Variável"])),
    _r("S94", SEC_SENS, "correlação de Spearman das três variáveis mais importantes",
       lambda r: lista(f"{_prosa_var(x['Variável'])} (ρ = {num(x['Spearman com VPL'], 3)})"
                       for _, x in r["monte_carlo"]["importancia"].head(3).iterrows())),
    _r("S95", SEC_SENS, "cenário pessimista: VPL e índice", lambda r: _cenario(r, "Pessimista")),
    _r("S96", SEC_SENS, "cenário otimista: VPL e índice", lambda r: _cenario(r, "Otimista")),
    Campo("S97", SEC_SENS, "alternativas comparadas ao projeto", lambda cfg, r, c: _topsis(c, lambda t: lista(_alternativas(cfg, c)))),
    Campo("S98", SEC_SENS, "alternativa melhor classificada no TOPSIS",
          lambda cfg, r, c: _topsis(c, lambda t: f"{t.iloc[0]['Alternativa']}, com proximidade relativa à solução ideal de "
                                                 f"{num(t.iloc[0]['TOPSIS'], 3)}")),
    Campo("S99", SEC_SENS, "VPL e P(VPL > 0) da alternativa melhor classificada",
          lambda cfg, r, c: _topsis(c, lambda t: f"VPL de {reais(t.iloc[0]['VPL (R$)'])} e P(VPL > 0) de {pct_br(t.iloc[0]['P(VPL>0)'])}")),
    # ---------------------------------------------------------------- verificação e execução
    _r("S100", SEC_VV, "verificações executadas e aprovadas",
       lambda r: f"{r['vv']['quantidade_verificacoes'] - r['vv']['quantidade_erros']} de "
                 f"{r['vv']['quantidade_verificacoes']} verificações aprovadas"),
    _r("S101", SEC_VV, "tempo de execução da análise (determinística, Monte Carlo, sensibilidade e verificação)",
       lambda r: f"{num(r.get('tempo_execucao_s', float('nan')), 0)} segundos"),
    Campo("S102", SEC_IND, "pesos obtidos pela escala Likert e concordância entre avaliadores", lambda cfg, r, c: _likert(r)),
]
CASAS_KPI = {"ph_azul": 0, "ph_cinza": 0, "lso": 1, "rvl": 1}
CAMPOS += [_r(f"K{i:02d}", SEC_IND, f"{ROTULO_KPI[k]} — valor bruto",
              (lambda kk: lambda r: num(r["indice"]["kpis_brutos"][kk], CASAS_KPI.get(kk, 3)))(k))
           for i, k in enumerate([k for ks in KPIS_POR_DIMENSAO.values() for k in ks], 1)]
CAMPOS += [_r(f"N{i:02d}", SEC_IND, f"{ROTULO_KPI[k]} — normalizado",
              (lambda kk: lambda r: num(r["indice"]["kpis_normalizados"][kk], 3))(k))
           for i, k in enumerate([k for ks in KPIS_POR_DIMENSAO.values() for k in ks], 1)]


def _precisao(r):
    ep = _f(r["monte_carlo"]["convergencia"].iloc[-1]["Erro-padrão"])
    media = abs(_f(_mc(r)["media_vpl"]))
    rel = f", equivalente a {pct_br(1.96 * ep / media)} da média em módulo" if media > 0 else ""
    return f"{reais(ep)}; a semiamplitude do IC 95% da média é de {reais(1.96 * ep)}{rel}"


def _critico(r, var):
    """Valor crítico (VPL = 0) e folga em relação ao valor informado, em forma curta (tabelas)."""
    if var.endswith("tma_aa_pct"):
        tir = _f(r[Ec]["tir_anual"])
        return f"{num(100 * tir, 1)}% a.a. (= TIR)" if math.isfinite(tir) else "não definido (TIR inexistente)"
    vc = r["valores_criticos"].set_index("Variável")
    if var not in vc.index or not math.isfinite(_f(vc.loc[var, "Valor crítico (VPL = 0)"])):
        return "sem valor crítico no intervalo analisado"
    escala, casas = (_VAR[var][2], _VAR[var][3]) if var in _VAR else (1.0, 2)
    valor = num(_f(vc.loc[var, "Valor crítico (VPL = 0)"]) * escala, min(casas + 1, 3))
    folga = _f(vc.loc[var, "Folga relativa"])
    if not math.isfinite(folga):
        return f"{valor} (valor informado nulo)"
    return f"{valor} ({'+' if folga >= 0 else '−'}{num(abs(folga) * 100, 1)}%)"


def _critico_frase(r, var):
    """Valor crítico (VPL = 0) da variável em frase completa (texto)."""
    if var.endswith("tma_aa_pct"):
        tir = _f(r[Ec]["tir_anual"])
        return (f"o VPL se anula quando a TMA iguala a TIR do projeto ({num(100 * tir, 1)}% a.a.)" if math.isfinite(tir)
                else "não há TMA que anule o VPL, pois a TIR não é definida")
    vc = r["valores_criticos"].set_index("Variável")
    if var not in vc.index or not math.isfinite(_f(vc.loc[var, "Valor crítico (VPL = 0)"])):
        return "não há valor que anule o VPL no intervalo analisado"
    escala, casas, unidade = (_VAR[var][2], _VAR[var][3], _VAR[var][5]) if var in _VAR else (1.0, 2, "")
    valor = num(_f(vc.loc[var, "Valor crítico (VPL = 0)"]) * escala, min(casas + 1, 3))
    if unidade == "%":
        valor = f"{valor}%"
    elif unidade.startswith("R$/"):
        valor = f"R$ {valor}/{unidade[3:]}"
    else:
        valor = f"{valor} {unidade}".strip()
    folga = _f(vc.loc[var, "Folga relativa"])
    if not math.isfinite(folga):
        return f"o valor que anula o VPL é {valor} (o valor informado é nulo)"
    return (f"o valor que anula o VPL é {valor}, {num(abs(folga) * 100, 1)}% "
            f"{'acima' if folga >= 0 else 'abaixo'} do valor informado")


def _cenario(r, nome):
    c = r["cenarios"].set_index("Cenário")
    return f"VPL de {reais(c.loc[nome, 'VPL'])} e índice de {num(c.loc[nome, 'Índice EVTEAS'], 3)}"


def _likert(r):
    pesos = r["indice"]["pesos"]
    lk = pesos.get("likert_dimensoes")
    if not lk:
        return "não se aplica (método de ponderação diferente da escala Likert)"
    w = lk["pesos"]
    return (lista([f"{NOMES_DIM[d]} {num(w[d], 3)}" for d in ("tecnico", "economico", "ambiental", "social")])
            + f", com concordância média de {num(lk['concordancia_media'], 3)}")


POR_CODIGO: Dict[str, Campo] = {c.codigo: c for c in CAMPOS}
CODIGO_KPI = {k: f"{i:02d}" for i, k in enumerate([k for ks in KPIS_POR_DIMENSAO.values() for k in ks], 1)}


# ---------------------------------------------------------------------------
# Prints (telas do wizard e saídas do notebook) que ilustram o texto
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class Print:
    codigo: str
    descricao: str
    onde: str                        # onde a tela ou a saída aparece no notebook
    como: str = ""                   # orientação de captura (roteiro)

    @property
    def lacuna(self) -> str:
        return f"[INSERIR PRINT {self.codigo} – {self.descricao} ({self.onde})]"


WIZARD = "wizard de entrada, seção “Entrada de dados do projeto” do notebook"
ANALISE = "seção “Análise com as entradas informadas” do notebook"
GRAFICOS = "seção “Exportação e gráficos” do notebook"
COMO_WIZARD = ("Capture as perguntas e as respostas dos blocos indicados, que o wizard anuncia com a linha ▶ PRINT. "
               "Se não couberem numa tela, use mais de uma captura, dispostas lado a lado ou em sequência. Use a opção "
               "“Novo projeto” ou, ao reabrir um arquivo, responda s à revisão bloco a bloco; se corrigir um bloco depois "
               "do resumo, refaça a captura correspondente.")
COMO_GRAFICO = ("Insira o arquivo PNG gravado em evteas_output/figuras (incluído em evteas_output.zip, baixado no Colab) "
                "ou capture o gráfico exibido abaixo da linha ▶ PRINT.")
COMO_TABELA = "Capture a tabela exibida abaixo da linha ▶ PRINT."
PRINTS: List[Print] = [
    Print("P01", "menu inicial e bloco 1 do wizard (identificação do projeto)", WIZARD, COMO_WIZARD),
    Print("P02", "resumo das entradas exibido antes da confirmação", WIZARD, COMO_TABELA),
    Print("P03", "blocos 2 e 3 do wizard (infraestrutura; parâmetros biológicos e insumos)", WIZARD, COMO_WIZARD),
    Print("P04", "curva de crescimento, biomassa e arraçoamento (gráfico 01_curva_crescimento)", GRAFICOS, COMO_GRAFICO),
    Print("P05", "blocos 4 a 6 do wizard (investimento; mix de produtos e receitas; custos operacionais)", WIZARD, COMO_WIZARD),
    Print("P06", "blocos 7 a 9 do wizard (pessoas; tributos; rampa, projeção e TMA)", WIZARD, COMO_WIZARD),
    Print("P07", "receita, custos e fluxo de caixa acumulado (gráfico 02_projecao_economica)", GRAFICOS, COMO_GRAFICO),
    Print("P08", "blocos 10 a 12 do wizard (água e efluentes; energia, emissões e resíduos; conformidade)", WIZARD, COMO_WIZARD),
    Print("P09", "blocos 13 e 14 do wizard (dimensão social; governança)", WIZARD, COMO_WIZARD),
    Print("P10", "Radar Social (gráfico 04_radar_social)", GRAFICOS, COMO_GRAFICO),
    Print("P11", "bloco 15 do wizard (pesos do índice e regras de decisão)", WIZARD, COMO_WIZARD),
    Print("P12", "scores dimensionais e índice EVTEAS (gráfico 03_dimensoes)", GRAFICOS, COMO_GRAFICO),
    Print("P13", "classificação e resumo executivo", ANALISE, COMO_TABELA),
    Print("P14", "bloco 16 do wizard (distribuições do Monte Carlo)", WIZARD, COMO_WIZARD),
    Print("P15", "distribuição e probabilidade acumulada do VPL (gráfico 05_monte_carlo_vpl)", GRAFICOS, COMO_GRAFICO),
    Print("P16", "convergência da simulação (gráfico 06_convergencia_mc)", GRAFICOS, COMO_GRAFICO),
    Print("P17", "tabela de sensibilidade, valores críticos e importância das variáveis", GRAFICOS, COMO_TABELA),
    Print("P18", "diagrama de tornado (gráfico 07_tornado)", GRAFICOS, COMO_GRAFICO),
    Print("P19", "ranking TOPSIS das alternativas", "seção “Comparação com alternativas de projeto” do notebook", COMO_TABELA),
    Print("P20", "proximidade relativa à solução ideal (gráfico 08_topsis)", GRAFICOS, COMO_GRAFICO),
]
PRINT_POR_CODIGO: Dict[str, Print] = {p.codigo: p for p in PRINTS}
FIGURA_DO_PRINT = {"01_curva_crescimento": "P04", "02_projecao_economica": "P07", "04_radar_social": "P10",
                   "03_dimensoes": "P12", "05_monte_carlo_vpl": "P15", "06_convergencia_mc": "P16",
                   "07_tornado": "P18", "08_topsis": "P20"}


def roteiro_de_prints() -> pd.DataFrame:
    """Prints que ilustram o texto, na ordem em que aparecem, com o local e o modo de captura."""
    return pd.DataFrame([{"Print": p.codigo, "Conteúdo": p.descricao, "Onde capturar": p.onde, "Como capturar": p.como}
                         for p in PRINTS])


def titulo_print(codigo: str) -> str:
    """Cabeçalho exibido no notebook (e no wizard) antes da tela ou da saída a capturar."""
    p = PRINT_POR_CODIGO[codigo]
    return f"▶ PRINT {p.codigo} — {p.descricao}"


def lacuna(codigo: str) -> str:
    """Texto da lacuna, como aparece na dissertação (ex.: '[S23 – VPL do projeto]')."""
    return POR_CODIGO[codigo].lacuna


def _origem(cfg: EVTEASConfig, caminhos) -> str:
    """Origem declarada dos parâmetros da lacuna; origens diferentes são listadas todas."""
    vistas = []
    for c in caminhos:
        f = cfg.fontes.get(c)
        if f is not None:
            cat = CATEGORIAS_FONTE.get(getattr(f, "categoria", ""), getattr(f, "categoria", ""))
            ref = getattr(f, "referencia", "")
            texto = f"{cat}{' — ' + ref if ref else ''}"
            if texto not in vistas:
                vistas.append(texto)
    if vistas:
        return "; ".join(vistas)
    return "—" if not caminhos else "não registrada"


def _assinatura(config) -> str:
    import json
    from dataclasses import asdict, is_dataclass
    return json.dumps(asdict(config) if is_dataclass(config) else config, sort_keys=True, default=str, ensure_ascii=False)


def valores_para_o_texto(cfg: EVTEASConfig, resultado: Dict[str, Any],
                         comparacao: Optional[Dict[str, Any]] = None) -> pd.DataFrame:
    """Valor de cada lacuna do texto na execução do usuário, na ordem do texto.

    Use após ``executar_evteas(cfg, executar_mc=True, executar_sens=True)``; ``comparacao``
    é o resultado de ``definir_e_comparar_alternativas`` (opcional). O resultado precisa ter
    sido calculado com estas mesmas entradas, para que entradas e resultados não se misturem."""
    if "config" in resultado and _assinatura(resultado["config"]) != _assinatura(cfg):
        raise ValueError("O resultado foi calculado com outras entradas. Execute novamente a análise "
                         "(seção “Análise com as entradas informadas”) antes de gerar os valores para o texto.")
    aviso_comparacao = None
    if comparacao and cfg.projeto in comparacao.get("resultados", {}):
        if _assinatura(comparacao["resultados"][cfg.projeto]["config"]) != _assinatura(cfg):
            comparacao, aviso_comparacao = None, ("comparação feita com outras entradas: refaça a seção "
                                                  "“Comparação com alternativas de projeto”")
    linhas = []
    for c in CAMPOS:
        try:
            valor = c.valor(cfg, resultado, comparacao)
        except (KeyError, IndexError, TypeError, ValueError, ZeroDivisionError) as exc:
            valor = f"indisponível ({exc})"
        if aviso_comparacao and c.codigo in ("S97", "S98", "S99"):
            valor = aviso_comparacao
        linhas.append({"Código": c.codigo, "Seção": c.secao, "Lacuna no texto": c.lacuna, "Valor": valor,
                       "Origem declarada": _origem(cfg, c.caminhos) if c.codigo.startswith("E") else ""})
    return pd.DataFrame(linhas)


def exportar_valores_para_o_texto(df: pd.DataFrame, pasta: str = "evteas_output", baixar: bool = False) -> str:
    """Grava a tabela em Excel (com o roteiro de prints numa segunda aba); no Colab, ``baixar`` faz o download."""
    from pathlib import Path
    Path(pasta).mkdir(parents=True, exist_ok=True)
    caminho = str(Path(pasta) / "valores_para_o_texto.xlsx")
    with pd.ExcelWriter(caminho) as xl:
        df.to_excel(xl, sheet_name="Valores para o texto", index=False)
        roteiro_de_prints().to_excel(xl, sheet_name="Roteiro de prints", index=False)
    if baixar:
        _baixar_no_colab(caminho)
    return caminho

## 13. Entrada de dados do projeto

Execute a célula e responda às perguntas. Nenhum resultado é calculado antes desta etapa. Para o relato do Capítulo 4, capture as telas do wizard indicadas no roteiro de prints.

In [ ]:
roteiro_de_prints()        # telas e saídas que ilustram o Capítulo 4

In [ ]:
cfg = resultado = comparacao = None   # nada de uma execução anterior é reaproveitado
cfg = iniciar_entradas()

## 14. Análise com as entradas informadas

In [ ]:
exigir_entradas(cfg)
assert teste_regressao_deterministica()
resultado = executar_evteas(cfg, executar_mc=True, executar_sens=True,
                            n_mc=int(cfg.monte_carlo.iteracoes))
print(titulo_print('P13'))
print('Classificação:', resultado['decisao']['classificacao'])
for motivo in resultado['decisao']['vetos'] + resultado['decisao']['ressalvas']:
    print(' -', motivo)
resumo_executivo(resultado)

In [ ]:
resultado['premissas']   # valor, origem e distribuição de cada premissa

In [ ]:
resultado['dre_anual'].round(0)

In [ ]:
st = resultado['monte_carlo']['estatisticas']
{k: v for k, v in st.items() if not isinstance(v, dict)}

In [ ]:
resultado['monte_carlo']['importancia']

In [ ]:
display(resultado['valores_criticos'])
display(resultado['cenarios'])

In [ ]:
resultado['ods']

In [ ]:
import pandas as pd
pd.DataFrame(resultado['vv']['verificacoes'])

## 15. Comparação com alternativas de projeto (TOPSIS)

Cada alternativa parte das entradas do projeto; escolha os blocos a alterar e informe os novos valores. É opcional para o uso do artefato, mas a Seção 4 da dissertação a utiliza: execute-a antes da Seção 16. O Monte Carlo de cada alternativa usa as mesmas iterações e a mesma seed da análise principal.

In [ ]:
comparacao = None
comparacao = definir_e_comparar_alternativas(exigir_entradas(cfg), n_mc=int(cfg.monte_carlo.iteracoes))
for aviso in (comparacao or {}).get('avisos', []):
    print('⚠', aviso)
if comparacao:
    print(titulo_print('P19'))
tabela_topsis(comparacao) if comparacao else 'Comparação não solicitada.'

## 16. Valores para o texto

Valor de cada lacuna dos Capítulos 4 e 5 nesta execução, na ordem do texto, com a origem declarada de cada entrada. A tabela é gravada em `evteas_output/valores_para_o_texto.xlsx` (baixada no Colab), com o roteiro de prints numa segunda aba.

In [ ]:
exigir_entradas(cfg)
valores = valores_para_o_texto(cfg, resultado, globals().get('comparacao'))
print('Planilha:', exportar_valores_para_o_texto(valores, 'evteas_output', baixar=True))
pd.set_option('display.max_rows', None, 'display.max_colwidth', None)
valores

## 17. Exportação e gráficos

Cada gráfico é precedido da linha `▶ PRINT P..` que indica a figura do Capítulo 4 a que se destina. Os PNGs ficam em `evteas_output/figuras`; no Colab, a pasta `evteas_output` é compactada e baixada.

In [ ]:
pasta = 'evteas_output'
arquivos = exportar(resultado, pasta)
figuras = gerar_graficos(resultado, pasta + '/figuras', globals().get('comparacao'))
diagrama_arquitetura(pasta + '/figuras/00_arquitetura.png')
from IPython.display import Image
saidas = {codigo: figuras[nome] for nome, codigo in FIGURA_DO_PRINT.items() if nome in figuras}
saidas['P17'] = None          # tabela de sensibilidade
for codigo in sorted(saidas):
    print(titulo_print(codigo))
    display(tabela_sensibilidade(resultado) if codigo == 'P17' else Image(saidas[codigo]))
import shutil
pacote = shutil.make_archive('evteas_output', 'zip', pasta)
baixar_no_colab(pacote)       # figuras, planilhas e relatório (só tem efeito no Colab)
arquivos

## 18. Rastreabilidade do artefato (Design Science Research) — opcional

Etapas da DSR (Dresch; Lacerda; Antunes Jr., 2015), comparação com artefatos preexistentes (Wazlawick, 2009), matriz requisito → função → teste e aprendizagens registradas. Fora do repositório, a pasta de testes não está disponível e os testes não são conferidos.

In [ ]:
for nome, tabela in relatorio_dsr().items():
    print(nome)
    display(tabela)